In [ ]:
# -*- coding: utf-8 -*-
"""
Ordinary SAC-FNN with one independent frozen new Hadamard-DeepONet action prior.

All environment and training settings are aligned with the latest TDPSAC
training program:

1. Qbar(x)=11.93 and Hbar(x) is solved from the steady Saint-Venant ODE.
2. Each episode starts from collection-matched truncated-Chebyshev
   perturbations around Hbar,Qbar.
3. SAC observes the raw physical state [H_1,...,H_N,Q_1,...,Q_N], with no
   cumulative-error state.
4. The reward uses H/Q tracking, inter-sample state-change, control, and delta-control costs.
5. The new DeepONet uses Hadamard Branch-Trunk fusion z=b(s)*t(L), followed by
   a 102->32->16->1 physical-gate head and the checkpoint pretrained from
   y_ul.dat.

Method retained from the uploaded program:

- Actor: ordinary SB3 SAC FNN, no DeepONet feature extraction.
- Critic: ordinary SB3 twin-Q FNN, no DeepONet feature extraction.
- Prior: one independent new Hadamard DeepONet is fully frozen and is used
  only to provide the physical control prior.
- Final normalized action:
      a_hybrid = (1-alpha_hybrid)*a_prior + alpha_hybrid*a_SAC.

Required external module:
    cac_saint_venant_env_no_bs.py
"""

from __future__ import annotations

import os
import random
import time
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import torch
from scipy.integrate import solve_ivp
from gymnasium import spaces
from torch import nn

from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import (
    BaseCallback,
    CallbackList,
    CheckpointCallback,
)
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.sac.policies import Actor, SACPolicy

from cac_saint_venant_env_no_bs import CACParams, CACSaintVenantPDE1D

try:
    from pde_control_gym.src.rewards.base_reward import BaseReward
except Exception:
    class BaseReward:
        pass


# ============================================================
# 1. Simulation parameters
# ============================================================

N = 102
T = 40000.0
dt = 5.0
control_sample_rate = 400.0

max_state_value = 1e6

# The PDE environment maps normalized SAC actions a in [-1,1] to the physical
# downstream gate U_L in this interval. These limits match the data collector.
min_control_value = 0.15
max_control_value = 0.40

# State/reward normalization scales. H and Q remain raw in the observation;
# these scales are used only inside the reward calculation.
H_scale = 0.30
Q_scale = 3
U_scale = 0.10

obs_clip = 50.0

cfl = 0.45
check_cfl = True
well_balanced = True


# ============================================================
# 2. Collection-matched equilibrium and initial-condition settings
# ============================================================

Q_BAR_FIXED = 11.93
REFERENCE_MODE = "ode"
U_EQ_VALUE = 0.30

CHEB_ORDER = 6
H_CHEB_AMPLITUDE = 0.30
Q_CHEB_AMPLITUDE = 2.50
CHEB_COEFF_DECAY = 2.0


# ============================================================
# 3. SAC parameters
# ============================================================

total_timesteps = int(1e5)
learning_rate = 1e-4

# Use a much smaller learning rate for trainable DeepONet parameters.
deeponet_learning_rate = 1e-6

buffer_size = 100000
batch_size = 256
tau = 0.005
sac_gamma = 0.995
ent_coef = 0.005
learning_starts = 5000

seeds = [42,43,44,45]

LATENT_DIM = 102
RAW_STATE_DIM = 2 * N
SAC_FEATURE_DIM = LATENT_DIM

# This checkpoint must come from the Hadamard-fusion DeepONet pretraining code.
DEEPNET_PRETRAINED_PATH = "modedlNet10000QqQqQ14444_hadamard.pt"


# ============================================================
# 4. Reward parameters
# ============================================================

reward_alpha = 1
reward_beta = 0.1
reward_gamma = 0.05
reward_delta =0.1

# Separate H/Q weights for tracking and inter-sample state-change terms.
tracking_w_H = 1.0
tracking_w_Q = 1
change_w_H = 1.0
change_w_Q = 1

terminal_tracking_weight = 0.0
truncate_penalty = -200.0


# ============================================================
# 5. CAC physical parameters, ODE reference, and Chebyshev initial conditions
# ============================================================

def make_cac_table_params(
    q_bar: float = Q_BAR_FIXED,
    h_bar_0: float = 3.80,
    h_bar_L: float = 4.70,
) -> CACParams:
    """Create the CAC parameter object used by collection and RL training."""
    return CACParams(
        L=11239.0,
        B=40.0,
        g=9.81,
        I=7.92e-5,
        K=33.0,
        mu0=0.4,
        muL=0.4,
        z_up=5.00,
        U0_bar=0.1548,
        U_L_bar=U_EQ_VALUE,
        h_s=4.094,
        Q_bar=float(q_bar),
        H_bar_0=float(h_bar_0),
        H_bar_L=float(h_bar_L),
        H_min=0.20,
        max_state_value=1e10,
    )


def make_grid(p: CACParams, n_grid: int) -> Tuple[np.ndarray, float]:
    X = np.linspace(0.0, p.L, n_grid)
    dx = p.L / (n_grid - 1)
    return X, dx


def downstream_equilibrium_depth_from_q(Q: float, p: CACParams) -> float:
    """Compute Hbar(L) from the downstream equilibrium gate relation."""
    Q = float(Q)
    if not np.isfinite(Q) or Q <= 0.0:
        raise ValueError("The equilibrium discharge must be positive and finite.")

    kinetic_head = (
        Q**2 / (2.0 * p.g * p.B**2 * p.muL**2)
    ) ** (1.0 / 3.0)
    return float(p.h_s + p.U_L_bar + kinetic_head)


def integrate_steady_fixed_q_ode(
    X: np.ndarray,
    p: CACParams,
    Q_candidate: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Integrate the steady Saint-Venant ODE backward from x=L to x=0."""
    X = np.asarray(X, dtype=float)
    if X.ndim != 1 or X.size < 2:
        raise ValueError("X must be a one-dimensional grid with at least two points.")
    if not np.all(np.diff(X) > 0.0):
        raise ValueError("X must be strictly increasing.")

    Q_candidate = float(Q_candidate)
    if not np.isfinite(Q_candidate) or Q_candidate <= 0.0:
        raise ValueError("Q_candidate must be positive and finite.")

    Hbar_L = downstream_equilibrium_depth_from_q(Q_candidate, p)

    def rhs(_x: float, state: np.ndarray) -> np.ndarray:
        H = float(state[0])
        Q = float(state[1])

        if not np.isfinite(H) or not np.isfinite(Q):
            raise FloatingPointError("Non-finite state in steady reference ODE.")
        if H <= float(p.H_min):
            raise FloatingPointError("Steady reference ODE violates minimum depth.")

        area = p.B * H
        hydraulic_radius = p.B * H / (p.B + 2.0 * H)
        friction_slope = Q * abs(Q) / (
            p.K**2 * area**2 * hydraulic_radius ** (4.0 / 3.0)
        )

        denominator = p.g * p.B * H - Q**2 / (p.B * H**2)
        if abs(denominator) <= 1e-10:
            raise FloatingPointError(
                "Steady ODE approaches the critical-flow singularity."
            )

        dHdx = p.g * p.B * H * (p.I - friction_slope) / denominator
        if not np.isfinite(dHdx):
            raise FloatingPointError("Non-finite dHbar/dx in steady ODE.")

        return np.asarray([dHdx, 0.0], dtype=float)

    sol = solve_ivp(
        fun=rhs,
        t_span=(float(X[-1]), float(X[0])),
        y0=np.asarray([Hbar_L, Q_candidate], dtype=float),
        t_eval=X[::-1],
        rtol=1e-9,
        atol=1e-11,
        max_step=p.L / 1000.0,
    )

    if not sol.success:
        raise RuntimeError(f"Failed to solve steady ODE: {sol.message}")
    if sol.y.shape[1] != X.size:
        raise RuntimeError("Steady ODE did not return all requested grid points.")

    Hbar = np.asarray(sol.y[0][::-1], dtype=float)
    Qbar = np.asarray(sol.y[1][::-1], dtype=float)

    if not np.all(np.isfinite(Hbar)) or not np.all(np.isfinite(Qbar)):
        raise FloatingPointError("Steady reference contains NaN or Inf.")
    if float(np.min(Hbar)) <= float(p.H_min):
        raise RuntimeError("Steady Hbar profile violates the minimum depth.")

    return Hbar, Qbar


def solve_steady_reference_ode(
    X: np.ndarray,
    p: CACParams,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Solve Hbar(x) for prescribed spatially constant Qbar=11.93."""
    q_bar_fixed = float(q_bar_fixed)
    Hbar, _ = integrate_steady_fixed_q_ode(X, p, q_bar_fixed)
    Qbar = np.full_like(X, q_bar_fixed, dtype=float)
    return Hbar, Qbar, q_bar_fixed


def make_reference_profiles(
    X: np.ndarray,
    p: CACParams,
    reference: str = REFERENCE_MODE,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    if reference in {"ode", "strict"}:
        return solve_steady_reference_ode(X, p, q_bar_fixed=q_bar_fixed)
    if reference == "linear":
        Hbar = p.H_bar_0 + (p.H_bar_L - p.H_bar_0) * (X / p.L)
        Qbar = np.full_like(X, q_bar_fixed, dtype=float)
        return Hbar, Qbar, float(q_bar_fixed)
    raise ValueError("reference must be 'ode', 'strict', or 'linear'.")


def build_collection_matched_reference():
    """Build one internally consistent p, X, Hbar, Qbar tuple."""
    nominal_p = make_cac_table_params(q_bar=Q_BAR_FIXED)
    X, dx = make_grid(nominal_p, N)
    Hbar, Qbar, q_equilibrium = make_reference_profiles(
        X,
        nominal_p,
        reference=REFERENCE_MODE,
        q_bar_fixed=Q_BAR_FIXED,
    )

    p = make_cac_table_params(
        q_bar=q_equilibrium,
        h_bar_0=float(Hbar[0]),
        h_bar_L=float(Hbar[-1]),
    )
    return p, X, dx, Hbar, Qbar


def sample_truncated_chebyshev(
    xi: np.ndarray,
    rng,
    order: int,
    max_amplitude: float,
    coefficient_decay: float,
) -> np.ndarray:
    """Sample the same smooth truncated-Chebyshev perturbation as collection."""
    if order < 0:
        raise ValueError("CHEB_ORDER must be nonnegative.")
    if max_amplitude < 0.0:
        raise ValueError("Chebyshev amplitude must be nonnegative.")
    if coefficient_decay < 0.0:
        raise ValueError("CHEB_COEFF_DECAY must be nonnegative.")
    if max_amplitude == 0.0:
        return np.zeros_like(xi, dtype=float)

    mode_index = np.arange(order + 1, dtype=float)
    coefficient_scale = 1.0 / np.power(mode_index + 1.0, coefficient_decay)
    coefficients = rng.normal(loc=0.0, scale=coefficient_scale)

    perturbation = np.polynomial.chebyshev.chebval(xi, coefficients)
    max_abs = float(np.max(np.abs(perturbation)))
    if max_abs <= 1e-12:
        return np.zeros_like(xi, dtype=float)

    realized_amplitude = rng.uniform(0.35, 1.0) * max_amplitude
    return perturbation * (realized_amplitude / max_abs)


def random_initial_condition_func(X, Hbar, Qbar, rng):
    """Generate H0,Q0 with the same Chebyshev distribution as collection."""
    X = np.asarray(X, dtype=float)
    Hbar = np.asarray(Hbar, dtype=float)
    Qbar = np.asarray(Qbar, dtype=float)
    xi = 2.0 * X / float(X[-1]) - 1.0

    # Rejection sampling preserves the polynomial shape; no pointwise clipping.
    for _ in range(100):
        H_perturbation = sample_truncated_chebyshev(
            xi,
            rng,
            CHEB_ORDER,
            H_CHEB_AMPLITUDE,
            CHEB_COEFF_DECAY,
        )
        Q_perturbation = sample_truncated_chebyshev(
            xi,
            rng,
            CHEB_ORDER,
            Q_CHEB_AMPLITUDE,
            CHEB_COEFF_DECAY,
        )

        H0 = Hbar + H_perturbation
        Q0 = Qbar + Q_perturbation

        finite = np.all(np.isfinite(H0)) and np.all(np.isfinite(Q0))
        valid_depth = float(np.min(H0)) > 0.20
        valid_discharge = float(np.min(Q0)) > 0.0

        if finite and valid_depth and valid_discharge:
            return H0, Q0

    raise RuntimeError(
        "Failed to generate a valid Chebyshev initial condition after 100 "
        "attempts. Reduce H_CHEB_AMPLITUDE or Q_CHEB_AMPLITUDE."
    )


# ============================================================
# 6. Reward class
# ============================================================

class CACTrackingSmoothReward(BaseReward):
    """
    Discrete reward at the control instant t_k:

        r_k = -alpha * tracking_cost
              -beta  * state_change_cost
              -gamma * control_cost
              -delta * delta_control_cost.

    Definitions:
        tracking_cost
            = tracking_w_H * mean(((H_k-H_eq)/H_scale)^2)
            + tracking_w_Q * mean(((Q_k-Q_eq)/Q_scale)^2)

        state_change_cost
            = change_w_H * mean(((H_k-H_{k-1})/H_scale)^2)
            + change_w_Q * mean(((Q_k-Q_{k-1})/Q_scale)^2)

        control_cost
            = ((U_k-U_eq)/U_scale)^2

        delta_control_cost
            = (((U_k-U_eq)/U_scale)
               -((U_{k-1}-U_eq)/U_scale))^2

    The state-change term is an inter-sample increment penalty. It is not a
    time derivative because it is not divided by the control interval.

    Spatial means are used instead of unnormalized sums so that the reward scale
    does not grow with the number of grid points N.
    """

    def __init__(
        self,
        H_eq,
        Q_eq,
        delta_t=control_sample_rate,
        H_scale=0.10,
        Q_scale=1.50,
        U_scale=0.10,
        alpha=1.0,
        beta=0.01,
        gamma=0.05,
        delta=0.50,
        tracking_w_H=1.0,
        tracking_w_Q=1.0,
        change_w_H=1.0,
        change_w_Q=0.50,
        U_ref=U_EQ_VALUE,
        terminal_tracking_weight=0.0,
        truncate_penalty=-200.0,
    ):
        self.H_eq = np.asarray(H_eq, dtype=float)
        self.Q_eq = np.asarray(Q_eq, dtype=float)
        self.N = len(self.H_eq)

        self.delta_t = float(delta_t)
        self.H_scale = float(H_scale)
        self.Q_scale = float(Q_scale)
        self.U_scale = float(U_scale)

        self.alpha = float(alpha)
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.delta = float(delta)

        self.tracking_w_H = float(tracking_w_H)
        self.tracking_w_Q = float(tracking_w_Q)
        self.change_w_H = float(change_w_H)
        self.change_w_Q = float(change_w_Q)

        self.U_ref = float(U_ref)
        self.terminal_tracking_weight = float(terminal_tracking_weight)
        self.truncate_penalty = float(truncate_penalty)

        # Previous values at the preceding reward/control call.
        self.prev_H: Optional[np.ndarray] = None
        self.prev_Q: Optional[np.ndarray] = None
        self.prev_action: Optional[float] = None

        # Diagnostics.
        self.last_terms = {
            "tracking_cost": 0.0,
            "state_change_cost": 0.0,
            "control_cost": 0.0,
            "delta_control_cost": 0.0,
        }

    def _reset_memory(self):
        self.prev_H = None
        self.prev_Q = None
        self.prev_action = None

    def _split_state(self, state_vec):
        state_vec = np.asarray(state_vec, dtype=float).reshape(-1)
        if state_vec.size < 2 * self.N:
            raise ValueError(
                f"PDE state length is {state_vec.size}, expected at least {2 * self.N}."
            )
        H = state_vec[:self.N]
        Q = state_vec[self.N:2 * self.N]
        return H, Q

    def _action_value(self, action):
        # The original environment passes the physical downstream gate to reward.
        if action is None:
            return self.U_ref
        value = float(np.asarray(action).reshape(-1)[0])
        return float(
            np.nan_to_num(
                value,
                nan=self.U_ref,
                posinf=self.U_ref,
                neginf=self.U_ref,
            )
        )

    def reward(
        self,
        uVec=None,
        time_index=None,
        terminate=None,
        truncate=None,
        action=None,
        control_sample_rate=None,
    ):
        if uVec is None or time_index is None:
            return 0.0

        # A new episode normally starts with a small time index.
        if int(time_index) <= 1:
            self._reset_memory()

        if truncate:
            self._reset_memory()
            return float(self.truncate_penalty)

        Hk, Qk = self._split_state(uVec[time_index])

        if (
            not np.all(np.isfinite(Hk))
            or not np.all(np.isfinite(Qk))
            or np.any(Hk <= 0.0)
        ):
            self._reset_memory()
            return float(self.truncate_penalty)

        # ----------------------------------------------------
        # 1) Tracking term
        # ----------------------------------------------------
        H_error_tilde = (Hk - self.H_eq) / self.H_scale
        Q_error_tilde = (Qk - self.Q_eq) / self.Q_scale

        tracking_cost = (
            self.tracking_w_H * np.mean(H_error_tilde ** 2)
            + self.tracking_w_Q * np.mean(Q_error_tilde ** 2)
        )

        # ----------------------------------------------------
        # 2) Inter-sample state-change term
        # ----------------------------------------------------
        if self.prev_H is None or self.prev_Q is None:
            state_change_cost = 0.0
        else:
            delta_H_tilde = (Hk - self.prev_H) / self.H_scale
            delta_Q_tilde = (Qk - self.prev_Q) / self.Q_scale

            state_change_cost = (
                self.change_w_H * np.mean(delta_H_tilde ** 2)
                + self.change_w_Q * np.mean(delta_Q_tilde ** 2)
            )

        # ----------------------------------------------------
        # 3) Control-magnitude term
        # ----------------------------------------------------
        Uk = self._action_value(action)
        U_tilde = (Uk - self.U_ref) / self.U_scale
        control_cost = U_tilde ** 2

        # ----------------------------------------------------
        # 4) Control-increment term
        # ----------------------------------------------------
        if self.prev_action is None:
            delta_control_cost = 0.0
        else:
            U_prev_tilde = (self.prev_action - self.U_ref) / self.U_scale
            delta_control_cost = (U_tilde - U_prev_tilde) ** 2

        reward_value = (
            -self.alpha * tracking_cost
            -self.beta * state_change_cost
            -self.gamma * control_cost
            -self.delta * delta_control_cost
        )

        if terminate:
            reward_value -= self.terminal_tracking_weight * tracking_cost

        self.prev_H = Hk.copy()
        self.prev_Q = Qk.copy()
        self.prev_action = Uk

        self.last_terms = {
            "tracking_cost": float(tracking_cost),
            "state_change_cost": float(state_change_cost),
            "control_cost": float(control_cost),
            "delta_control_cost": float(delta_control_cost),
        }

        return float(reward_value)


# ============================================================
# 7. New Hadamard DeepONet prior + ordinary SAC FNN modules
# ============================================================

def preprocess_obs(obs: torch.Tensor) -> torch.Tensor:
    if not isinstance(obs, torch.Tensor):
        obs = torch.as_tensor(obs)
    return obs.float()


class BranchNet(nn.Module):
    def __init__(self, shape=N, output_dim=LATENT_DIM):
        super().__init__()
        self.shape = int(shape)
        self.output_dim = int(output_dim)

        self.conv1 = nn.Conv1d(
            in_channels=2,
            out_channels=16,
            kernel_size=5,
            stride=2,
        )
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(
            in_channels=16,
            out_channels=32,
            kernel_size=5,
            stride=2,
        )

        with torch.no_grad():
            tmp = torch.zeros(1, 2, self.shape)
            tmp = self.relu(self.conv1(tmp))
            tmp = self.relu(self.conv2(tmp))
            flat_dim = tmp.reshape(1, -1).shape[1]

        self.fc1 = nn.Linear(flat_dim, self.output_dim)

    def forward(self, x):
        x = preprocess_obs(x)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        if x.shape[1] != 2 * self.shape:
            raise ValueError(
                f"Branch input dimension is {x.shape[1]}, expected {2 * self.shape}."
            )

        x = x.reshape(x.shape[0], 2, self.shape)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = self.relu(self.fc1(x))
        return x


class TrunkNet(nn.Module):
    def __init__(self, dim_x=1, width=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_x, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, width),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x.float())


class DeepONetModified(nn.Module):
    """
    New Hadamard-fusion DeepONet controller:

        b(s) in R^p,
        t(x) in R^p,
        z(s,x) = b(s) * t(x) in R^p,
        U_prior = f_ctrl(z) in R.

    The output is the physical downstream gate U_L, not a normalized SAC action.
    """

    def __init__(self, shape=N, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = int(latent_dim)

        self.branch = BranchNet(shape, output_dim=self.latent_dim)
        self.trunk = TrunkNet(dim_x=1, width=self.latent_dim)

        self.fc3 = nn.Linear(self.latent_dim, 32)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, 1)

    def extract_latent(self, obs, grid):
        branch_feature = self.branch(obs)
        trunk_feature = self.trunk(grid)

        if trunk_feature.shape[0] == 1:
            trunk_feature = trunk_feature.expand(branch_feature.shape[0], -1)
        elif trunk_feature.shape[0] != branch_feature.shape[0]:
            raise ValueError(
                f"Branch batch {branch_feature.shape[0]} and trunk batch "
                f"{trunk_feature.shape[0]} do not match."
            )

        return branch_feature * trunk_feature

    def control_head(self, latent):
        value = self.relu(self.fc3(latent))
        value = self.relu(self.fc4(value))
        return self.fc5(value)

    def forward(self, obs, grid, return_latent=False):
        latent = self.extract_latent(obs, grid)
        physical_gate = self.control_head(latent)

        if return_latent:
            return physical_gate, latent
        return physical_gate


def _clean_state_dict_keys(state_dict):
    cleaned = {}
    prefixes = ["module.", "model.", "net1.", "deeponet."]

    for key, value in state_dict.items():
        new_key = key
        changed = True
        while changed:
            changed = False
            for prefix in prefixes:
                if new_key.startswith(prefix):
                    new_key = new_key[len(prefix):]
                    changed = True
        cleaned[new_key] = value

    return cleaned


def _load_new_deeponet_weights(
    net: DeepONetModified,
    pretrained_path: str,
    name: str,
):
    """Strictly load the new Hadamard-fusion DeepONet checkpoint."""
    path = Path(pretrained_path)

    if not path.exists():
        raise FileNotFoundError(
            f"Pretrained Hadamard DeepONet checkpoint not found for {name}: {path}"
        )

    state_dict = torch.load(str(path), map_location="cpu")
    if isinstance(state_dict, dict) and "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    if not isinstance(state_dict, dict):
        raise TypeError(
            f"Unsupported checkpoint type for {name}: "
            f"{type(state_dict).__name__}"
        )

    cleaned = _clean_state_dict_keys(state_dict)

    try:
        net.load_state_dict(cleaned, strict=True)
    except RuntimeError as exc:
        raise RuntimeError(
            "The checkpoint is incompatible with the new Hadamard-fusion "
            "DeepONet. The required architecture uses z=b*t and "
            "fc3=Linear(102,32).\n"
            f"Original error for {name}:\n{exc}"
        ) from exc

    print(f"Loaded pretrained Hadamard DeepONet for {name}: {path}")


def _freeze_all_deeponet(net: DeepONetModified, name: str):
    """Fully freeze the independent control-prior DeepONet."""
    for parameter in net.parameters():
        parameter.requires_grad = False

    trainable = sum(
        parameter.numel()
        for parameter in net.parameters()
        if parameter.requires_grad
    )
    total = sum(parameter.numel() for parameter in net.parameters())

    print(f"{name} is fully frozen: trainable={trainable}/{total}.")


def _physical_gate_to_normalized_action(physical_gate):
    scale = max_control_value - min_control_value
    if scale <= 0.0:
        raise ValueError(
            "max_control_value must be greater than min_control_value."
        )
    return 2.0 * (physical_gate - min_control_value) / scale - 1.0


class FrozenDeepONetHybridActor(Actor):
    """
    Ordinary SAC FNN actor plus one independent frozen new DeepONet prior.

    Method retained from the uploaded program:
        - SAC Actor uses the raw [H,Q] observation through the ordinary SB3 FNN;
        - SAC Critic remains the ordinary SB3 twin-Q FNN;
        - DeepONet is not used as an Actor/Critic feature extractor;
        - one independent new Hadamard DeepONet is fully frozen and only supplies
          the action prior.
    """

    def __init__(
        self,
        *args,
        alpha: float = 0.4,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        query_point: float = 1.0,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        initial_alpha = float(np.clip(alpha, 0.0, 1.0))
        self.register_buffer(
            "alpha_tensor",
            torch.tensor(initial_alpha, dtype=torch.float32),
        )

        self.query_point = float(query_point)
        self.control_deeponet = DeepONetModified(
            N,
            latent_dim=LATENT_DIM,
        )

        _load_new_deeponet_weights(
            self.control_deeponet,
            pretrained_path,
            "frozen control-prior DeepONet",
        )
        _freeze_all_deeponet(
            self.control_deeponet,
            "Frozen control-prior DeepONet",
        )

        self.register_buffer(
            "deeponet_grid",
            torch.tensor([[self.query_point]], dtype=torch.float32),
        )

        self.last_prior_physical = None
        self.last_deep_action = None
        self.last_rl_action = None
        self.last_hybrid_action = None

    @property
    def alpha(self) -> float:
        return float(self.alpha_tensor.detach().cpu().item())

    def set_alpha(self, alpha: float):
        alpha_value = float(np.clip(alpha, 0.0, 1.0))
        with torch.no_grad():
            self.alpha_tensor.fill_(alpha_value)

    def _deep_action(self, obs: torch.Tensor) -> torch.Tensor:
        """Compute the normalized prior action from the frozen new DeepONet."""
        raw_obs = preprocess_obs(obs)
        if raw_obs.dim() == 1:
            raw_obs = raw_obs.unsqueeze(0)

        if raw_obs.shape[1] != RAW_STATE_DIM:
            raise ValueError(
                f"Raw observation dimension is {raw_obs.shape[1]}, "
                f"expected {RAW_STATE_DIM}."
            )

        with torch.no_grad():
            physical_gate = self.control_deeponet(
                raw_obs,
                self.deeponet_grid.to(raw_obs.device),
            )

            # The new DeepONet outputs physical U_L.
            physical_gate = torch.clamp(
                physical_gate,
                min_control_value,
                max_control_value,
            )
            normalized_prior = _physical_gate_to_normalized_action(
                physical_gate
            )
            normalized_prior = torch.clamp(
                normalized_prior,
                -1.0,
                1.0,
            )

        self.last_prior_physical = physical_gate.detach()
        return normalized_prior

    def _hybridize(
        self,
        rl_action: torch.Tensor,
        obs: torch.Tensor,
    ) -> torch.Tensor:
        prior_action = self._deep_action(obs)
        alpha = self.alpha_tensor.to(
            device=rl_action.device,
            dtype=rl_action.dtype,
        )

        hybrid_action = (
            (1.0 - alpha) * prior_action
            + alpha * rl_action
        )
        hybrid_action = torch.clamp(
            hybrid_action,
            -1.0,
            1.0,
        )

        self.last_deep_action = prior_action.detach()
        self.last_rl_action = rl_action.detach()
        self.last_hybrid_action = hybrid_action.detach()

        return hybrid_action

    def forward(
        self,
        obs: torch.Tensor,
        deterministic: bool = False,
    ) -> torch.Tensor:
        # Standard Actor.forward uses ordinary SB3 flattened raw-state features.
        rl_action = super().forward(
            obs,
            deterministic=deterministic,
        )
        return self._hybridize(rl_action, obs)

    def action_log_prob(self, obs: torch.Tensor):
        # Keep the uploaded method: log_prob belongs to the SAC FNN component.
        rl_action, log_prob = super().action_log_prob(obs)
        hybrid_action = self._hybridize(rl_action, obs)
        return hybrid_action, log_prob


class FrozenDeepONetHybridSACPolicy(SACPolicy):
    """
    SAC policy with ordinary Actor/Critic FNNs and one frozen action-prior
    DeepONet inside the Actor.

    The Critic is intentionally not overridden. It is the standard SB3
    twin-Q critic and receives the raw flattened [H,Q] state together with the
    hybrid action.
    """

    def __init__(
        self,
        *args,
        hybrid_alpha_initial: float = 0.4,
        deeponet_pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        **kwargs,
    ):
        self.hybrid_alpha_initial = float(hybrid_alpha_initial)
        self.deeponet_pretrained_path = deeponet_pretrained_path
        super().__init__(*args, **kwargs)

    def make_actor(self, features_extractor=None):
        actor_kwargs = self._update_features_extractor(
            self.actor_kwargs,
            features_extractor,
        )
        return FrozenDeepONetHybridActor(
            **actor_kwargs,
            alpha=self.hybrid_alpha_initial,
            pretrained_path=self.deeponet_pretrained_path,
            query_point=1.0,
        ).to(self.device)

    def set_alpha(self, alpha: float):
        self.actor.set_alpha(alpha)


# ============================================================
# 8. Optimizer helper
# ============================================================

def set_sac_fnn_optimizers(model):
    """
    Optimize only the ordinary SAC FNN parameters.

    The independent new DeepONet prior is fully frozen and is excluded from
    the Actor optimizer. The Critic is the ordinary SB3 twin-Q FNN.
    """
    actor_parameters = [
        parameter
        for parameter in model.actor.parameters()
        if parameter.requires_grad
    ]
    critic_parameters = [
        parameter
        for parameter in model.critic.parameters()
        if parameter.requires_grad
    ]

    model.actor.optimizer = torch.optim.Adam(
        actor_parameters,
        lr=learning_rate,
        eps=1e-5,
    )
    model.critic.optimizer = torch.optim.Adam(
        critic_parameters,
        lr=learning_rate,
        eps=1e-5,
    )

    actor_count = sum(
        parameter.numel()
        for parameter in actor_parameters
    )
    critic_count = sum(
        parameter.numel()
        for parameter in critic_parameters
    )
    frozen_prior_count = sum(
        parameter.numel()
        for parameter in model.actor.control_deeponet.parameters()
    )

    print("Ordinary SAC FNN optimizers are set.")
    print(f"Actor trainable parameters={actor_count}")
    print(f"Critic trainable parameters={critic_count}")
    print(
        f"Frozen new DeepONet prior parameters={frozen_prior_count}, "
        "optimized=0"
    )


class HybridSAC(SAC):
    """Compatibility subclass; the action hybridization is implemented in policy."""
    pass


# ============================================================
# 9. Hybrid alpha scheduling
# ============================================================

def alpha_schedule(step: int) -> float:
    """
    alpha is the SAC weight:
        alpha small -> DeepONet prior dominates,
        alpha large -> SAC compensation dominates.
    """
    if step < 10000:
        return 0.1
    if step < 20000:
        return 0.5
    return 0.9


class AlphaCallback(BaseCallback):
    def _on_training_start(self):
        alpha = alpha_schedule(self.num_timesteps)
        self.model.policy.set_alpha(alpha)
        print(f"Initial hybrid alpha = {alpha}")

    def _on_step(self):
        alpha = alpha_schedule(self.num_timesteps)
        self.model.policy.set_alpha(alpha)
        return True


# ============================================================
# 10. Environment creation
# ============================================================

def make_env():
    p, X, dx, H_eq, Q_eq = build_collection_matched_reference()

    # Constructor fallback is the reference itself. Every reset then samples
    # the same Chebyshev initial-condition distribution as data collection.
    H_init = H_eq.copy()
    Q_init = Q_eq.copy()

    reward_class = CACTrackingSmoothReward(
        H_eq=H_eq,
        Q_eq=Q_eq,
        delta_t=control_sample_rate,
        H_scale=H_scale,
        Q_scale=Q_scale,
        U_scale=U_scale,
        alpha=reward_alpha,
        beta=reward_beta,
        gamma=reward_gamma,
        delta=reward_delta,
        tracking_w_H=tracking_w_H,
        tracking_w_Q=tracking_w_Q,
        change_w_H=change_w_H,
        change_w_Q=change_w_Q,
        U_ref=U_EQ_VALUE,
        terminal_tracking_weight=terminal_tracking_weight,
        truncate_penalty=truncate_penalty,
    )

    base_env = CACSaintVenantPDE1D(
        params=p,
        X_grid=X,
        Hbar=H_eq,
        Qbar=Q_eq,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=random_initial_condition_func,
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=reward_class,
        # SAC emits normalized actions; the environment maps them to physical U_L.
        normalize=True,
        # The SAC observation is the raw physical state [H,Q].
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    env = base_env

    # Expose the exact reference for diagnostics and test-program reuse.
    env.reference_X = X
    env.reference_H = H_eq
    env.reference_Q = Q_eq
    env.reference_dx = dx

    print(
        "Collection-matched ODE reference: "
        f"Hbar(0)={H_eq[0]:.9f}, Hbar(L)={H_eq[-1]:.9f}, "
        f"Qbar={Q_eq[0]:.9f}, dx={dx:.6f}"
    )

    return env


# ============================================================
# 11. Training
# ============================================================

def train_with_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    env = make_env()

    reset_obs, _ = env.reset(seed=seed)
    print(f"Raw observation shape: {reset_obs.shape}")

    if reset_obs.shape != (RAW_STATE_DIM,):
        raise RuntimeError(
            f"Expected raw observation shape {(RAW_STATE_DIM,)}, "
            f"got {reset_obs.shape}."
        )

    policy_kwargs = dict(
        deeponet_pretrained_path=DEEPNET_PRETRAINED_PATH,
        hybrid_alpha_initial=alpha_schedule(0),
        net_arch=dict(pi=[256, 256], qf=[256, 256]),
        optimizer_kwargs={"eps": 1e-6},
        share_features_extractor=False,
    )

    model = HybridSAC(
        policy=FrozenDeepONetHybridSACPolicy,
        policy_kwargs=policy_kwargs,
        env=env,
        learning_rate=learning_rate,
        buffer_size=buffer_size,
        batch_size=batch_size,
        tau=tau,
        gamma=sac_gamma,
        #ent_coef=ent_coef,
        #learning_starts=learning_starts,
        verbose=1,
        tensorboard_log=f"./tbsacnoxv/seed_{seed}/",
    )

    set_sac_fnn_optimizers(model)

    checkpoint_callback = CheckpointCallback(
        save_freq=10000,
        save_path=f"./sacnoxv/seed_{seed}/",
        name_prefix="sac_fnn_frozen_new_deeponet_prior_raw_HQ",
        save_replay_buffer=True,
        save_vecnormalize=True,
    )

    alpha_callback = AlphaCallback()
    callbacks = CallbackList([alpha_callback, checkpoint_callback])

    start_time = time.perf_counter()

    model.learn(
        total_timesteps=total_timesteps,
        callback=callbacks,
    )

    training_time = time.perf_counter() - start_time
    hours = int(training_time // 3600)
    minutes = int((training_time % 3600) // 60)
    seconds = training_time % 60

    print("\n" + "=" * 70)
    print(f"Seed {seed} training completed")
    print(
        f"Training time for this seed: "
        f"{hours:02d} h {minutes:02d} min {seconds:06.2f} s"
    )
    print(f"Total training time: {training_time:.2f} seconds")
    print("=" * 70)

    save_path = (
        f"./sacnoxv/seed_{seed}/"
        "sac_fnn_frozen_new_deeponet_prior_raw_HQ_final"
    )
    model.save(save_path)
    env.close()

    return model, save_path


def main():
    print("Ordinary SAC-FNN + frozen new Hadamard-DeepONet action prior")
    print("=" * 90)
    print(f"N={N}, T={T}, dt={dt}, control_sample_rate={control_sample_rate}")
    print(f"Physical gate U_L in [{min_control_value}, {max_control_value}]")
    print("Observation: raw physical [H_1,...,H_N,Q_1,...,Q_N]")
    print(
        f"Observation dimension={RAW_STATE_DIM}, "
        #f"ordinary SAC FNN input dimension={SAC_FNN_INPUT_DIM}"
    )
    print("Actor: ordinary SAC FNN; no DeepONet feature extraction")
    print("Critic: ordinary twin-Q FNN; no DeepONet feature extraction")
    print("Prior controller: one independent fully frozen new DeepONet")
    print("DeepONet input: raw [H,Q]")
    print("DeepONet output: physical U_L pretrained from y_ul.dat")
    print("DeepONet fusion: z=b(s)*t(L), followed by 102->32->16->1")
    print("Hybrid action: a=(1-alpha_hybrid)*a_prior+alpha_hybrid*a_SAC")
    print(f"DeepONet checkpoint: {DEEPNET_PRETRAINED_PATH}")
    print(
        f"Reference: Qbar={Q_BAR_FIXED} fixed; Hbar solved by steady ODE "
        f"(mode={REFERENCE_MODE})"
    )
    print(
        "Reward: -alpha*tracking - beta*state_change "
        "- gamma*control - delta*delta_control"
    )
    print(
        f"Reward weights: alpha={reward_alpha}, beta={reward_beta}, "
        f"gamma={reward_gamma}, delta={reward_delta}"
    )
    print(
        f"Tracking H/Q weights=({tracking_w_H}, {tracking_w_Q}); "
        f"state-change H/Q weights=({change_w_H}, {change_w_Q})"
    )
    print(f"Reward scales: H={H_scale}, Q={Q_scale}, U={U_scale}")
    print("Cumulative-error state: removed")
    print(
        "Chebyshev initial conditions: "
        f"order={CHEB_ORDER}, A_H={H_CHEB_AMPLITUDE}, "
        f"A_Q={Q_CHEB_AMPLITUDE}, decay={CHEB_COEFF_DECAY}"
    )
    print(
        f"Learning rates: SAC={learning_rate}, "
        f"frozen DeepONet={deeponet_learning_rate}"
    )

    for index, seed in enumerate(seeds):
        print(f"\n===== Training {index + 1}/{len(seeds)}, seed={seed} =====")
        _, save_path = train_with_seed(seed)
        print(f"Saved model: {save_path}")


if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-
"""
Hybrid Hadamard-DeepONet SAC training for the CAC Saint-Venant system.

This program is aligned with the latest DeepONet data collector:

1. Reference/equilibrium:
       Qbar(x) = 11.93 m^3/s is prescribed and spatially constant;
       Hbar(L) is obtained from the downstream equilibrium gate relation;
       Hbar(x) is obtained by integrating the steady Saint-Venant ODE
       backward from x=L to x=0.
2. Initial conditions:
       H(x,0) and Q(x,0) are sampled around Hbar(x), Qbar(x) by truncated
       Chebyshev expansions, using the same default order/amplitudes as the
       collector: order=6, A_H=0.30 m, A_Q=2.50 m^3/s, decay=2.0.
3. State supplied to SAC:
       [H_1,...,H_N,Q_1,...,Q_N].
   H and Q are raw physical states, not error states. No cumulative-error state
   is appended to the observation.
4. Reward:
       The reward uses H/Q tracking, inter-sample state-change, control,
       and delta-control costs.
5. DeepONet:
       raw [H,Q] -> physical gate U_L,
   with Hadamard Branch-Trunk fusion z=b(s)*t(L) and a 102->32->16->1
   control head. The checkpoint must have been pretrained using y_ul.dat.
6. Hybrid action:
       a_hybrid = (1-alpha_hybrid)*a_prior + alpha_hybrid*a_SAC,
   where the physical DeepONet gate prior is mapped to [-1,1] before fusion.

Required external module:
    cac_saint_venant_env_no_bs.py
"""

from __future__ import annotations

import os
import random
import time
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import torch
from scipy.integrate import solve_ivp
from gymnasium import spaces
from torch import nn

from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import (
    BaseCallback,
    CallbackList,
    CheckpointCallback,
)
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.sac.policies import Actor, SACPolicy

from cac_saint_venant_env_no_bs import CACParams, CACSaintVenantPDE1D

try:
    from pde_control_gym.src.rewards.base_reward import BaseReward
except Exception:
    class BaseReward:
        pass


# ============================================================
# 1. Simulation parameters
# ============================================================

N = 102
T = 40000.0
dt = 5.0
control_sample_rate = 400.0

max_state_value = 1e6

# The PDE environment maps normalized SAC actions a in [-1,1] to the physical
# downstream gate U_L in this interval. These limits match the data collector.
min_control_value = 0.15
max_control_value = 0.40

# State/reward normalization scales. H and Q remain raw in the observation;
# these scales are used only inside the reward calculation.
H_scale = 0.30
Q_scale = 3.0
U_scale = 0.10

obs_clip = 50.0

cfl = 0.45
check_cfl = True
well_balanced = True


# ============================================================
# 2. Collection-matched equilibrium and initial-condition settings
# ============================================================

Q_BAR_FIXED = 11.93
REFERENCE_MODE = "ode"
U_EQ_VALUE = 0.30

CHEB_ORDER = 6
H_CHEB_AMPLITUDE = 0.30
Q_CHEB_AMPLITUDE = 2.50
CHEB_COEFF_DECAY = 2.0


# ============================================================
# 3. SAC parameters
# ============================================================

total_timesteps = int(1e5)
learning_rate = 1e-4

# Use a much smaller learning rate for trainable DeepONet parameters.
deeponet_learning_rate = 1e-6

buffer_size = 100000
batch_size = 256
tau = 0.005
sac_gamma = 0.995
ent_coef = 0.005
learning_starts = 5000

seeds = [40]

LATENT_DIM = 102
RAW_STATE_DIM = 2 * N
SAC_FEATURE_DIM = LATENT_DIM

# This checkpoint must come from the Hadamard-fusion DeepONet pretraining code.
DEEPNET_PRETRAINED_PATH = "modedlNet10000QqQqQ14444_hadamard.pt"


# ============================================================
# 4. Reward parameters
# ============================================================

reward_alpha = 1
reward_beta = 0.1
reward_gamma = 0.05
reward_delta =0.1

# Separate H/Q weights for tracking and inter-sample state-change terms.
tracking_w_H = 1.0
tracking_w_Q = 1
change_w_H = 1.0
change_w_Q = 1

terminal_tracking_weight = 0.0
truncate_penalty = -200.0



# ============================================================
# 5. CAC physical parameters, ODE reference, and Chebyshev initial conditions
# ============================================================

def make_cac_table_params(
    q_bar: float = Q_BAR_FIXED,
    h_bar_0: float = 3.80,
    h_bar_L: float = 4.70,
) -> CACParams:
    """Create the CAC parameter object used by collection and RL training."""
    return CACParams(
        L=11239.0,
        B=40.0,
        g=9.81,
        I=7.92e-5,
        K=33.0,
        mu0=0.4,
        muL=0.4,
        z_up=5.00,
        U0_bar=0.1548,
        U_L_bar=U_EQ_VALUE,
        h_s=4.094,
        Q_bar=float(q_bar),
        H_bar_0=float(h_bar_0),
        H_bar_L=float(h_bar_L),
        H_min=0.20,
        max_state_value=1e10,
    )


def make_grid(p: CACParams, n_grid: int) -> Tuple[np.ndarray, float]:
    X = np.linspace(0.0, p.L, n_grid)
    dx = p.L / (n_grid - 1)
    return X, dx


def downstream_equilibrium_depth_from_q(Q: float, p: CACParams) -> float:
    """Compute Hbar(L) from the downstream equilibrium gate relation."""
    Q = float(Q)
    if not np.isfinite(Q) or Q <= 0.0:
        raise ValueError("The equilibrium discharge must be positive and finite.")

    kinetic_head = (
        Q**2 / (2.0 * p.g * p.B**2 * p.muL**2)
    ) ** (1.0 / 3.0)
    return float(p.h_s + p.U_L_bar + kinetic_head)


def integrate_steady_fixed_q_ode(
    X: np.ndarray,
    p: CACParams,
    Q_candidate: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Integrate the steady Saint-Venant ODE backward from x=L to x=0."""
    X = np.asarray(X, dtype=float)
    if X.ndim != 1 or X.size < 2:
        raise ValueError("X must be a one-dimensional grid with at least two points.")
    if not np.all(np.diff(X) > 0.0):
        raise ValueError("X must be strictly increasing.")

    Q_candidate = float(Q_candidate)
    if not np.isfinite(Q_candidate) or Q_candidate <= 0.0:
        raise ValueError("Q_candidate must be positive and finite.")

    Hbar_L = downstream_equilibrium_depth_from_q(Q_candidate, p)

    def rhs(_x: float, state: np.ndarray) -> np.ndarray:
        H = float(state[0])
        Q = float(state[1])

        if not np.isfinite(H) or not np.isfinite(Q):
            raise FloatingPointError("Non-finite state in steady reference ODE.")
        if H <= float(p.H_min):
            raise FloatingPointError("Steady reference ODE violates minimum depth.")

        area = p.B * H
        hydraulic_radius = p.B * H / (p.B + 2.0 * H)
        friction_slope = Q * abs(Q) / (
            p.K**2 * area**2 * hydraulic_radius ** (4.0 / 3.0)
        )

        denominator = p.g * p.B * H - Q**2 / (p.B * H**2)
        if abs(denominator) <= 1e-10:
            raise FloatingPointError(
                "Steady ODE approaches the critical-flow singularity."
            )

        dHdx = p.g * p.B * H * (p.I - friction_slope) / denominator
        if not np.isfinite(dHdx):
            raise FloatingPointError("Non-finite dHbar/dx in steady ODE.")

        return np.asarray([dHdx, 0.0], dtype=float)

    sol = solve_ivp(
        fun=rhs,
        t_span=(float(X[-1]), float(X[0])),
        y0=np.asarray([Hbar_L, Q_candidate], dtype=float),
        t_eval=X[::-1],
        rtol=1e-9,
        atol=1e-11,
        max_step=p.L / 1000.0,
    )

    if not sol.success:
        raise RuntimeError(f"Failed to solve steady ODE: {sol.message}")
    if sol.y.shape[1] != X.size:
        raise RuntimeError("Steady ODE did not return all requested grid points.")

    Hbar = np.asarray(sol.y[0][::-1], dtype=float)
    Qbar = np.asarray(sol.y[1][::-1], dtype=float)

    if not np.all(np.isfinite(Hbar)) or not np.all(np.isfinite(Qbar)):
        raise FloatingPointError("Steady reference contains NaN or Inf.")
    if float(np.min(Hbar)) <= float(p.H_min):
        raise RuntimeError("Steady Hbar profile violates the minimum depth.")

    return Hbar, Qbar


def solve_steady_reference_ode(
    X: np.ndarray,
    p: CACParams,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Solve Hbar(x) for prescribed spatially constant Qbar=11.93."""
    q_bar_fixed = float(q_bar_fixed)
    Hbar, _ = integrate_steady_fixed_q_ode(X, p, q_bar_fixed)
    Qbar = np.full_like(X, q_bar_fixed, dtype=float)
    return Hbar, Qbar, q_bar_fixed


def make_reference_profiles(
    X: np.ndarray,
    p: CACParams,
    reference: str = REFERENCE_MODE,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    if reference in {"ode", "strict"}:
        return solve_steady_reference_ode(X, p, q_bar_fixed=q_bar_fixed)
    if reference == "linear":
        Hbar = p.H_bar_0 + (p.H_bar_L - p.H_bar_0) * (X / p.L)
        Qbar = np.full_like(X, q_bar_fixed, dtype=float)
        return Hbar, Qbar, float(q_bar_fixed)
    raise ValueError("reference must be 'ode', 'strict', or 'linear'.")


def build_collection_matched_reference():
    """Build one internally consistent p, X, Hbar, Qbar tuple."""
    nominal_p = make_cac_table_params(q_bar=Q_BAR_FIXED)
    X, dx = make_grid(nominal_p, N)
    Hbar, Qbar, q_equilibrium = make_reference_profiles(
        X,
        nominal_p,
        reference=REFERENCE_MODE,
        q_bar_fixed=Q_BAR_FIXED,
    )

    p = make_cac_table_params(
        q_bar=q_equilibrium,
        h_bar_0=float(Hbar[0]),
        h_bar_L=float(Hbar[-1]),
    )
    return p, X, dx, Hbar, Qbar


def sample_truncated_chebyshev(
    xi: np.ndarray,
    rng,
    order: int,
    max_amplitude: float,
    coefficient_decay: float,
) -> np.ndarray:
    """Sample the same smooth truncated-Chebyshev perturbation as collection."""
    if order < 0:
        raise ValueError("CHEB_ORDER must be nonnegative.")
    if max_amplitude < 0.0:
        raise ValueError("Chebyshev amplitude must be nonnegative.")
    if coefficient_decay < 0.0:
        raise ValueError("CHEB_COEFF_DECAY must be nonnegative.")
    if max_amplitude == 0.0:
        return np.zeros_like(xi, dtype=float)

    mode_index = np.arange(order + 1, dtype=float)
    coefficient_scale = 1.0 / np.power(mode_index + 1.0, coefficient_decay)
    coefficients = rng.normal(loc=0.0, scale=coefficient_scale)

    perturbation = np.polynomial.chebyshev.chebval(xi, coefficients)
    max_abs = float(np.max(np.abs(perturbation)))
    if max_abs <= 1e-12:
        return np.zeros_like(xi, dtype=float)

    realized_amplitude = rng.uniform(0.35, 1.0) * max_amplitude
    return perturbation * (realized_amplitude / max_abs)


def random_initial_condition_func(X, Hbar, Qbar, rng):
    """Generate H0,Q0 with the same Chebyshev distribution as collection."""
    X = np.asarray(X, dtype=float)
    Hbar = np.asarray(Hbar, dtype=float)
    Qbar = np.asarray(Qbar, dtype=float)
    xi = 2.0 * X / float(X[-1]) - 1.0

    # Rejection sampling preserves the polynomial shape; no pointwise clipping.
    for _ in range(100):
        H_perturbation = sample_truncated_chebyshev(
            xi,
            rng,
            CHEB_ORDER,
            H_CHEB_AMPLITUDE,
            CHEB_COEFF_DECAY,
        )
        Q_perturbation = sample_truncated_chebyshev(
            xi,
            rng,
            CHEB_ORDER,
            Q_CHEB_AMPLITUDE,
            CHEB_COEFF_DECAY,
        )

        H0 = Hbar + H_perturbation
        Q0 = Qbar + Q_perturbation

        finite = np.all(np.isfinite(H0)) and np.all(np.isfinite(Q0))
        valid_depth = float(np.min(H0)) > 0.20
        valid_discharge = float(np.min(Q0)) > 0.0

        if finite and valid_depth and valid_discharge:
            return H0, Q0

    raise RuntimeError(
        "Failed to generate a valid Chebyshev initial condition after 100 "
        "attempts. Reduce H_CHEB_AMPLITUDE or Q_CHEB_AMPLITUDE."
    )


# ============================================================
# 6. Reward class
# ============================================================

class CACTrackingSmoothReward(BaseReward):
    """
    Discrete reward at the control instant t_k:

        r_k = -alpha * tracking_cost
              -beta  * state_change_cost
              -gamma * control_cost
              -delta * delta_control_cost.

    Definitions:
        tracking_cost
            = tracking_w_H * mean(((H_k-H_eq)/H_scale)^2)
            + tracking_w_Q * mean(((Q_k-Q_eq)/Q_scale)^2)

        state_change_cost
            = change_w_H * mean(((H_k-H_{k-1})/H_scale)^2)
            + change_w_Q * mean(((Q_k-Q_{k-1})/Q_scale)^2)

        control_cost
            = ((U_k-U_eq)/U_scale)^2

        delta_control_cost
            = (((U_k-U_eq)/U_scale)
               -((U_{k-1}-U_eq)/U_scale))^2

    The state-change term is an inter-sample increment penalty. It is not a
    time derivative because it is not divided by the control interval.

    Spatial means are used instead of unnormalized sums so that the reward scale
    does not grow with the number of grid points N.
    """

    def __init__(
        self,
        H_eq,
        Q_eq,
        delta_t=control_sample_rate,
        H_scale=0.10,
        Q_scale=1.50,
        U_scale=0.10,
        alpha=1.0,
        beta=0.01,
        gamma=0.05,
        delta=0.50,
        tracking_w_H=1.0,
        tracking_w_Q=1.0,
        change_w_H=1.0,
        change_w_Q=0.50,
        U_ref=U_EQ_VALUE,
        terminal_tracking_weight=0.0,
        truncate_penalty=-200.0,
    ):
        self.H_eq = np.asarray(H_eq, dtype=float)
        self.Q_eq = np.asarray(Q_eq, dtype=float)
        self.N = len(self.H_eq)

        self.delta_t = float(delta_t)
        self.H_scale = float(H_scale)
        self.Q_scale = float(Q_scale)
        self.U_scale = float(U_scale)

        self.alpha = float(alpha)
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.delta = float(delta)

        self.tracking_w_H = float(tracking_w_H)
        self.tracking_w_Q = float(tracking_w_Q)
        self.change_w_H = float(change_w_H)
        self.change_w_Q = float(change_w_Q)

        self.U_ref = float(U_ref)
        self.terminal_tracking_weight = float(terminal_tracking_weight)
        self.truncate_penalty = float(truncate_penalty)

        # Previous values at the preceding reward/control call.
        self.prev_H: Optional[np.ndarray] = None
        self.prev_Q: Optional[np.ndarray] = None
        self.prev_action: Optional[float] = None

        # Diagnostics.
        self.last_terms = {
            "tracking_cost": 0.0,
            "state_change_cost": 0.0,
            "control_cost": 0.0,
            "delta_control_cost": 0.0,
        }

    def _reset_memory(self):
        self.prev_H = None
        self.prev_Q = None
        self.prev_action = None

    def _split_state(self, state_vec):
        state_vec = np.asarray(state_vec, dtype=float).reshape(-1)
        if state_vec.size < 2 * self.N:
            raise ValueError(
                f"PDE state length is {state_vec.size}, expected at least {2 * self.N}."
            )
        H = state_vec[:self.N]
        Q = state_vec[self.N:2 * self.N]
        return H, Q

    def _action_value(self, action):
        # The original environment passes the physical downstream gate to reward.
        if action is None:
            return self.U_ref
        value = float(np.asarray(action).reshape(-1)[0])
        return float(
            np.nan_to_num(
                value,
                nan=self.U_ref,
                posinf=self.U_ref,
                neginf=self.U_ref,
            )
        )

    def reward(
        self,
        uVec=None,
        time_index=None,
        terminate=None,
        truncate=None,
        action=None,
        control_sample_rate=None,
    ):
        if uVec is None or time_index is None:
            return 0.0

        # A new episode normally starts with a small time index.
        if int(time_index) <= 1:
            self._reset_memory()

        if truncate:
            self._reset_memory()
            return float(self.truncate_penalty)

        Hk, Qk = self._split_state(uVec[time_index])

        if (
            not np.all(np.isfinite(Hk))
            or not np.all(np.isfinite(Qk))
            or np.any(Hk <= 0.0)
        ):
            self._reset_memory()
            return float(self.truncate_penalty)

        # ----------------------------------------------------
        # 1) Tracking term
        # ----------------------------------------------------
        H_error_tilde = (Hk - self.H_eq) / self.H_scale
        Q_error_tilde = (Qk - self.Q_eq) / self.Q_scale

        tracking_cost = (
            self.tracking_w_H * np.mean(H_error_tilde ** 2)
            + self.tracking_w_Q * np.mean(Q_error_tilde ** 2)
        )

        # ----------------------------------------------------
        # 2) Inter-sample state-change term
        # ----------------------------------------------------
        if self.prev_H is None or self.prev_Q is None:
            state_change_cost = 0.0
        else:
            delta_H_tilde = (Hk - self.prev_H) / self.H_scale
            delta_Q_tilde = (Qk - self.prev_Q) / self.Q_scale

            state_change_cost = (
                self.change_w_H * np.mean(delta_H_tilde ** 2)
                + self.change_w_Q * np.mean(delta_Q_tilde ** 2)
            )

        # ----------------------------------------------------
        # 3) Control-magnitude term
        # ----------------------------------------------------
        Uk = self._action_value(action)
        U_tilde = (Uk - self.U_ref) / self.U_scale
        control_cost = U_tilde ** 2

        # ----------------------------------------------------
        # 4) Control-increment term
        # ----------------------------------------------------
        if self.prev_action is None:
            delta_control_cost = 0.0
        else:
            U_prev_tilde = (self.prev_action - self.U_ref) / self.U_scale
            delta_control_cost = (U_tilde - U_prev_tilde) ** 2

        reward_value = (
            -self.alpha * tracking_cost
            -self.beta * state_change_cost
            -self.gamma * control_cost
            -self.delta * delta_control_cost
        )

        if terminate:
            reward_value -= self.terminal_tracking_weight * tracking_cost

        self.prev_H = Hk.copy()
        self.prev_Q = Qk.copy()
        self.prev_action = Uk

        self.last_terms = {
            "tracking_cost": float(tracking_cost),
            "state_change_cost": float(state_change_cost),
            "control_cost": float(control_cost),
            "delta_control_cost": float(delta_control_cost),
        }

        return float(reward_value)


# ============================================================
# 7. DeepONet controller and hybrid SAC modules
# ============================================================

def preprocess_obs(obs: torch.Tensor) -> torch.Tensor:
    if not isinstance(obs, torch.Tensor):
        obs = torch.as_tensor(obs)
    return obs.float()


class BranchNet(nn.Module):
    def __init__(self, shape=N, output_dim=LATENT_DIM):
        super().__init__()
        self.shape = int(shape)
        self.output_dim = int(output_dim)

        self.conv1 = nn.Conv1d(
            in_channels=2,
            out_channels=16,
            kernel_size=5,
            stride=2,
        )
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(
            in_channels=16,
            out_channels=32,
            kernel_size=5,
            stride=2,
        )

        with torch.no_grad():
            tmp = torch.zeros(1, 2, self.shape)
            tmp = self.relu(self.conv1(tmp))
            tmp = self.relu(self.conv2(tmp))
            flat_dim = tmp.reshape(1, -1).shape[1]

        self.fc1 = nn.Linear(flat_dim, self.output_dim)

    def forward(self, x):
        x = preprocess_obs(x)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        if x.shape[1] != 2 * self.shape:
            raise ValueError(
                f"Branch input dimension is {x.shape[1]}, expected {2 * self.shape}."
            )

        x = x.reshape(x.shape[0], 2, self.shape)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = self.relu(self.fc1(x))
        return x


class TrunkNet(nn.Module):
    def __init__(self, dim_x=1, width=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_x, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, width),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x.float())


class DeepONetModified(nn.Module):
    """
    Hadamard-fusion DeepONet controller.

        b(s) in R^p,
        t(x) in R^p,
        z(s,x) = b(s) * t(x) in R^p,
        U_prior = f_ctrl(z) in R.
    """

    def __init__(self, shape=N, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = int(latent_dim)

        self.branch = BranchNet(shape, output_dim=self.latent_dim)
        self.trunk = TrunkNet(dim_x=1, width=self.latent_dim)

        # Hadamard fusion preserves latent_dim features.
        self.fc3 = nn.Linear(self.latent_dim, 32)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, 1)

    def extract_latent(self, obs, grid):
        b = self.branch(obs)       # [batch, latent_dim]
        t = self.trunk(grid)       # [n_query, latent_dim]

        if t.shape[0] == 1:
            t = t.expand(b.shape[0], -1)
        elif t.shape[0] != b.shape[0]:
            raise ValueError(
                f"Branch batch {b.shape[0]} and trunk batch {t.shape[0]} do not match."
            )

        z = b * t                  # Hadamard product
        return z

    def control_head(self, z):
        y = self.relu(self.fc3(z))
        y = self.relu(self.fc4(y))
        y = self.fc5(y)
        return y

    def forward(self, obs, grid, return_latent=False):
        z = self.extract_latent(obs, grid)
        u_prior_physical = self.control_head(z)

        if return_latent:
            return u_prior_physical, z

        return u_prior_physical


class HybridCACDeepONetFeatureExtractor(BaseFeaturesExtractor):
    """
    Return the Hadamard-fused DeepONet latent feature.

    Input observation and DeepONet input:
        [H_1,...,H_N,Q_1,...,Q_N]

    SAC feature:
        [z_1,...,z_p]
    """

    def __init__(
        self,
        observation_space: spaces.Box,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        query_point: float = 1.0,
        freeze_deeponet: bool = False,
    ):
        if features_dim != SAC_FEATURE_DIM:
            raise ValueError(
                f"features_dim must be {SAC_FEATURE_DIM}, got {features_dim}."
            )

        super().__init__(observation_space, features_dim)

        self.query_point = float(query_point)
        self.net1 = DeepONetModified(N, latent_dim=LATENT_DIM)

        # Normalized action used by the hybrid SAC actor.
        self.u_deep_norm: Optional[torch.Tensor] = None

        # Physical prior gate retained for diagnostics.
        self.u_prior_physical: Optional[torch.Tensor] = None

        self.register_buffer(
            "grid",
            torch.tensor([[self.query_point]], dtype=torch.float32),
        )

        self._load_pretrained(pretrained_path)
        self._apply_freeze(freeze_all=freeze_deeponet)

    def _apply_freeze(self, freeze_all=False):
        if freeze_all:
            for param in self.net1.parameters():
                param.requires_grad = False
            print("All DeepONet parameters are frozen.")
            return

        # 1) Freeze Branch convolution layers.
        for param in self.net1.branch.conv1.parameters():
            param.requires_grad = False
        for param in self.net1.branch.conv2.parameters():
            param.requires_grad = False

        # 2) Keep Branch fc1 trainable.
        for param in self.net1.branch.fc1.parameters():
            param.requires_grad = True

        # 3) Freeze the whole Trunk, then unfreeze its last Linear layer.
        for param in self.net1.trunk.parameters():
            param.requires_grad = False

        last_linear = None
        for module in reversed(list(self.net1.trunk.net)):
            if isinstance(module, nn.Linear):
                last_linear = module
                break

        if last_linear is not None:
            for param in last_linear.parameters():
                param.requires_grad = True

        # 4) Keep the post-fusion control MLP trainable.
        for layer in [self.net1.fc3, self.net1.fc4, self.net1.fc5]:
            for param in layer.parameters():
                param.requires_grad = True

        trainable = sum(
            p.numel() for p in self.net1.parameters() if p.requires_grad
        )
        total = sum(p.numel() for p in self.net1.parameters())

        print(
            f"DeepONet partial freeze: trainable={trainable}/{total} "
            f"({100.0 * trainable / max(total, 1):.2f}%)."
        )
        print("Frozen: Branch conv1/conv2 and previous Trunk layers.")
        print("Trainable: Branch fc1, last Trunk Linear, and fc3/fc4/fc5.")

    @staticmethod
    def _clean_state_dict_keys(state_dict):
        cleaned = {}
        prefixes = ["module.", "model.", "net1.", "deeponet."]

        for key, value in state_dict.items():
            new_key = key
            changed = True
            while changed:
                changed = False
                for prefix in prefixes:
                    if new_key.startswith(prefix):
                        new_key = new_key[len(prefix):]
                        changed = True
            cleaned[new_key] = value

        return cleaned

    def _load_pretrained(self, pretrained_path: str):
        path = Path(pretrained_path)

        if not path.exists():
            raise FileNotFoundError(
                f"Pretrained Hadamard DeepONet checkpoint not found: {path}"
            )

        state_dict = torch.load(str(path), map_location="cpu")

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]

        if not isinstance(state_dict, dict):
            raise TypeError(
                f"Unsupported checkpoint type: {type(state_dict).__name__}"
            )

        cleaned = self._clean_state_dict_keys(state_dict)

        try:
            self.net1.load_state_dict(cleaned, strict=True)
        except RuntimeError as exc:
            raise RuntimeError(
                "The checkpoint is incompatible with the Hadamard-fusion "
                "DeepONet. Re-pretrain DeepONet with fc3=Linear(102,32) and "
                "z=b*t before running this RL script.\n"
                f"Original error:\n{exc}"
            ) from exc

        print(f"Loaded pretrained Hadamard DeepONet: {path}")

    @staticmethod
    def _physical_gate_to_normalized_action(u_physical):
        scale = max_control_value - min_control_value
        if scale <= 0.0:
            raise ValueError("max_control_value must be greater than min_control_value.")
        return 2.0 * (u_physical - min_control_value) / scale - 1.0

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        obs = preprocess_obs(observations)
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)

        if obs.shape[1] != RAW_STATE_DIM:
            raise ValueError(
                f"Raw observation dimension is {obs.shape[1]}, "
                f"expected {RAW_STATE_DIM}."
            )

        u_prior_physical, z = self.net1(
            obs,
            self.grid.to(obs.device),
            return_latent=True,
        )

        # Clip the physical prior to the admissible gate interval before mapping.
        u_prior_physical = torch.clamp(
            u_prior_physical,
            min_control_value,
            max_control_value,
        )
        self.u_prior_physical = u_prior_physical

        u_deep_norm = self._physical_gate_to_normalized_action(u_prior_physical)
        self.u_deep_norm = torch.clamp(u_deep_norm, -1.0, 1.0)

        # Representation-level transfer: use the fused DeepONet feature z only.
        return z.view(z.size(0), -1)


class HybridCACActor(Actor):
    """SAC actor whose normalized output is blended with DeepONet prior action."""

    def __init__(self, *args, alpha: float = 0.4, **kwargs):
        super().__init__(*args, **kwargs)
        init_alpha = float(np.clip(alpha, 0.0, 1.0))

        self.register_buffer(
            "alpha_tensor",
            torch.tensor(init_alpha, dtype=torch.float32),
        )

        self.last_deep_action = None
        self.last_rl_action = None
        self.last_hybrid_action = None

    @property
    def alpha(self) -> float:
        return float(self.alpha_tensor.detach().cpu().item())

    def set_alpha(self, alpha: float):
        alpha_value = float(np.clip(alpha, 0.0, 1.0))
        with torch.no_grad():
            self.alpha_tensor.fill_(alpha_value)

    def _deep_action(self, obs: torch.Tensor) -> torch.Tensor:
        u_deep = getattr(self.features_extractor, "u_deep_norm", None)

        if u_deep is None or u_deep.shape[0] != obs.shape[0]:
            _ = self.features_extractor(obs)
            u_deep = self.features_extractor.u_deep_norm

        return torch.clamp(u_deep, -1.0, 1.0)

    def _hybridize(self, rl_action: torch.Tensor, obs: torch.Tensor) -> torch.Tensor:
        u_deep = self._deep_action(obs)

        alpha = self.alpha_tensor.to(
            device=rl_action.device,
            dtype=rl_action.dtype,
        )

        hybrid = (1.0 - alpha) * u_deep + alpha * rl_action
        hybrid = torch.clamp(hybrid, -1.0, 1.0)

        self.last_deep_action = u_deep.detach()
        self.last_rl_action = rl_action.detach()
        self.last_hybrid_action = hybrid.detach()

        return hybrid

    def forward(self, obs: torch.Tensor, deterministic: bool = False):
        rl_action = super().forward(obs, deterministic=deterministic)
        return self._hybridize(rl_action, obs)

    def action_log_prob(self, obs: torch.Tensor):
        # The log probability is that of the SAC stochastic component.
        # The deterministic affine hybrid mapping is not included in log_prob.
        rl_action, log_prob = super().action_log_prob(obs)
        hybrid_action = self._hybridize(rl_action, obs)
        return hybrid_action, log_prob


class HybridCACCritic(nn.Module):
    """Twin-Q critic using the Hadamard-fused DeepONet feature."""

    def __init__(
        self,
        observation_space: spaces.Box,
        action_space: spaces.Box,
        hidden_dim: int = 256,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
    ):
        super().__init__()

        self.features_extractor = HybridCACDeepONetFeatureExtractor(
            observation_space,
            features_dim=features_dim,
            pretrained_path=pretrained_path,
            query_point=1.0,
            freeze_deeponet=False,
        )

        action_dim = int(np.prod(action_space.shape))
        input_dim = features_dim + action_dim

        self.q1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.q2 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def set_training_mode(self, mode: bool):
        self.train(mode)

    def forward(self, obs: torch.Tensor, actions: torch.Tensor):
        features = self.features_extractor(obs)
        q_input = torch.cat([features, actions], dim=1)
        return self.q1(q_input), self.q2(q_input)


class HybridCACSACPolicy(SACPolicy):
    """SAC policy with Hadamard DeepONet representation/control transfer."""

    def __init__(
        self,
        *args,
        hybrid_alpha_initial: float = 0.4,
        deeponet_pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        **kwargs,
    ):
        self.hybrid_alpha_initial = float(hybrid_alpha_initial)
        self.deeponet_pretrained_path = deeponet_pretrained_path

        kwargs.setdefault(
            "features_extractor_class",
            HybridCACDeepONetFeatureExtractor,
        )
        kwargs.setdefault(
            "features_extractor_kwargs",
            dict(
                features_dim=SAC_FEATURE_DIM,
                pretrained_path=deeponet_pretrained_path,
                query_point=1.0,
                freeze_deeponet=False,
            ),
        )

        super().__init__(*args, **kwargs)

    def make_actor(self, features_extractor=None):
        kwargs = self._update_features_extractor(
            self.actor_kwargs,
            features_extractor,
        )
        return HybridCACActor(
            **kwargs,
            alpha=self.hybrid_alpha_initial,
        ).to(self.device)

    def make_critic(self, features_extractor=None):
        return HybridCACCritic(
            self.observation_space,
            self.action_space,
            hidden_dim=256,
            features_dim=SAC_FEATURE_DIM,
            pretrained_path=self.deeponet_pretrained_path,
        ).to(self.device)

    def set_alpha(self, alpha: float):
        self.actor.set_alpha(alpha)


# ============================================================
# 8. Optimizer helper: separate DeepONet learning rate
# ============================================================

def _make_param_groups_with_deeponet_lr(module: nn.Module):
    deeponet_params = []
    other_params = []

    for name, param in module.named_parameters():
        if not param.requires_grad:
            continue

        if "features_extractor.net1" in name or name.startswith("net1."):
            deeponet_params.append(param)
        else:
            other_params.append(param)

    groups = []

    if other_params:
        groups.append(
            {
                "params": other_params,
                "lr": learning_rate,
                "is_deeponet_group": False,
            }
        )

    if deeponet_params:
        groups.append(
            {
                "params": deeponet_params,
                "lr": deeponet_learning_rate,
                "is_deeponet_group": True,
            }
        )

    return groups


def set_deeponet_small_lr_optimizers(model):
    actor_groups = _make_param_groups_with_deeponet_lr(model.actor)
    critic_groups = _make_param_groups_with_deeponet_lr(model.critic)

    model.actor.optimizer = torch.optim.Adam(actor_groups, eps=1e-5)
    model.critic.optimizer = torch.optim.Adam(critic_groups, eps=1e-5)

    n_actor_deep = sum(
        p.numel()
        for group in actor_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_actor_other = sum(
        p.numel()
        for group in actor_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_deep = sum(
        p.numel()
        for group in critic_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_other = sum(
        p.numel()
        for group in critic_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )

    print("Custom optimizers with separate DeepONet learning rate are set.")
    print(f"Actor params: DeepONet={n_actor_deep}, other={n_actor_other}")
    print(f"Critic params: DeepONet={n_critic_deep}, other={n_critic_other}")
    print(
        f"Base SAC learning_rate={learning_rate}, "
        f"DeepONet learning_rate={deeponet_learning_rate}"
    )


class HybridSAC(SAC):
    """Preserve the smaller learning rate of DeepONet parameter groups."""

    def _update_learning_rate(self, optimizers):
        super()._update_learning_rate(optimizers)

        if not isinstance(optimizers, (list, tuple)):
            optimizers = [optimizers]

        current_lr = self.lr_schedule(self._current_progress_remaining)

        for optimizer in optimizers:
            for group in optimizer.param_groups:
                if group.get("is_deeponet_group", False):
                    group["lr"] = deeponet_learning_rate
                elif "is_deeponet_group" in group:
                    group["lr"] = current_lr


# ============================================================
# 9. Hybrid alpha scheduling
# ============================================================

def alpha_schedule(step: int) -> float:
    """
    alpha is the SAC weight:
        alpha small -> DeepONet prior dominates,
        alpha large -> SAC compensation dominates.
    """
    if step < 10000:
        return 0.1
    if step < 20000:
        return 0.5
    return 0.9


class AlphaCallback(BaseCallback):
    def _on_training_start(self):
        alpha = alpha_schedule(self.num_timesteps)
        self.model.policy.set_alpha(alpha)
        print(f"Initial hybrid alpha = {alpha}")

    def _on_step(self):
        alpha = alpha_schedule(self.num_timesteps)
        self.model.policy.set_alpha(alpha)
        return True


# ============================================================
# 10. Environment creation
# ============================================================

def make_env():
    p, X, dx, H_eq, Q_eq = build_collection_matched_reference()

    # Constructor fallback is the reference itself. Every reset then samples
    # the same Chebyshev initial-condition distribution as data collection.
    H_init = H_eq.copy()
    Q_init = Q_eq.copy()

    reward_class = CACTrackingSmoothReward(
        H_eq=H_eq,
        Q_eq=Q_eq,
        delta_t=control_sample_rate,
        H_scale=H_scale,
        Q_scale=Q_scale,
        U_scale=U_scale,
        alpha=reward_alpha,
        beta=reward_beta,
        gamma=reward_gamma,
        delta=reward_delta,
        tracking_w_H=tracking_w_H,
        tracking_w_Q=tracking_w_Q,
        change_w_H=change_w_H,
        change_w_Q=change_w_Q,
        U_ref=U_EQ_VALUE,
        terminal_tracking_weight=terminal_tracking_weight,
        truncate_penalty=truncate_penalty,
    )

    base_env = CACSaintVenantPDE1D(
        params=p,
        X_grid=X,
        Hbar=H_eq,
        Qbar=Q_eq,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=random_initial_condition_func,
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=reward_class,
        # SAC emits normalized actions; the environment maps them to physical U_L.
        normalize=True,
        # The SAC observation is the raw physical state [H,Q].
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    env = base_env

    # Expose the exact reference for diagnostics and test-program reuse.
    env.reference_X = X
    env.reference_H = H_eq
    env.reference_Q = Q_eq
    env.reference_dx = dx

    print(
        "Collection-matched ODE reference: "
        f"Hbar(0)={H_eq[0]:.9f}, Hbar(L)={H_eq[-1]:.9f}, "
        f"Qbar={Q_eq[0]:.9f}, dx={dx:.6f}"
    )

    return env


# ============================================================
# 11. Training
# ============================================================

def train_with_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    env = make_env()

    reset_obs, _ = env.reset(seed=seed)
    print(f"Raw observation shape: {reset_obs.shape}")

    if reset_obs.shape != (RAW_STATE_DIM,):
        raise RuntimeError(
            f"Expected raw observation shape {(RAW_STATE_DIM,)}, "
            f"got {reset_obs.shape}."
        )

    policy_kwargs = dict(
        deeponet_pretrained_path=DEEPNET_PRETRAINED_PATH,
        hybrid_alpha_initial=alpha_schedule(0),
        net_arch=dict(pi=[256, 256], qf=[256, 256]),
        optimizer_kwargs={"eps": 1e-6},
        share_features_extractor=False,
    )

    model = HybridSAC(
        policy=HybridCACSACPolicy,
        policy_kwargs=policy_kwargs,
        env=env,
        learning_rate=learning_rate,
        buffer_size=buffer_size,
        batch_size=batch_size,
        tau=tau,
        gamma=sac_gamma,
        #ent_coef=ent_coef,
        #learning_starts=learning_starts,
        verbose=1,
        tensorboard_log=f"./tbsactx/seed_{seed}/",
    )

    set_deeponet_small_lr_optimizers(model)

    checkpoint_callback = CheckpointCallback(
        save_freq=10000,
        save_path=f"./sactx/seed_{seed}/",
        name_prefix="sac_cac_hybrid_yul_ode_cheb_raw_HQ",
        save_replay_buffer=True,
        save_vecnormalize=True,
    )

    alpha_callback = AlphaCallback()
    callbacks = CallbackList([alpha_callback, checkpoint_callback])

    start_time = time.perf_counter()

    model.learn(
        total_timesteps=total_timesteps,
        callback=callbacks,
    )

    training_time = time.perf_counter() - start_time
    hours = int(training_time // 3600)
    minutes = int((training_time % 3600) // 60)
    seconds = training_time % 60

    print("\n" + "=" * 70)
    print(f"Seed {seed} training completed")
    print(
        f"Training time for this seed: "
        f"{hours:02d} h {minutes:02d} min {seconds:06.2f} s"
    )
    print(f"Total training time: {training_time:.2f} seconds")
    print("=" * 70)

    save_path = (
        f"./sactx/seed_{seed}/"
        "sac_cac_hybrid_yul_ode_cheb_raw_HQ_final"
    )
    model.save(save_path)
    env.close()

    return model, save_path


def main():
    print("Hybrid Hadamard-DeepONet SAC: ODE reference + Chebyshev IC")
    print("=" * 90)
    print(f"N={N}, T={T}, dt={dt}, control_sample_rate={control_sample_rate}")
    print(f"Physical gate U_L in [{min_control_value}, {max_control_value}]")
    print("Observation: raw physical [H_1,...,H_N,Q_1,...,Q_N]")
    print(
        f"Observation dimension={RAW_STATE_DIM}, "
        f"SAC feature dimension={SAC_FEATURE_DIM}"
    )
    print("DeepONet input: raw [H,Q]")
    print("DeepONet output: physical U_L pretrained from y_ul.dat")
    print("DeepONet fusion: z=b(s)*t(L), followed by 102->32->16->1")
    print("Hybrid action: a=(1-alpha_hybrid)*a_prior+alpha_hybrid*a_SAC")
    print(f"DeepONet checkpoint: {DEEPNET_PRETRAINED_PATH}")
    print(
        "Transfer strategy: freeze Branch conv1/conv2 and previous Trunk "
        "layers; train Branch fc1, last Trunk Linear, fc3/fc4/fc5"
    )
    print(
        f"Reference: Qbar={Q_BAR_FIXED} fixed; Hbar solved by steady ODE "
        f"(mode={REFERENCE_MODE})"
    )
    print(
        "Reward: -alpha*tracking - beta*state_change "
        "- gamma*control - delta*delta_control"
    )
    print(
        f"Reward weights: alpha={reward_alpha}, beta={reward_beta}, "
        f"gamma={reward_gamma}, delta={reward_delta}"
    )
    print(
        f"Tracking H/Q weights=({tracking_w_H}, {tracking_w_Q}); "
        f"state-change H/Q weights=({change_w_H}, {change_w_Q})"
    )
    print(f"Reward scales: H={H_scale}, Q={Q_scale}, U={U_scale}")
    print("Cumulative-error state: removed")
    print(
        "Chebyshev initial conditions: "
        f"order={CHEB_ORDER}, A_H={H_CHEB_AMPLITUDE}, "
        f"A_Q={Q_CHEB_AMPLITUDE}, decay={CHEB_COEFF_DECAY}"
    )
    print(
        f"Learning rates: SAC={learning_rate}, "
        f"DeepONet={deeponet_learning_rate}"
    )

    for index, seed in enumerate(seeds):
        print(f"\n===== Training {index + 1}/{len(seeds)}, seed={seed} =====")
        _, save_path = train_with_seed(seed)
        print(f"Saved model: {save_path}")


if __name__ == "__main__":
    main()


In [1]:
# -*- coding: utf-8 -*-
"""
Pure SAC training for the CAC Saint-Venant system.

This program keeps the original direct-gate SAC controller and does not embed
DeepONet. The training environment is aligned with the latest Hybrid
DeepONet-SAC program in the following four aspects:

1. Reference/equilibrium:
       Qbar(x) = 11.93 m^3/s is prescribed and spatially constant;
       Hbar(L) is obtained from the downstream equilibrium gate relation;
       Hbar(x) is obtained by integrating the steady Saint-Venant ODE
       backward from x=L to x=0.
2. Initial conditions:
       H(x,0) and Q(x,0) are sampled around Hbar(x), Qbar(x) by truncated
       Chebyshev expansions with order=6, A_H=0.30 m, A_Q=2.50 m^3/s,
       and coefficient decay=2.0.
3. State supplied to SAC:
       [H_1,...,H_N,Q_1,...,Q_N].
   H and Q are raw physical states. There is no normalized-error observation
   and no cumulative-error state.
4. Reward:
       r_k = -alpha*tracking
             -beta*state_rate
             -gamma*control
             -delta*delta_control.

The controller remains a standard Stable-Baselines3 SAC MLP policy. No
DeepONet feature extractor, prior action, hybrid action, or transfer-learning
module is used.

Required external module:
    cac_saint_venant_env_no_bs.py
"""

from __future__ import annotations

import os
import random
import time
from typing import Optional, Tuple

import numpy as np
import torch
from scipy.integrate import solve_ivp
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback

from cac_saint_venant_env_no_bs import CACParams, CACSaintVenantPDE1D

try:
    from pde_control_gym.src.rewards.base_reward import BaseReward
except Exception:
    class BaseReward:
        pass


# ============================================================
# 1. Simulation parameters
# ============================================================

N = 102
T = 40000.0
dt = 5.0
control_sample_rate = 400.0

max_state_value = 1e6

# SAC outputs normalized actions a in [-1,1]. The environment maps them to the
# physical downstream gate U_L in this interval.
min_control_value = 0.15
max_control_value = 0.40

# State/reward normalization scales. H and Q remain raw in the observation;
# these scales are used only inside the reward calculation.
H_scale = 0.30
Q_scale = 3.0
U_scale = 0.10

obs_clip = 50.0

cfl = 0.45
check_cfl = True
well_balanced = True


# ============================================================
# 2. ODE equilibrium and Chebyshev initial-condition settings
# ============================================================

Q_BAR_FIXED = 11.93
REFERENCE_MODE = "ode"
U_EQ_VALUE = 0.30

# The following values directly control the training initial-condition range.
CHEB_ORDER = 6
H_CHEB_AMPLITUDE = 0.30
Q_CHEB_AMPLITUDE = 2.50
CHEB_COEFF_DECAY = 2.0


# ============================================================
# 3. SAC parameters
# ============================================================

total_timesteps = int(1e5)
learning_rate = 1e-4
buffer_size = 100000
batch_size = 256
tau = 0.005
sac_gamma = 0.995
ent_coef = 0.005
learning_starts = 5000

seeds = [42, 43, 44, 45, 46]

RAW_STATE_DIM = 2 * N


# ============================================================
# 4. Reward parameters
# ============================================================

reward_alpha = 1
reward_beta = 0.1
reward_gamma = 0.05
reward_delta =0.1

# Separate H/Q weights for tracking and inter-sample state-change terms.
tracking_w_H = 1.0
tracking_w_Q = 1
change_w_H = 1.0
change_w_Q = 1

terminal_tracking_weight = 0.0
truncate_penalty = -200.0



# ============================================================
# 5. Physical parameters and steady ODE reference
# ============================================================

def make_cac_table_params(
    q_bar: float = Q_BAR_FIXED,
    h_bar_0: float = 3.80,
    h_bar_L: float = 4.70,
) -> CACParams:
    """Create a CAC parameter object consistent with the selected reference."""
    return CACParams(
        L=11239.0,
        B=40.0,
        g=9.81,
        I=7.92e-5,
        K=33.0,
        mu0=0.4,
        muL=0.4,
        z_up=5.00,
        U0_bar=0.1548,
        U_L_bar=U_EQ_VALUE,
        h_s=4.094,
        Q_bar=float(q_bar),
        H_bar_0=float(h_bar_0),
        H_bar_L=float(h_bar_L),
        H_min=0.20,
        max_state_value=1e10,
    )


def make_grid(p: CACParams, n_grid: int) -> Tuple[np.ndarray, float]:
    X = np.linspace(0.0, p.L, n_grid)
    dx = p.L / (n_grid - 1)
    return X, dx


def downstream_equilibrium_depth_from_q(Q: float, p: CACParams) -> float:
    """Compute Hbar(L) from the downstream equilibrium gate relation."""
    Q = float(Q)
    if not np.isfinite(Q) or Q <= 0.0:
        raise ValueError("The equilibrium discharge must be positive and finite.")

    kinetic_head = (
        Q**2 / (2.0 * p.g * p.B**2 * p.muL**2)
    ) ** (1.0 / 3.0)
    return float(p.h_s + p.U_L_bar + kinetic_head)


def integrate_steady_fixed_q_ode(
    X: np.ndarray,
    p: CACParams,
    Q_candidate: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Integrate the steady Saint-Venant ODE backward from x=L to x=0."""
    X = np.asarray(X, dtype=float)
    if X.ndim != 1 or X.size < 2:
        raise ValueError("X must be a one-dimensional grid with at least two points.")
    if not np.all(np.diff(X) > 0.0):
        raise ValueError("X must be strictly increasing.")

    Q_candidate = float(Q_candidate)
    if not np.isfinite(Q_candidate) or Q_candidate <= 0.0:
        raise ValueError("Q_candidate must be positive and finite.")

    Hbar_L = downstream_equilibrium_depth_from_q(Q_candidate, p)

    def rhs(_x: float, state: np.ndarray) -> np.ndarray:
        H = float(state[0])
        Q = float(state[1])

        if not np.isfinite(H) or not np.isfinite(Q):
            raise FloatingPointError("Non-finite state in steady reference ODE.")
        if H <= float(p.H_min):
            raise FloatingPointError("Steady reference ODE violates minimum depth.")

        area = p.B * H
        hydraulic_radius = p.B * H / (p.B + 2.0 * H)
        friction_slope = Q * abs(Q) / (
            p.K**2 * area**2 * hydraulic_radius ** (4.0 / 3.0)
        )

        denominator = p.g * p.B * H - Q**2 / (p.B * H**2)
        if abs(denominator) <= 1e-10:
            raise FloatingPointError(
                "Steady ODE approaches the critical-flow singularity."
            )

        dHdx = p.g * p.B * H * (p.I - friction_slope) / denominator
        if not np.isfinite(dHdx):
            raise FloatingPointError("Non-finite dHbar/dx in steady ODE.")

        return np.asarray([dHdx, 0.0], dtype=float)

    sol = solve_ivp(
        fun=rhs,
        t_span=(float(X[-1]), float(X[0])),
        y0=np.asarray([Hbar_L, Q_candidate], dtype=float),
        t_eval=X[::-1],
        rtol=1e-9,
        atol=1e-11,
        max_step=p.L / 1000.0,
    )

    if not sol.success:
        raise RuntimeError(f"Failed to solve steady ODE: {sol.message}")
    if sol.y.shape[1] != X.size:
        raise RuntimeError("Steady ODE did not return all requested grid points.")

    Hbar = np.asarray(sol.y[0][::-1], dtype=float)
    Qbar = np.asarray(sol.y[1][::-1], dtype=float)

    if not np.all(np.isfinite(Hbar)) or not np.all(np.isfinite(Qbar)):
        raise FloatingPointError("Steady reference contains NaN or Inf.")
    if float(np.min(Hbar)) <= float(p.H_min):
        raise RuntimeError("Steady Hbar profile violates the minimum depth.")

    return Hbar, Qbar


def solve_steady_reference_ode(
    X: np.ndarray,
    p: CACParams,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Solve Hbar(x) for prescribed spatially constant Qbar=11.93."""
    q_bar_fixed = float(q_bar_fixed)
    Hbar, _ = integrate_steady_fixed_q_ode(X, p, q_bar_fixed)
    Qbar = np.full_like(X, q_bar_fixed, dtype=float)
    return Hbar, Qbar, q_bar_fixed


def make_reference_profiles(
    X: np.ndarray,
    p: CACParams,
    reference: str = REFERENCE_MODE,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    if reference in {"ode", "strict"}:
        return solve_steady_reference_ode(X, p, q_bar_fixed=q_bar_fixed)
    if reference == "linear":
        Hbar = p.H_bar_0 + (p.H_bar_L - p.H_bar_0) * (X / p.L)
        Qbar = np.full_like(X, q_bar_fixed, dtype=float)
        return Hbar, Qbar, float(q_bar_fixed)
    raise ValueError("reference must be 'ode', 'strict', or 'linear'.")


def build_collection_matched_reference():
    """Build an internally consistent p, X, Hbar, and Qbar tuple."""
    nominal_p = make_cac_table_params(q_bar=Q_BAR_FIXED)
    X, dx = make_grid(nominal_p, N)
    Hbar, Qbar, q_equilibrium = make_reference_profiles(
        X,
        nominal_p,
        reference=REFERENCE_MODE,
        q_bar_fixed=Q_BAR_FIXED,
    )

    p = make_cac_table_params(
        q_bar=q_equilibrium,
        h_bar_0=float(Hbar[0]),
        h_bar_L=float(Hbar[-1]),
    )
    return p, X, dx, Hbar, Qbar


# ============================================================
# 6. Chebyshev initial conditions
# ============================================================

def sample_truncated_chebyshev(
    xi: np.ndarray,
    rng,
    order: int,
    max_amplitude: float,
    coefficient_decay: float,
) -> np.ndarray:
    """Generate one smooth truncated-Chebyshev perturbation."""
    if order < 0:
        raise ValueError("CHEB_ORDER must be nonnegative.")
    if max_amplitude < 0.0:
        raise ValueError("Chebyshev amplitude must be nonnegative.")
    if coefficient_decay < 0.0:
        raise ValueError("CHEB_COEFF_DECAY must be nonnegative.")
    if max_amplitude == 0.0:
        return np.zeros_like(xi, dtype=float)

    mode_index = np.arange(order + 1, dtype=float)
    coefficient_scale = 1.0 / np.power(mode_index + 1.0, coefficient_decay)
    coefficients = rng.normal(loc=0.0, scale=coefficient_scale)

    perturbation = np.polynomial.chebyshev.chebval(xi, coefficients)
    max_abs = float(np.max(np.abs(perturbation)))
    if max_abs <= 1e-12:
        return np.zeros_like(xi, dtype=float)

    # Match the data collector/new Hybrid program: the realized maximum
    # amplitude is randomly selected from 35% to 100% of the configured bound.
    realized_amplitude = rng.uniform(0.35, 1.0) * max_amplitude
    return perturbation * (realized_amplitude / max_abs)


def random_initial_condition_func(X, Hbar, Qbar, rng):
    """Generate H0,Q0 with the same Chebyshev distribution as the new program."""
    X = np.asarray(X, dtype=float)
    Hbar = np.asarray(Hbar, dtype=float)
    Qbar = np.asarray(Qbar, dtype=float)
    xi = 2.0 * X / float(X[-1]) - 1.0

    # Reject the whole candidate instead of clipping individual grid points, so
    # the smooth Chebyshev shape is preserved.
    for _ in range(100):
        H_perturbation = sample_truncated_chebyshev(
            xi,
            rng,
            CHEB_ORDER,
            H_CHEB_AMPLITUDE,
            CHEB_COEFF_DECAY,
        )
        Q_perturbation = sample_truncated_chebyshev(
            xi,
            rng,
            CHEB_ORDER,
            Q_CHEB_AMPLITUDE,
            CHEB_COEFF_DECAY,
        )

        H0 = Hbar + H_perturbation
        Q0 = Qbar + Q_perturbation

        finite = np.all(np.isfinite(H0)) and np.all(np.isfinite(Q0))
        valid_depth = float(np.min(H0)) > 0.20
        valid_discharge = float(np.min(Q0)) > 0.0

        if finite and valid_depth and valid_discharge:
            return H0, Q0

    raise RuntimeError(
        "Failed to generate a valid Chebyshev initial condition after 100 "
        "attempts. Reduce H_CHEB_AMPLITUDE or Q_CHEB_AMPLITUDE."
    )


# ============================================================
# 7. Reward class
# ============================================================

class CACTrackingSmoothReward(BaseReward):
    """
    Discrete reward at the control instant t_k:

        r_k = -alpha * tracking_cost
              -beta  * state_change_cost
              -gamma * control_cost
              -delta * delta_control_cost.

    Definitions:
        tracking_cost
            = tracking_w_H * mean(((H_k-H_eq)/H_scale)^2)
            + tracking_w_Q * mean(((Q_k-Q_eq)/Q_scale)^2)

        state_change_cost
            = change_w_H * mean(((H_k-H_{k-1})/H_scale)^2)
            + change_w_Q * mean(((Q_k-Q_{k-1})/Q_scale)^2)

        control_cost
            = ((U_k-U_eq)/U_scale)^2

        delta_control_cost
            = (((U_k-U_eq)/U_scale)
               -((U_{k-1}-U_eq)/U_scale))^2

    The state-change term is an inter-sample increment penalty. It is not a
    time derivative because it is not divided by the control interval.

    Spatial means are used instead of unnormalized sums so that the reward scale
    does not grow with the number of grid points N.
    """

    def __init__(
        self,
        H_eq,
        Q_eq,
        delta_t=control_sample_rate,
        H_scale=0.10,
        Q_scale=1.50,
        U_scale=0.10,
        alpha=1.0,
        beta=0.01,
        gamma=0.05,
        delta=0.50,
        tracking_w_H=1.0,
        tracking_w_Q=1.0,
        change_w_H=1.0,
        change_w_Q=0.50,
        U_ref=U_EQ_VALUE,
        terminal_tracking_weight=0.0,
        truncate_penalty=-200.0,
    ):
        self.H_eq = np.asarray(H_eq, dtype=float)
        self.Q_eq = np.asarray(Q_eq, dtype=float)
        self.N = len(self.H_eq)

        self.delta_t = float(delta_t)
        self.H_scale = float(H_scale)
        self.Q_scale = float(Q_scale)
        self.U_scale = float(U_scale)

        self.alpha = float(alpha)
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.delta = float(delta)

        self.tracking_w_H = float(tracking_w_H)
        self.tracking_w_Q = float(tracking_w_Q)
        self.change_w_H = float(change_w_H)
        self.change_w_Q = float(change_w_Q)

        self.U_ref = float(U_ref)
        self.terminal_tracking_weight = float(terminal_tracking_weight)
        self.truncate_penalty = float(truncate_penalty)

        # Previous values at the preceding reward/control call.
        self.prev_H: Optional[np.ndarray] = None
        self.prev_Q: Optional[np.ndarray] = None
        self.prev_action: Optional[float] = None

        # Diagnostics.
        self.last_terms = {
            "tracking_cost": 0.0,
            "state_change_cost": 0.0,
            "control_cost": 0.0,
            "delta_control_cost": 0.0,
        }

    def _reset_memory(self):
        self.prev_H = None
        self.prev_Q = None
        self.prev_action = None

    def _split_state(self, state_vec):
        state_vec = np.asarray(state_vec, dtype=float).reshape(-1)
        if state_vec.size < 2 * self.N:
            raise ValueError(
                f"PDE state length is {state_vec.size}, expected at least {2 * self.N}."
            )
        H = state_vec[:self.N]
        Q = state_vec[self.N:2 * self.N]
        return H, Q

    def _action_value(self, action):
        # The original environment passes the physical downstream gate to reward.
        if action is None:
            return self.U_ref
        value = float(np.asarray(action).reshape(-1)[0])
        return float(
            np.nan_to_num(
                value,
                nan=self.U_ref,
                posinf=self.U_ref,
                neginf=self.U_ref,
            )
        )

    def reward(
        self,
        uVec=None,
        time_index=None,
        terminate=None,
        truncate=None,
        action=None,
        control_sample_rate=None,
    ):
        if uVec is None or time_index is None:
            return 0.0

        # A new episode normally starts with a small time index.
        if int(time_index) <= 1:
            self._reset_memory()

        if truncate:
            self._reset_memory()
            return float(self.truncate_penalty)

        Hk, Qk = self._split_state(uVec[time_index])

        if (
            not np.all(np.isfinite(Hk))
            or not np.all(np.isfinite(Qk))
            or np.any(Hk <= 0.0)
        ):
            self._reset_memory()
            return float(self.truncate_penalty)

        # ----------------------------------------------------
        # 1) Tracking term
        # ----------------------------------------------------
        H_error_tilde = (Hk - self.H_eq) / self.H_scale
        Q_error_tilde = (Qk - self.Q_eq) / self.Q_scale

        tracking_cost = (
            self.tracking_w_H * np.mean(H_error_tilde ** 2)
            + self.tracking_w_Q * np.mean(Q_error_tilde ** 2)
        )

        # ----------------------------------------------------
        # 2) Inter-sample state-change term
        # ----------------------------------------------------
        if self.prev_H is None or self.prev_Q is None:
            state_change_cost = 0.0
        else:
            delta_H_tilde = (Hk - self.prev_H) / self.H_scale
            delta_Q_tilde = (Qk - self.prev_Q) / self.Q_scale

            state_change_cost = (
                self.change_w_H * np.mean(delta_H_tilde ** 2)
                + self.change_w_Q * np.mean(delta_Q_tilde ** 2)
            )

        # ----------------------------------------------------
        # 3) Control-magnitude term
        # ----------------------------------------------------
        Uk = self._action_value(action)
        U_tilde = (Uk - self.U_ref) / self.U_scale
        control_cost = U_tilde ** 2

        # ----------------------------------------------------
        # 4) Control-increment term
        # ----------------------------------------------------
        if self.prev_action is None:
            delta_control_cost = 0.0
        else:
            U_prev_tilde = (self.prev_action - self.U_ref) / self.U_scale
            delta_control_cost = (U_tilde - U_prev_tilde) ** 2

        reward_value = (
            -self.alpha * tracking_cost
            -self.beta * state_change_cost
            -self.gamma * control_cost
            -self.delta * delta_control_cost
        )

        if terminate:
            reward_value -= self.terminal_tracking_weight * tracking_cost

        self.prev_H = Hk.copy()
        self.prev_Q = Qk.copy()
        self.prev_action = Uk

        self.last_terms = {
            "tracking_cost": float(tracking_cost),
            "state_change_cost": float(state_change_cost),
            "control_cost": float(control_cost),
            "delta_control_cost": float(delta_control_cost),
        }

        return float(reward_value)


# ============================================================
# 8. Environment creation
# ============================================================

def make_env():
    p, X, dx, H_eq, Q_eq = build_collection_matched_reference()

    # These constructor values are fallbacks only. Every reset uses the
    # Chebyshev initial-condition generator below.
    H_init = H_eq.copy()
    Q_init = Q_eq.copy()

    reward_class = CACTrackingSmoothReward(
        H_eq=H_eq,
        Q_eq=Q_eq,
        delta_t=control_sample_rate,
        H_scale=H_scale,
        Q_scale=Q_scale,
        U_scale=U_scale,
        alpha=reward_alpha,
        beta=reward_beta,
        gamma=reward_gamma,
        delta=reward_delta,
        tracking_w_H=tracking_w_H,
        tracking_w_Q=tracking_w_Q,
        change_w_H=change_w_H,
        change_w_Q=change_w_Q,
        U_ref=U_EQ_VALUE,
        terminal_tracking_weight=terminal_tracking_weight,
        truncate_penalty=truncate_penalty,
    )

    env = CACSaintVenantPDE1D(
        params=p,
        X_grid=X,
        Hbar=H_eq,
        Qbar=Q_eq,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=random_initial_condition_func,
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=reward_class,
        normalize=True,
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    # Expose the exact ODE reference for diagnostics and test-program reuse.
    env.reference_X = X
    env.reference_H = H_eq
    env.reference_Q = Q_eq
    env.reference_dx = dx

    print(
        "Collection-matched ODE reference: "
        f"Hbar(0)={H_eq[0]:.9f}, Hbar(L)={H_eq[-1]:.9f}, "
        f"Qbar={Q_eq[0]:.9f}, dx={dx:.6f}"
    )

    return env


# ============================================================
# 9. Training
# ============================================================

def train_with_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    env = make_env()

    reset_obs, _ = env.reset(seed=seed)
    reset_obs = np.asarray(reset_obs, dtype=np.float32).reshape(-1)
    print(f"Raw observation shape: {reset_obs.shape}")

    if reset_obs.shape != (RAW_STATE_DIM,):
        raise RuntimeError(
            f"Expected raw observation shape {(RAW_STATE_DIM,)}, "
            f"got {reset_obs.shape}."
        )

    # Standard SAC MLP only. No DeepONet feature extractor or hybrid action.
    policy_kwargs = dict(
        net_arch=dict(pi=[256, 256], qf=[256, 256]),
        optimizer_kwargs={"eps": 1e-6},
        share_features_extractor=False,
    )

    model = SAC(
        policy="MlpPolicy",
        #policy_kwargs=policy_kwargs,
        env=env,
        learning_rate=learning_rate,
        buffer_size=buffer_size,
        batch_size=batch_size,
        tau=tau,
        gamma=sac_gamma,
        # Keep these commented exactly as in the supplied program/new program.
        # ent_coef=ent_coef,
        # learning_starts=learning_starts,
        verbose=1,
        tensorboard_log=f"./tbsaczx/seed_{seed}/",
    )

    checkpoint_callback = CheckpointCallback(
        save_freq=10000,
        save_path=f"./saczx/seed_{seed}/",
        name_prefix="sac_cac_raw_hq_ode_cheb",
        save_replay_buffer=True,
        save_vecnormalize=True,
    )

    start_time = time.perf_counter()

    model.learn(
        total_timesteps=total_timesteps,
        callback=checkpoint_callback,
    )

    training_time = time.perf_counter() - start_time
    hours = int(training_time // 3600)
    minutes = int((training_time % 3600) // 60)
    seconds = training_time % 60

    print("\n" + "=" * 70)
    print(f"Seed {seed} training completed")
    print(
        f"Training time for this seed: "
        f"{hours:02d} h {minutes:02d} min {seconds:06.2f} s"
    )
    print(f"Total training time: {training_time:.2f} seconds")
    print("=" * 70)

    save_path = (
        f"./saczx/seed_{seed}/"
        "sac_cac_raw_hq_ode_cheb_final"
    )
    model.save(save_path)
    env.close()

    return model, save_path


def main():
    print("Pure SAC: raw H/Q state + ODE reference + Chebyshev initial conditions")
    print("=" * 90)
    print(f"N={N}, T={T}, dt={dt}, control_sample_rate={control_sample_rate}")
    print(f"Physical gate U_L in [{min_control_value}, {max_control_value}]")
    print("Controller: standard SAC MLP; DeepONet is not embedded")
    print("Observation: raw physical [H_1,...,H_N,Q_1,...,Q_N]")
    print(f"Observation dimension={RAW_STATE_DIM}")
    print(
        f"Reference: Qbar={Q_BAR_FIXED} fixed; Hbar solved by steady ODE "
        f"(mode={REFERENCE_MODE})"
    )
    print(
        "Reward: -alpha*tracking - beta*state_change "
        "- gamma*control - delta*delta_control"
    )
    print(
        f"Reward weights: alpha={reward_alpha}, beta={reward_beta}, "
        f"gamma={reward_gamma}, delta={reward_delta}"
    )
    print(
        f"Tracking H/Q weights=({tracking_w_H}, {tracking_w_Q}); "
        f"state-change H/Q weights=({change_w_H}, {change_w_Q})"
    )
    print(f"Reward scales: H={H_scale}, Q={Q_scale}, U={U_scale}")
    print(
        "Chebyshev initial conditions: "
        f"order={CHEB_ORDER}, A_H={H_CHEB_AMPLITUDE}, "
        f"A_Q={Q_CHEB_AMPLITUDE}, decay={CHEB_COEFF_DECAY}"
    )
    print(f"Learning rate={learning_rate}")

    for index, seed in enumerate(seeds):
        print(f"\n===== Training {index + 1}/{len(seeds)}, seed={seed} =====")
        _, save_path = train_with_seed(seed)
        print(f"Saved model: {save_path}")


if __name__ == "__main__":
    main()


Pure SAC: raw H/Q state + ODE reference + Chebyshev initial conditions
N=102, T=40000.0, dt=5.0, control_sample_rate=400.0
Physical gate U_L in [0.15, 0.4]
Controller: standard SAC MLP; DeepONet is not embedded
Observation: raw physical [H_1,...,H_N,Q_1,...,Q_N]
Observation dimension=204
Reference: Qbar=11.93 fixed; Hbar solved by steady ODE (mode=ode)
Reward: -alpha*tracking - beta*state_change - gamma*control - delta*delta_control
Reward weights: alpha=1, beta=0.1, gamma=0.05, delta=0.1
Tracking H/Q weights=(1.0, 1); state-change H/Q weights=(1.0, 1)
Reward scales: H=0.3, Q=3.0, U=0.1
Chebyshev initial conditions: order=6, A_H=0.3, A_Q=2.5, decay=2.0
Learning rate=0.0001

===== Training 1/5, seed=42 =====
Collection-matched ODE reference: Hbar(0)=3.818346502, Hbar(L)=4.698869496, Qbar=11.930000000, dx=111.277228
Raw observation shape: (204,)
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./tbsaczx/seed_42/SAC_1
--------------

In [2]:
# -*- coding: utf-8 -*-
"""
Hybrid Hadamard-DeepONet SAC training for the CAC Saint-Venant system.

All environment, equilibrium, initial-condition, state, reward, SAC, and
training settings are aligned with the latest TDPSAC program. The only method
change retained from Pasted code(22).py is the transfer/control organization:

1. Actor/Critic feature DeepONets
       raw [H,Q] -> Hadamard feature z=b(s)*t(L),
   with partial freezing:
       - freeze Branch conv1/conv2;
       - train Branch fc1;
       - freeze the Trunk except its last Linear layer;
       - train fc3/fc4/fc5.

2. Independent fixed control DeepONet
   The actor owns a second DeepONet loaded from the same new Hadamard checkpoint.
   It is fully frozen and is used only to generate the physical gate prior
   U_prior. Its output is clipped to [0.15,0.40] and mapped to [-1,1].

3. Hybrid action
       a_hybrid = (1-alpha_hybrid)*a_prior + alpha_hybrid*a_SAC.

Thus, unlike the latest TDPSAC program, the prior action comes from an
independent fully frozen control DeepONet rather than from the actor's trainable
feature DeepONet. The new DeepONet uses Hadamard Branch-Trunk fusion and the
102->32->16->1 physical-gate control head pretrained with y_ul.dat.

Required external module:
    cac_saint_venant_env_no_bs.py
"""

from __future__ import annotations

import os
import random
import time
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import torch
from scipy.integrate import solve_ivp
from gymnasium import spaces
from torch import nn

from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import (
    BaseCallback,
    CallbackList,
    CheckpointCallback,
)
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.sac.policies import Actor, SACPolicy

from cac_saint_venant_env_no_bs import CACParams, CACSaintVenantPDE1D

try:
    from pde_control_gym.src.rewards.base_reward import BaseReward
except Exception:
    class BaseReward:
        pass


# ============================================================
# 1. Simulation parameters
# ============================================================

N = 102
T = 40000.0
dt = 5.0
control_sample_rate = 400.0

max_state_value = 1e6

# The PDE environment maps normalized SAC actions a in [-1,1] to the physical
# downstream gate U_L in this interval. These limits match the data collector.
min_control_value = 0.15
max_control_value = 0.40

# State/reward normalization scales. H and Q remain raw in the observation;
# these scales are used only inside the reward calculation.
H_scale = 0.30
Q_scale = 3.0
U_scale = 0.10

obs_clip = 50.0

cfl = 0.45
check_cfl = True
well_balanced = True


# ============================================================
# 2. Collection-matched equilibrium and initial-condition settings
# ============================================================

Q_BAR_FIXED = 11.93
REFERENCE_MODE = "ode"
U_EQ_VALUE = 0.30

CHEB_ORDER = 6
H_CHEB_AMPLITUDE = 0.30
Q_CHEB_AMPLITUDE = 2.50
CHEB_COEFF_DECAY = 2.0


# ============================================================
# 3. SAC parameters
# ============================================================

total_timesteps = int(1e5)
learning_rate = 1e-4

# Use a much smaller learning rate for trainable DeepONet parameters.
deeponet_learning_rate = 1e-6

buffer_size = 100000
batch_size = 256
tau = 0.005
sac_gamma = 0.995
ent_coef = 0.005
learning_starts = 5000

seeds = [42,43,44,45,46]

LATENT_DIM = 102
RAW_STATE_DIM = 2 * N
SAC_FEATURE_DIM = LATENT_DIM

# This checkpoint must come from the Hadamard-fusion DeepONet pretraining code.
DEEPNET_PRETRAINED_PATH = "modedlNet10000QqQqQ14444_hadamard.pt"


# ============================================================
# 4. Reward parameters
# ============================================================

# r_k = -alpha*tracking - beta*state_change - gamma*control - delta*delta_control
reward_alpha = 1.0
reward_beta = 0.10
reward_gamma = 0.05
reward_delta = 0.10

# Separate H/Q weights for tracking and inter-sample state-change terms.
tracking_w_H = 1.0
tracking_w_Q = 1.0
change_w_H = 1.0
change_w_Q = 1.0

terminal_tracking_weight = 0.0
truncate_penalty = -200.0


# ============================================================
# 5. CAC physical parameters, ODE reference, and Chebyshev initial conditions
# ============================================================

def make_cac_table_params(
    q_bar: float = Q_BAR_FIXED,
    h_bar_0: float = 3.80,
    h_bar_L: float = 4.70,
) -> CACParams:
    """Create the CAC parameter object used by collection and RL training."""
    return CACParams(
        L=11239.0,
        B=40.0,
        g=9.81,
        I=7.92e-5,
        K=33.0,
        mu0=0.4,
        muL=0.4,
        z_up=5.00,
        U0_bar=0.1548,
        U_L_bar=U_EQ_VALUE,
        h_s=4.094,
        Q_bar=float(q_bar),
        H_bar_0=float(h_bar_0),
        H_bar_L=float(h_bar_L),
        H_min=0.20,
        max_state_value=1e10,
    )


def make_grid(p: CACParams, n_grid: int) -> Tuple[np.ndarray, float]:
    X = np.linspace(0.0, p.L, n_grid)
    dx = p.L / (n_grid - 1)
    return X, dx


def downstream_equilibrium_depth_from_q(Q: float, p: CACParams) -> float:
    """Compute Hbar(L) from the downstream equilibrium gate relation."""
    Q = float(Q)
    if not np.isfinite(Q) or Q <= 0.0:
        raise ValueError("The equilibrium discharge must be positive and finite.")

    kinetic_head = (
        Q**2 / (2.0 * p.g * p.B**2 * p.muL**2)
    ) ** (1.0 / 3.0)
    return float(p.h_s + p.U_L_bar + kinetic_head)


def integrate_steady_fixed_q_ode(
    X: np.ndarray,
    p: CACParams,
    Q_candidate: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Integrate the steady Saint-Venant ODE backward from x=L to x=0."""
    X = np.asarray(X, dtype=float)
    if X.ndim != 1 or X.size < 2:
        raise ValueError("X must be a one-dimensional grid with at least two points.")
    if not np.all(np.diff(X) > 0.0):
        raise ValueError("X must be strictly increasing.")

    Q_candidate = float(Q_candidate)
    if not np.isfinite(Q_candidate) or Q_candidate <= 0.0:
        raise ValueError("Q_candidate must be positive and finite.")

    Hbar_L = downstream_equilibrium_depth_from_q(Q_candidate, p)

    def rhs(_x: float, state: np.ndarray) -> np.ndarray:
        H = float(state[0])
        Q = float(state[1])

        if not np.isfinite(H) or not np.isfinite(Q):
            raise FloatingPointError("Non-finite state in steady reference ODE.")
        if H <= float(p.H_min):
            raise FloatingPointError("Steady reference ODE violates minimum depth.")

        area = p.B * H
        hydraulic_radius = p.B * H / (p.B + 2.0 * H)
        friction_slope = Q * abs(Q) / (
            p.K**2 * area**2 * hydraulic_radius ** (4.0 / 3.0)
        )

        denominator = p.g * p.B * H - Q**2 / (p.B * H**2)
        if abs(denominator) <= 1e-10:
            raise FloatingPointError(
                "Steady ODE approaches the critical-flow singularity."
            )

        dHdx = p.g * p.B * H * (p.I - friction_slope) / denominator
        if not np.isfinite(dHdx):
            raise FloatingPointError("Non-finite dHbar/dx in steady ODE.")

        return np.asarray([dHdx, 0.0], dtype=float)

    sol = solve_ivp(
        fun=rhs,
        t_span=(float(X[-1]), float(X[0])),
        y0=np.asarray([Hbar_L, Q_candidate], dtype=float),
        t_eval=X[::-1],
        rtol=1e-9,
        atol=1e-11,
        max_step=p.L / 1000.0,
    )

    if not sol.success:
        raise RuntimeError(f"Failed to solve steady ODE: {sol.message}")
    if sol.y.shape[1] != X.size:
        raise RuntimeError("Steady ODE did not return all requested grid points.")

    Hbar = np.asarray(sol.y[0][::-1], dtype=float)
    Qbar = np.asarray(sol.y[1][::-1], dtype=float)

    if not np.all(np.isfinite(Hbar)) or not np.all(np.isfinite(Qbar)):
        raise FloatingPointError("Steady reference contains NaN or Inf.")
    if float(np.min(Hbar)) <= float(p.H_min):
        raise RuntimeError("Steady Hbar profile violates the minimum depth.")

    return Hbar, Qbar


def solve_steady_reference_ode(
    X: np.ndarray,
    p: CACParams,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Solve Hbar(x) for prescribed spatially constant Qbar=11.93."""
    q_bar_fixed = float(q_bar_fixed)
    Hbar, _ = integrate_steady_fixed_q_ode(X, p, q_bar_fixed)
    Qbar = np.full_like(X, q_bar_fixed, dtype=float)
    return Hbar, Qbar, q_bar_fixed


def make_reference_profiles(
    X: np.ndarray,
    p: CACParams,
    reference: str = REFERENCE_MODE,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    if reference in {"ode", "strict"}:
        return solve_steady_reference_ode(X, p, q_bar_fixed=q_bar_fixed)
    if reference == "linear":
        Hbar = p.H_bar_0 + (p.H_bar_L - p.H_bar_0) * (X / p.L)
        Qbar = np.full_like(X, q_bar_fixed, dtype=float)
        return Hbar, Qbar, float(q_bar_fixed)
    raise ValueError("reference must be 'ode', 'strict', or 'linear'.")


def build_collection_matched_reference():
    """Build one internally consistent p, X, Hbar, Qbar tuple."""
    nominal_p = make_cac_table_params(q_bar=Q_BAR_FIXED)
    X, dx = make_grid(nominal_p, N)
    Hbar, Qbar, q_equilibrium = make_reference_profiles(
        X,
        nominal_p,
        reference=REFERENCE_MODE,
        q_bar_fixed=Q_BAR_FIXED,
    )

    p = make_cac_table_params(
        q_bar=q_equilibrium,
        h_bar_0=float(Hbar[0]),
        h_bar_L=float(Hbar[-1]),
    )
    return p, X, dx, Hbar, Qbar


def sample_truncated_chebyshev(
    xi: np.ndarray,
    rng,
    order: int,
    max_amplitude: float,
    coefficient_decay: float,
) -> np.ndarray:
    """Sample the same smooth truncated-Chebyshev perturbation as collection."""
    if order < 0:
        raise ValueError("CHEB_ORDER must be nonnegative.")
    if max_amplitude < 0.0:
        raise ValueError("Chebyshev amplitude must be nonnegative.")
    if coefficient_decay < 0.0:
        raise ValueError("CHEB_COEFF_DECAY must be nonnegative.")
    if max_amplitude == 0.0:
        return np.zeros_like(xi, dtype=float)

    mode_index = np.arange(order + 1, dtype=float)
    coefficient_scale = 1.0 / np.power(mode_index + 1.0, coefficient_decay)
    coefficients = rng.normal(loc=0.0, scale=coefficient_scale)

    perturbation = np.polynomial.chebyshev.chebval(xi, coefficients)
    max_abs = float(np.max(np.abs(perturbation)))
    if max_abs <= 1e-12:
        return np.zeros_like(xi, dtype=float)

    realized_amplitude = rng.uniform(0.35, 1.0) * max_amplitude
    return perturbation * (realized_amplitude / max_abs)


def random_initial_condition_func(X, Hbar, Qbar, rng):
    """Generate H0,Q0 with the same Chebyshev distribution as collection."""
    X = np.asarray(X, dtype=float)
    Hbar = np.asarray(Hbar, dtype=float)
    Qbar = np.asarray(Qbar, dtype=float)
    xi = 2.0 * X / float(X[-1]) - 1.0

    # Rejection sampling preserves the polynomial shape; no pointwise clipping.
    for _ in range(100):
        H_perturbation = sample_truncated_chebyshev(
            xi,
            rng,
            CHEB_ORDER,
            H_CHEB_AMPLITUDE,
            CHEB_COEFF_DECAY,
        )
        Q_perturbation = sample_truncated_chebyshev(
            xi,
            rng,
            CHEB_ORDER,
            Q_CHEB_AMPLITUDE,
            CHEB_COEFF_DECAY,
        )

        H0 = Hbar + H_perturbation
        Q0 = Qbar + Q_perturbation

        finite = np.all(np.isfinite(H0)) and np.all(np.isfinite(Q0))
        valid_depth = float(np.min(H0)) > 0.20
        valid_discharge = float(np.min(Q0)) > 0.0

        if finite and valid_depth and valid_discharge:
            return H0, Q0

    raise RuntimeError(
        "Failed to generate a valid Chebyshev initial condition after 100 "
        "attempts. Reduce H_CHEB_AMPLITUDE or Q_CHEB_AMPLITUDE."
    )


# ============================================================
# 6. Reward class
# ============================================================

class CACTrackingSmoothReward(BaseReward):
    """
    Discrete reward at the control instant t_k:

        r_k = -alpha * tracking_cost
              -beta  * state_change_cost
              -gamma * control_cost
              -delta * delta_control_cost.

    Definitions:
        tracking_cost
            = tracking_w_H * mean(((H_k-H_eq)/H_scale)^2)
            + tracking_w_Q * mean(((Q_k-Q_eq)/Q_scale)^2)

        state_change_cost
            = change_w_H * mean(((H_k-H_{k-1})/H_scale)^2)
            + change_w_Q * mean(((Q_k-Q_{k-1})/Q_scale)^2)

        control_cost
            = ((U_k-U_eq)/U_scale)^2

        delta_control_cost
            = (((U_k-U_eq)/U_scale)
               -((U_{k-1}-U_eq)/U_scale))^2

    The state-change term is an inter-sample increment penalty. It is not a
    time derivative because it is not divided by the control interval.

    Spatial means are used instead of unnormalized sums so that the reward scale
    does not grow with the number of grid points N.
    """

    def __init__(
        self,
        H_eq,
        Q_eq,
        delta_t=control_sample_rate,
        H_scale=0.30,
        Q_scale=3.0,
        U_scale=0.10,
        alpha=1.0,
        beta=0.10,
        gamma=0.05,
        delta=0.10,
        tracking_w_H=1.0,
        tracking_w_Q=1.0,
        change_w_H=1.0,
        change_w_Q=1.0,
        U_ref=U_EQ_VALUE,
        terminal_tracking_weight=0.0,
        truncate_penalty=-200.0,
    ):
        self.H_eq = np.asarray(H_eq, dtype=float)
        self.Q_eq = np.asarray(Q_eq, dtype=float)
        self.N = len(self.H_eq)

        # Kept only for compatibility with the environment interface.
        self.delta_t = float(delta_t)

        self.H_scale = float(H_scale)
        self.Q_scale = float(Q_scale)
        self.U_scale = float(U_scale)

        self.alpha = float(alpha)
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.delta = float(delta)

        self.tracking_w_H = float(tracking_w_H)
        self.tracking_w_Q = float(tracking_w_Q)
        self.change_w_H = float(change_w_H)
        self.change_w_Q = float(change_w_Q)

        self.U_ref = float(U_ref)
        self.terminal_tracking_weight = float(terminal_tracking_weight)
        self.truncate_penalty = float(truncate_penalty)

        self.prev_H: Optional[np.ndarray] = None
        self.prev_Q: Optional[np.ndarray] = None
        self.prev_action: Optional[float] = None

        self.last_terms = {
            "tracking_cost": 0.0,
            "state_change_cost": 0.0,
            "control_cost": 0.0,
            "delta_control_cost": 0.0,
        }

    def _reset_memory(self):
        self.prev_H = None
        self.prev_Q = None
        self.prev_action = None

    def _split_state(self, state_vec):
        state_vec = np.asarray(state_vec, dtype=float).reshape(-1)
        if state_vec.size < 2 * self.N:
            raise ValueError(
                f"PDE state length is {state_vec.size}, expected at least {2 * self.N}."
            )
        H = state_vec[:self.N]
        Q = state_vec[self.N:2 * self.N]
        return H, Q

    def _action_value(self, action):
        # The environment passes the physical downstream gate U_L to the reward.
        if action is None:
            return self.U_ref
        value = float(np.asarray(action).reshape(-1)[0])
        return float(
            np.nan_to_num(
                value,
                nan=self.U_ref,
                posinf=self.U_ref,
                neginf=self.U_ref,
            )
        )

    def reward(
        self,
        uVec=None,
        time_index=None,
        terminate=None,
        truncate=None,
        action=None,
        control_sample_rate=None,
    ):
        if uVec is None or time_index is None:
            return 0.0

        if int(time_index) <= 1:
            self._reset_memory()

        if truncate:
            self._reset_memory()
            return float(self.truncate_penalty)

        Hk, Qk = self._split_state(uVec[time_index])

        if (
            not np.all(np.isfinite(Hk))
            or not np.all(np.isfinite(Qk))
            or np.any(Hk <= 0.0)
        ):
            self._reset_memory()
            return float(self.truncate_penalty)

        # 1) H/Q tracking cost
        H_error_tilde = (Hk - self.H_eq) / self.H_scale
        Q_error_tilde = (Qk - self.Q_eq) / self.Q_scale

        tracking_cost = (
            self.tracking_w_H * np.mean(H_error_tilde ** 2)
            + self.tracking_w_Q * np.mean(Q_error_tilde ** 2)
        )

        # 2) Inter-sample H/Q state-change cost
        if self.prev_H is None or self.prev_Q is None:
            state_change_cost = 0.0
        else:
            delta_H_tilde = (Hk - self.prev_H) / self.H_scale
            delta_Q_tilde = (Qk - self.prev_Q) / self.Q_scale

            state_change_cost = (
                self.change_w_H * np.mean(delta_H_tilde ** 2)
                + self.change_w_Q * np.mean(delta_Q_tilde ** 2)
            )

        # 3) Control-magnitude cost
        Uk = self._action_value(action)
        U_tilde = (Uk - self.U_ref) / self.U_scale
        control_cost = U_tilde ** 2

        # 4) Control-increment cost
        if self.prev_action is None:
            delta_control_cost = 0.0
        else:
            U_prev_tilde = (self.prev_action - self.U_ref) / self.U_scale
            delta_control_cost = (U_tilde - U_prev_tilde) ** 2

        reward_value = (
            -self.alpha * tracking_cost
            -self.beta * state_change_cost
            -self.gamma * control_cost
            -self.delta * delta_control_cost
        )

        if terminate:
            reward_value -= self.terminal_tracking_weight * tracking_cost

        self.prev_H = Hk.copy()
        self.prev_Q = Qk.copy()
        self.prev_action = Uk

        self.last_terms = {
            "tracking_cost": float(tracking_cost),
            "state_change_cost": float(state_change_cost),
            "control_cost": float(control_cost),
            "delta_control_cost": float(delta_control_cost),
        }

        return float(reward_value)


# ============================================================
# 7. DeepONet controller and hybrid SAC modules
# ============================================================

def preprocess_obs(obs: torch.Tensor) -> torch.Tensor:
    if not isinstance(obs, torch.Tensor):
        obs = torch.as_tensor(obs)
    return obs.float()


class BranchNet(nn.Module):
    def __init__(self, shape=N, output_dim=LATENT_DIM):
        super().__init__()
        self.shape = int(shape)
        self.output_dim = int(output_dim)

        self.conv1 = nn.Conv1d(
            in_channels=2,
            out_channels=16,
            kernel_size=5,
            stride=2,
        )
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(
            in_channels=16,
            out_channels=32,
            kernel_size=5,
            stride=2,
        )

        with torch.no_grad():
            tmp = torch.zeros(1, 2, self.shape)
            tmp = self.relu(self.conv1(tmp))
            tmp = self.relu(self.conv2(tmp))
            flat_dim = tmp.reshape(1, -1).shape[1]

        self.fc1 = nn.Linear(flat_dim, self.output_dim)

    def forward(self, x):
        x = preprocess_obs(x)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        if x.shape[1] != 2 * self.shape:
            raise ValueError(
                f"Branch input dimension is {x.shape[1]}, expected {2 * self.shape}."
            )

        x = x.reshape(x.shape[0], 2, self.shape)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = self.relu(self.fc1(x))
        return x


class TrunkNet(nn.Module):
    def __init__(self, dim_x=1, width=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_x, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, width),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x.float())


class DeepONetModified(nn.Module):
    """
    Hadamard-fusion DeepONet controller.

        b(s) in R^p,
        t(x) in R^p,
        z(s,x) = b(s) * t(x) in R^p,
        U_prior = f_ctrl(z) in R.
    """

    def __init__(self, shape=N, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = int(latent_dim)

        self.branch = BranchNet(shape, output_dim=self.latent_dim)
        self.trunk = TrunkNet(dim_x=1, width=self.latent_dim)

        # Hadamard fusion preserves latent_dim features.
        self.fc3 = nn.Linear(self.latent_dim, 32)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, 1)

    def extract_latent(self, obs, grid):
        b = self.branch(obs)       # [batch, latent_dim]
        t = self.trunk(grid)       # [n_query, latent_dim]

        if t.shape[0] == 1:
            t = t.expand(b.shape[0], -1)
        elif t.shape[0] != b.shape[0]:
            raise ValueError(
                f"Branch batch {b.shape[0]} and trunk batch {t.shape[0]} do not match."
            )

        z = b * t                  # Hadamard product
        return z

    def control_head(self, z):
        y = self.relu(self.fc3(z))
        y = self.relu(self.fc4(y))
        y = self.fc5(y)
        return y

    def forward(self, obs, grid, return_latent=False):
        z = self.extract_latent(obs, grid)
        u_prior_physical = self.control_head(z)

        if return_latent:
            return u_prior_physical, z

        return u_prior_physical


def _clean_deeponet_state_dict_keys(state_dict):
    """Remove common wrapper prefixes from a DeepONet checkpoint."""
    cleaned = {}
    prefixes = ["module.", "model.", "net1.", "deeponet."]

    for key, value in state_dict.items():
        new_key = key
        changed = True
        while changed:
            changed = False
            for prefix in prefixes:
                if new_key.startswith(prefix):
                    new_key = new_key[len(prefix):]
                    changed = True
        cleaned[new_key] = value

    return cleaned


def _load_deeponet_weights(
    net: DeepONetModified,
    pretrained_path: str,
    name: str,
):
    """Strictly load the new Hadamard-fusion DeepONet checkpoint."""
    path = Path(pretrained_path)

    if not path.exists():
        raise FileNotFoundError(
            f"Pretrained Hadamard DeepONet checkpoint not found for {name}: {path}"
        )

    state_dict = torch.load(str(path), map_location="cpu")
    if isinstance(state_dict, dict) and "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    if not isinstance(state_dict, dict):
        raise TypeError(
            f"Unsupported checkpoint type for {name}: "
            f"{type(state_dict).__name__}"
        )

    cleaned = _clean_deeponet_state_dict_keys(state_dict)

    try:
        net.load_state_dict(cleaned, strict=True)
    except RuntimeError as exc:
        raise RuntimeError(
            f"The checkpoint for {name} is incompatible with the new "
            "Hadamard-fusion DeepONet. It must use fc3=Linear(102,32), "
            "z=b*t, and a physical-gate output pretrained with y_ul.dat.\n"
            f"Original error:\n{exc}"
        ) from exc

    print(f"Loaded pretrained Hadamard DeepONet for {name}: {path}")


def _apply_partial_freeze_to_deeponet(
    net: DeepONetModified,
    name: str,
):
    """Apply the same partial-freeze strategy as the latest TDPSAC program."""
    for param in net.branch.conv1.parameters():
        param.requires_grad = False
    for param in net.branch.conv2.parameters():
        param.requires_grad = False

    for param in net.branch.fc1.parameters():
        param.requires_grad = True

    for param in net.trunk.parameters():
        param.requires_grad = False

    last_linear = None
    for module in reversed(list(net.trunk.net)):
        if isinstance(module, nn.Linear):
            last_linear = module
            break

    if last_linear is not None:
        for param in last_linear.parameters():
            param.requires_grad = True

    for layer in [net.fc3, net.fc4, net.fc5]:
        for param in layer.parameters():
            param.requires_grad = True

    trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
    total = sum(p.numel() for p in net.parameters())
    print(
        f"{name} partial freeze: trainable={trainable}/{total} "
        f"({100.0 * trainable / max(total, 1):.2f}%)."
    )


def _freeze_all_deeponet(net: DeepONetModified, name: str):
    """Fully freeze the independent control-prior DeepONet."""
    for param in net.parameters():
        param.requires_grad = False

    total = sum(p.numel() for p in net.parameters())
    print(f"{name} fully frozen: trainable=0/{total}.")


def _physical_gate_to_normalized_action(u_physical):
    scale = max_control_value - min_control_value
    if scale <= 0.0:
        raise ValueError("max_control_value must be greater than min_control_value.")
    return 2.0 * (u_physical - min_control_value) / scale - 1.0


class HybridCACDeepONetFeatureExtractor(BaseFeaturesExtractor):
    """
    Actor-side feature extractor with two independent new DeepONets.

    net1:
        Partially frozen feature DeepONet. It supplies the Hadamard feature
        z=b(s)*t(L) to the standard SAC actor network.

    control_deeponet:
        Fully frozen independent DeepONet. It supplies only the physical gate
        prior U_prior used in the hybrid action.
    """

    def __init__(
        self,
        observation_space: spaces.Box,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        query_point: float = 1.0,
        freeze_deeponet: bool = False,
    ):
        if features_dim != SAC_FEATURE_DIM:
            raise ValueError(
                f"features_dim must be {SAC_FEATURE_DIM}, got {features_dim}."
            )

        super().__init__(observation_space, features_dim)
        self.query_point = float(query_point)

        # Feature representation DeepONet: partially frozen and fine-tuned.
        self.net1 = DeepONetModified(N, latent_dim=LATENT_DIM)

        # Independent control-prior DeepONet: fully frozen.
        self.control_deeponet = DeepONetModified(N, latent_dim=LATENT_DIM)

        self.u_deep_norm: Optional[torch.Tensor] = None
        self.u_prior_physical: Optional[torch.Tensor] = None

        self.register_buffer(
            "grid",
            torch.tensor([[self.query_point]], dtype=torch.float32),
        )

        _load_deeponet_weights(
            self.net1,
            pretrained_path,
            "actor feature DeepONet",
        )
        _load_deeponet_weights(
            self.control_deeponet,
            pretrained_path,
            "fixed control DeepONet",
        )

        # Preserve this method's two-role transfer strategy.
        if freeze_deeponet:
            _freeze_all_deeponet(self.net1, "Actor feature DeepONet")
        else:
            _apply_partial_freeze_to_deeponet(
                self.net1,
                "Actor feature DeepONet",
            )
        _freeze_all_deeponet(
            self.control_deeponet,
            "Fixed control DeepONet",
        )

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        obs = preprocess_obs(observations)
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)

        if obs.shape[1] != RAW_STATE_DIM:
            raise ValueError(
                f"Raw observation dimension is {obs.shape[1]}, "
                f"expected {RAW_STATE_DIM}."
            )

        grid = self.grid.to(obs.device)

        # The independent frozen network generates the physical gate prior.
        with torch.no_grad():
            u_prior_physical = self.control_deeponet(obs, grid)
            u_prior_physical = torch.clamp(
                u_prior_physical,
                min_control_value,
                max_control_value,
            )
            u_deep_norm = _physical_gate_to_normalized_action(u_prior_physical)

        self.u_prior_physical = u_prior_physical
        self.u_deep_norm = torch.clamp(u_deep_norm, -1.0, 1.0)

        # The partially frozen feature network supplies the Hadamard feature only.
        z = self.net1.extract_latent(obs, grid)
        return z.view(z.size(0), -1)


class HybridCACActor(Actor):
    """SAC actor whose normalized output is blended with DeepONet prior action."""

    def __init__(self, *args, alpha: float = 0.4, **kwargs):
        super().__init__(*args, **kwargs)
        init_alpha = float(np.clip(alpha, 0.0, 1.0))

        self.register_buffer(
            "alpha_tensor",
            torch.tensor(init_alpha, dtype=torch.float32),
        )

        self.last_deep_action = None
        self.last_rl_action = None
        self.last_hybrid_action = None

    @property
    def alpha(self) -> float:
        return float(self.alpha_tensor.detach().cpu().item())

    def set_alpha(self, alpha: float):
        alpha_value = float(np.clip(alpha, 0.0, 1.0))
        with torch.no_grad():
            self.alpha_tensor.fill_(alpha_value)

    def _deep_action(self, obs: torch.Tensor) -> torch.Tensor:
        u_deep = getattr(self.features_extractor, "u_deep_norm", None)

        if u_deep is None or u_deep.shape[0] != obs.shape[0]:
            _ = self.features_extractor(obs)
            u_deep = self.features_extractor.u_deep_norm

        return torch.clamp(u_deep, -1.0, 1.0)

    def _hybridize(self, rl_action: torch.Tensor, obs: torch.Tensor) -> torch.Tensor:
        u_deep = self._deep_action(obs)

        alpha = self.alpha_tensor.to(
            device=rl_action.device,
            dtype=rl_action.dtype,
        )

        hybrid = (1.0 - alpha) * u_deep + alpha * rl_action
        hybrid = torch.clamp(hybrid, -1.0, 1.0)

        self.last_deep_action = u_deep.detach()
        self.last_rl_action = rl_action.detach()
        self.last_hybrid_action = hybrid.detach()

        return hybrid

    def forward(self, obs: torch.Tensor, deterministic: bool = False):
        rl_action = super().forward(obs, deterministic=deterministic)
        return self._hybridize(rl_action, obs)

    def action_log_prob(self, obs: torch.Tensor):
        # The log probability is that of the SAC stochastic component.
        # The deterministic affine hybrid mapping is not included in log_prob.
        rl_action, log_prob = super().action_log_prob(obs)
        hybrid_action = self._hybridize(rl_action, obs)
        return hybrid_action, log_prob


class CriticDeepONetFeatureExtractor(nn.Module):
    """Critic-side partially frozen Hadamard DeepONet feature extractor."""

    def __init__(
        self,
        observation_space: spaces.Box,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
    ):
        super().__init__()

        if features_dim != SAC_FEATURE_DIM:
            raise ValueError(
                f"features_dim must be {SAC_FEATURE_DIM}, got {features_dim}."
            )

        self.net1 = DeepONetModified(N, latent_dim=LATENT_DIM)
        self.register_buffer(
            "grid",
            torch.tensor([[1.0]], dtype=torch.float32),
        )

        _load_deeponet_weights(
            self.net1,
            pretrained_path,
            "critic feature DeepONet",
        )
        _apply_partial_freeze_to_deeponet(
            self.net1,
            "Critic feature DeepONet",
        )

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        obs = preprocess_obs(observations)
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)

        if obs.shape[1] != RAW_STATE_DIM:
            raise ValueError(
                f"Raw observation dimension is {obs.shape[1]}, "
                f"expected {RAW_STATE_DIM}."
            )

        z = self.net1.extract_latent(obs, self.grid.to(obs.device))
        return z.view(z.size(0), -1)


class HybridCACCritic(nn.Module):
    """Twin-Q critic using an independent partially frozen Hadamard feature net."""

    def __init__(
        self,
        observation_space: spaces.Box,
        action_space: spaces.Box,
        hidden_dim: int = 256,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
    ):
        super().__init__()

        self.features_extractor = CriticDeepONetFeatureExtractor(
            observation_space,
            features_dim=features_dim,
            pretrained_path=pretrained_path,
        )

        action_dim = int(np.prod(action_space.shape))
        input_dim = features_dim + action_dim

        self.q1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.q2 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def set_training_mode(self, mode: bool):
        self.train(mode)

    def forward(self, obs: torch.Tensor, actions: torch.Tensor):
        features = self.features_extractor(obs)
        q_input = torch.cat([features, actions], dim=1)
        return self.q1(q_input), self.q2(q_input)


class HybridCACSACPolicy(SACPolicy):
    """SAC policy with separate feature and fixed-control Hadamard DeepONets."""

    def __init__(
        self,
        *args,
        hybrid_alpha_initial: float = 0.4,
        deeponet_pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        **kwargs,
    ):
        self.hybrid_alpha_initial = float(hybrid_alpha_initial)
        self.deeponet_pretrained_path = deeponet_pretrained_path

        kwargs.setdefault(
            "features_extractor_class",
            HybridCACDeepONetFeatureExtractor,
        )
        kwargs.setdefault(
            "features_extractor_kwargs",
            dict(
                features_dim=SAC_FEATURE_DIM,
                pretrained_path=deeponet_pretrained_path,
                query_point=1.0,
                freeze_deeponet=False,
            ),
        )

        super().__init__(*args, **kwargs)

    def make_actor(self, features_extractor=None):
        kwargs = self._update_features_extractor(
            self.actor_kwargs,
            features_extractor,
        )
        return HybridCACActor(
            **kwargs,
            alpha=self.hybrid_alpha_initial,
        ).to(self.device)

    def make_critic(self, features_extractor=None):
        return HybridCACCritic(
            self.observation_space,
            self.action_space,
            hidden_dim=256,
            features_dim=SAC_FEATURE_DIM,
            pretrained_path=self.deeponet_pretrained_path,
        ).to(self.device)

    def set_alpha(self, alpha: float):
        self.actor.set_alpha(alpha)


# ============================================================
# 8. Optimizer helper: separate DeepONet learning rate
# ============================================================

def _make_param_groups_with_deeponet_lr(module: nn.Module):
    deeponet_params = []
    other_params = []

    for name, param in module.named_parameters():
        if not param.requires_grad:
            continue

        if "features_extractor.net1" in name or name.startswith("net1."):
            deeponet_params.append(param)
        else:
            other_params.append(param)

    groups = []

    if other_params:
        groups.append(
            {
                "params": other_params,
                "lr": learning_rate,
                "is_deeponet_group": False,
            }
        )

    if deeponet_params:
        groups.append(
            {
                "params": deeponet_params,
                "lr": deeponet_learning_rate,
                "is_deeponet_group": True,
            }
        )

    return groups


def set_deeponet_small_lr_optimizers(model):
    actor_groups = _make_param_groups_with_deeponet_lr(model.actor)
    critic_groups = _make_param_groups_with_deeponet_lr(model.critic)

    model.actor.optimizer = torch.optim.Adam(actor_groups, eps=1e-5)
    model.critic.optimizer = torch.optim.Adam(critic_groups, eps=1e-5)

    n_actor_deep = sum(
        p.numel()
        for group in actor_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_actor_other = sum(
        p.numel()
        for group in actor_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_deep = sum(
        p.numel()
        for group in critic_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_other = sum(
        p.numel()
        for group in critic_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )

    print("Custom optimizers with separate DeepONet learning rate are set.")
    print(f"Actor params: DeepONet={n_actor_deep}, other={n_actor_other}")
    print(f"Critic params: DeepONet={n_critic_deep}, other={n_critic_other}")
    print(
        f"Base SAC learning_rate={learning_rate}, "
        f"DeepONet learning_rate={deeponet_learning_rate}"
    )


class HybridSAC(SAC):
    """Preserve the smaller learning rate of DeepONet parameter groups."""

    def _update_learning_rate(self, optimizers):
        super()._update_learning_rate(optimizers)

        if not isinstance(optimizers, (list, tuple)):
            optimizers = [optimizers]

        current_lr = self.lr_schedule(self._current_progress_remaining)

        for optimizer in optimizers:
            for group in optimizer.param_groups:
                if group.get("is_deeponet_group", False):
                    group["lr"] = deeponet_learning_rate
                elif "is_deeponet_group" in group:
                    group["lr"] = current_lr


# ============================================================
# 9. Hybrid alpha scheduling
# ============================================================

def alpha_schedule(step: int) -> float:
    """
    alpha is the SAC weight:
        alpha small -> DeepONet prior dominates,
        alpha large -> SAC compensation dominates.
    """
    if step < 10000:
        return 0.1
    if step < 20000:
        return 0.5
    return 0.9


class AlphaCallback(BaseCallback):
    def _on_training_start(self):
        alpha = alpha_schedule(self.num_timesteps)
        self.model.policy.set_alpha(alpha)
        print(f"Initial hybrid alpha = {alpha}")

    def _on_step(self):
        alpha = alpha_schedule(self.num_timesteps)
        self.model.policy.set_alpha(alpha)
        return True


# ============================================================
# 10. Environment creation
# ============================================================

def make_env():
    p, X, dx, H_eq, Q_eq = build_collection_matched_reference()

    # Constructor fallback is the reference itself. Every reset then samples
    # the same Chebyshev initial-condition distribution as data collection.
    H_init = H_eq.copy()
    Q_init = Q_eq.copy()

    reward_class = CACTrackingSmoothReward(
        H_eq=H_eq,
        Q_eq=Q_eq,
        delta_t=control_sample_rate,
        H_scale=H_scale,
        Q_scale=Q_scale,
        U_scale=U_scale,
        alpha=reward_alpha,
        beta=reward_beta,
        gamma=reward_gamma,
        delta=reward_delta,
        tracking_w_H=tracking_w_H,
        tracking_w_Q=tracking_w_Q,
        change_w_H=change_w_H,
        change_w_Q=change_w_Q,
        U_ref=U_EQ_VALUE,
        terminal_tracking_weight=terminal_tracking_weight,
        truncate_penalty=truncate_penalty,
    )

    base_env = CACSaintVenantPDE1D(
        params=p,
        X_grid=X,
        Hbar=H_eq,
        Qbar=Q_eq,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=random_initial_condition_func,
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=reward_class,
        # SAC emits normalized actions; the environment maps them to physical U_L.
        normalize=True,
        # The SAC observation is the raw physical state [H,Q].
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    env = base_env

    # Expose the exact reference for diagnostics and test-program reuse.
    env.reference_X = X
    env.reference_H = H_eq
    env.reference_Q = Q_eq
    env.reference_dx = dx

    print(
        "Collection-matched ODE reference: "
        f"Hbar(0)={H_eq[0]:.9f}, Hbar(L)={H_eq[-1]:.9f}, "
        f"Qbar={Q_eq[0]:.9f}, dx={dx:.6f}"
    )

    return env


# ============================================================
# 11. Training
# ============================================================

def train_with_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    env = make_env()

    reset_obs, _ = env.reset(seed=seed)
    print(f"Raw observation shape: {reset_obs.shape}")

    if reset_obs.shape != (RAW_STATE_DIM,):
        raise RuntimeError(
            f"Expected raw observation shape {(RAW_STATE_DIM,)}, "
            f"got {reset_obs.shape}."
        )

    policy_kwargs = dict(
        deeponet_pretrained_path=DEEPNET_PRETRAINED_PATH,
        hybrid_alpha_initial=alpha_schedule(0),
        net_arch=dict(pi=[256, 256], qf=[256, 256]),
        optimizer_kwargs={"eps": 1e-6},
        share_features_extractor=False,
    )

    model = HybridSAC(
        policy=HybridCACSACPolicy,
        policy_kwargs=policy_kwargs,
        env=env,
        learning_rate=learning_rate,
        buffer_size=buffer_size,
        batch_size=batch_size,
        tau=tau,
        gamma=sac_gamma,
        #ent_coef=ent_coef,
        #learning_starts=learning_starts,
        verbose=1,
        tensorboard_log=f"./tbsacddx/seed_{seed}/",
    )

    set_deeponet_small_lr_optimizers(model)

    checkpoint_callback = CheckpointCallback(
        save_freq=10000,
        save_path=f"./sacddx/seed_{seed}/",
        name_prefix="sac_cac_hybrid_fixed_control_yul_ode_cheb_raw_HQ",
        save_replay_buffer=True,
        save_vecnormalize=True,
    )

    alpha_callback = AlphaCallback()
    callbacks = CallbackList([alpha_callback, checkpoint_callback])

    start_time = time.perf_counter()

    model.learn(
        total_timesteps=total_timesteps,
        callback=callbacks,
    )

    training_time = time.perf_counter() - start_time
    hours = int(training_time // 3600)
    minutes = int((training_time % 3600) // 60)
    seconds = training_time % 60

    print("\n" + "=" * 70)
    print(f"Seed {seed} training completed")
    print(
        f"Training time for this seed: "
        f"{hours:02d} h {minutes:02d} min {seconds:06.2f} s"
    )
    print(f"Total training time: {training_time:.2f} seconds")
    print("=" * 70)

    save_path = (
        f"./sacddx/seed_{seed}/"
        "sac_cac_hybrid_fixed_control_yul_ode_cheb_raw_HQ_final"
    )
    model.save(save_path)
    env.close()

    return model, save_path


def main():
    print("Hybrid Hadamard-DeepONet SAC with independent fixed control prior")
    print("=" * 90)
    print(f"N={N}, T={T}, dt={dt}, control_sample_rate={control_sample_rate}")
    print(f"Physical gate U_L in [{min_control_value}, {max_control_value}]")
    print("Observation: raw physical [H_1,...,H_N,Q_1,...,Q_N]")
    print(
        f"Observation dimension={RAW_STATE_DIM}, "
        f"SAC feature dimension={SAC_FEATURE_DIM}"
    )
    print("Feature/control DeepONet input: raw [H,Q]")
    print("Fixed control DeepONet output: physical U_L pretrained from y_ul.dat")
    print("New DeepONet fusion: z=b(s)*t(L), followed by 102->32->16->1")
    print("Hybrid action: a=(1-alpha_hybrid)*a_prior+alpha_hybrid*a_SAC")
    print(f"DeepONet checkpoint: {DEEPNET_PRETRAINED_PATH}")
    print(
        "Transfer strategy: actor/critic feature DeepONets are partially "
        "frozen; an independent control DeepONet is fully frozen"
    )
    print(
        f"Reference: Qbar={Q_BAR_FIXED} fixed; Hbar solved by steady ODE "
        f"(mode={REFERENCE_MODE})"
    )
    print(
        "Reward: -alpha*tracking - beta*state_change "
        "- gamma*control - delta*delta_control"
    )
    print(
        f"Reward weights: alpha={reward_alpha}, beta={reward_beta}, "
        f"gamma={reward_gamma}, delta={reward_delta}"
    )
    print(
        f"Tracking H/Q weights=({tracking_w_H}, {tracking_w_Q}); "
        f"state-change H/Q weights=({change_w_H}, {change_w_Q})"
    )
    print(f"Reward scales: H={H_scale}, Q={Q_scale}, U={U_scale}")
    print("Cumulative-error state: removed")
    print(
        "Chebyshev initial conditions: "
        f"order={CHEB_ORDER}, A_H={H_CHEB_AMPLITUDE}, "
        f"A_Q={Q_CHEB_AMPLITUDE}, decay={CHEB_COEFF_DECAY}"
    )
    print(
        f"Learning rates: SAC={learning_rate}, "
        f"DeepONet={deeponet_learning_rate}"
    )

    for index, seed in enumerate(seeds):
        print(f"\n===== Training {index + 1}/{len(seeds)}, seed={seed} =====")
        _, save_path = train_with_seed(seed)
        print(f"Saved model: {save_path}")


if __name__ == "__main__":
    main()


Hybrid Hadamard-DeepONet SAC with independent fixed control prior
N=102, T=40000.0, dt=5.0, control_sample_rate=400.0
Physical gate U_L in [0.15, 0.4]
Observation: raw physical [H_1,...,H_N,Q_1,...,Q_N]
Observation dimension=204, SAC feature dimension=102
Feature/control DeepONet input: raw [H,Q]
Fixed control DeepONet output: physical U_L pretrained from y_ul.dat
New DeepONet fusion: z=b(s)*t(L), followed by 102->32->16->1
Hybrid action: a=(1-alpha_hybrid)*a_prior+alpha_hybrid*a_SAC
DeepONet checkpoint: modedlNet10000QqQqQ14444_hadamard.pt
Transfer strategy: actor/critic feature DeepONets are partially frozen; an independent control DeepONet is fully frozen
Reference: Qbar=11.93 fixed; Hbar solved by steady ODE (mode=ode)
Reward: -alpha*tracking - beta*state_change - gamma*control - delta*delta_control
Reward weights: alpha=1.0, beta=0.1, gamma=0.05, delta=0.1
Tracking H/Q weights=(1.0, 1.0); state-change H/Q weights=(1.0, 1.0)
Reward scales: H=0.3, Q=3.0, U=0.1
Cumulative-error state

In [3]:
# -*- coding: utf-8 -*-
"""
DeepONet-feature SAC training for the CAC Saint-Venant system.

This program keeps the method of the supplied program unchanged:
DeepONet is embedded only as the SAC feature extractor. The final control
action is still produced directly by the standard SAC actor; there is no
DeepONet prior-action mixing and no hybrid action.

The environment and training setup are aligned with the latest training
program:
1. Qbar(x)=11.93 m^3/s and Hbar(x) is solved from the steady Saint-Venant ODE.
2. Each episode starts from truncated-Chebyshev perturbations around Hbar,Qbar.
3. The observation is raw physical [H_1,...,H_N,Q_1,...,Q_N], with no
   cumulative-error state.
4. The reward is tracking + inter-sample state-change + control + control-increment penalty.
5. The embedded DeepONet uses Hadamard Branch-Trunk fusion
       z = b(H,Q) * t(L)
   and SAC receives the 102-dimensional fused feature z.

Required external module:
    cac_saint_venant_env_no_bs.py
Required pretrained checkpoint:
    modedlNet10000QqQqQ14444_hadamard.pt
"""

from __future__ import annotations

import os
import random
import time
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import torch
from scipy.integrate import solve_ivp
from gymnasium import spaces
from torch import nn

from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

from cac_saint_venant_env_no_bs import CACParams, CACSaintVenantPDE1D

try:
    from pde_control_gym.src.rewards.base_reward import BaseReward
except Exception:
    class BaseReward:
        pass


# ============================================================
# 1. Simulation parameters
# ============================================================

N = 102
T = 40000.0
dt = 5.0
control_sample_rate = 400.0

max_state_value = 1e6

# SAC outputs normalized actions a in [-1,1]. The environment maps them to the
# physical downstream gate U_L in this interval.
min_control_value = 0.15
max_control_value = 0.40

# State/reward normalization scales. H and Q remain raw in the observation;
# these scales are used only inside the reward calculation.
H_scale = 0.30
Q_scale = 3.0
U_scale = 0.10

obs_clip = 50.0

cfl = 0.45
check_cfl = True
well_balanced = True


# ============================================================
# 2. ODE equilibrium and Chebyshev initial-condition settings
# ============================================================

Q_BAR_FIXED = 11.93
REFERENCE_MODE = "ode"
U_EQ_VALUE = 0.30

# The following values directly control the training initial-condition range.
CHEB_ORDER = 6
H_CHEB_AMPLITUDE = 0.30
Q_CHEB_AMPLITUDE = 2.50
CHEB_COEFF_DECAY = 2.0


# ============================================================
# 3. SAC parameters
# ============================================================

total_timesteps = int(1e5)
learning_rate = 1e-4
# Smaller learning rate for trainable DeepONet representation parameters.
deeponet_learning_rate = 1e-6
buffer_size = 100000
batch_size = 256
tau = 0.005
sac_gamma = 0.995
ent_coef = 0.005
learning_starts = 5000

seeds = [42, 43, 44, 45, 46]

LATENT_DIM = 102
SAC_FEATURE_DIM = LATENT_DIM
RAW_STATE_DIM = 2 * N

# Hadamard-fusion DeepONet checkpoint pretrained with physical gate label y_ul.dat.
DEEPNET_PRETRAINED_PATH = "modedlNet10000QqQqQ14444_hadamard.pt"


# ============================================================
# 4. Reward parameters
# ============================================================

# r_k = -alpha*tracking - beta*state_change - gamma*control - delta*delta_control
reward_alpha = 1.0
reward_beta = 0.10
reward_gamma = 0.05
reward_delta = 0.10

# Separate H/Q weights for tracking and inter-sample state-change terms.
tracking_w_H = 1.0
tracking_w_Q = 1.0
change_w_H = 1.0
change_w_Q = 1.0

terminal_tracking_weight = 0.0
truncate_penalty = -200.0


# ============================================================
# 5. Physical parameters and steady ODE reference
# ============================================================

def make_cac_table_params(
    q_bar: float = Q_BAR_FIXED,
    h_bar_0: float = 3.80,
    h_bar_L: float = 4.70,
) -> CACParams:
    """Create a CAC parameter object consistent with the selected reference."""
    return CACParams(
        L=11239.0,
        B=40.0,
        g=9.81,
        I=7.92e-5,
        K=33.0,
        mu0=0.4,
        muL=0.4,
        z_up=5.00,
        U0_bar=0.1548,
        U_L_bar=U_EQ_VALUE,
        h_s=4.094,
        Q_bar=float(q_bar),
        H_bar_0=float(h_bar_0),
        H_bar_L=float(h_bar_L),
        H_min=0.20,
        max_state_value=1e10,
    )


def make_grid(p: CACParams, n_grid: int) -> Tuple[np.ndarray, float]:
    X = np.linspace(0.0, p.L, n_grid)
    dx = p.L / (n_grid - 1)
    return X, dx


def downstream_equilibrium_depth_from_q(Q: float, p: CACParams) -> float:
    """Compute Hbar(L) from the downstream equilibrium gate relation."""
    Q = float(Q)
    if not np.isfinite(Q) or Q <= 0.0:
        raise ValueError("The equilibrium discharge must be positive and finite.")

    kinetic_head = (
        Q**2 / (2.0 * p.g * p.B**2 * p.muL**2)
    ) ** (1.0 / 3.0)
    return float(p.h_s + p.U_L_bar + kinetic_head)


def integrate_steady_fixed_q_ode(
    X: np.ndarray,
    p: CACParams,
    Q_candidate: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Integrate the steady Saint-Venant ODE backward from x=L to x=0."""
    X = np.asarray(X, dtype=float)
    if X.ndim != 1 or X.size < 2:
        raise ValueError("X must be a one-dimensional grid with at least two points.")
    if not np.all(np.diff(X) > 0.0):
        raise ValueError("X must be strictly increasing.")

    Q_candidate = float(Q_candidate)
    if not np.isfinite(Q_candidate) or Q_candidate <= 0.0:
        raise ValueError("Q_candidate must be positive and finite.")

    Hbar_L = downstream_equilibrium_depth_from_q(Q_candidate, p)

    def rhs(_x: float, state: np.ndarray) -> np.ndarray:
        H = float(state[0])
        Q = float(state[1])

        if not np.isfinite(H) or not np.isfinite(Q):
            raise FloatingPointError("Non-finite state in steady reference ODE.")
        if H <= float(p.H_min):
            raise FloatingPointError("Steady reference ODE violates minimum depth.")

        area = p.B * H
        hydraulic_radius = p.B * H / (p.B + 2.0 * H)
        friction_slope = Q * abs(Q) / (
            p.K**2 * area**2 * hydraulic_radius ** (4.0 / 3.0)
        )

        denominator = p.g * p.B * H - Q**2 / (p.B * H**2)
        if abs(denominator) <= 1e-10:
            raise FloatingPointError(
                "Steady ODE approaches the critical-flow singularity."
            )

        dHdx = p.g * p.B * H * (p.I - friction_slope) / denominator
        if not np.isfinite(dHdx):
            raise FloatingPointError("Non-finite dHbar/dx in steady ODE.")

        return np.asarray([dHdx, 0.0], dtype=float)

    sol = solve_ivp(
        fun=rhs,
        t_span=(float(X[-1]), float(X[0])),
        y0=np.asarray([Hbar_L, Q_candidate], dtype=float),
        t_eval=X[::-1],
        rtol=1e-9,
        atol=1e-11,
        max_step=p.L / 1000.0,
    )

    if not sol.success:
        raise RuntimeError(f"Failed to solve steady ODE: {sol.message}")
    if sol.y.shape[1] != X.size:
        raise RuntimeError("Steady ODE did not return all requested grid points.")

    Hbar = np.asarray(sol.y[0][::-1], dtype=float)
    Qbar = np.asarray(sol.y[1][::-1], dtype=float)

    if not np.all(np.isfinite(Hbar)) or not np.all(np.isfinite(Qbar)):
        raise FloatingPointError("Steady reference contains NaN or Inf.")
    if float(np.min(Hbar)) <= float(p.H_min):
        raise RuntimeError("Steady Hbar profile violates the minimum depth.")

    return Hbar, Qbar


def solve_steady_reference_ode(
    X: np.ndarray,
    p: CACParams,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Solve Hbar(x) for prescribed spatially constant Qbar=11.93."""
    q_bar_fixed = float(q_bar_fixed)
    Hbar, _ = integrate_steady_fixed_q_ode(X, p, q_bar_fixed)
    Qbar = np.full_like(X, q_bar_fixed, dtype=float)
    return Hbar, Qbar, q_bar_fixed


def make_reference_profiles(
    X: np.ndarray,
    p: CACParams,
    reference: str = REFERENCE_MODE,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    if reference in {"ode", "strict"}:
        return solve_steady_reference_ode(X, p, q_bar_fixed=q_bar_fixed)
    if reference == "linear":
        Hbar = p.H_bar_0 + (p.H_bar_L - p.H_bar_0) * (X / p.L)
        Qbar = np.full_like(X, q_bar_fixed, dtype=float)
        return Hbar, Qbar, float(q_bar_fixed)
    raise ValueError("reference must be 'ode', 'strict', or 'linear'.")


def build_collection_matched_reference():
    """Build an internally consistent p, X, Hbar, and Qbar tuple."""
    nominal_p = make_cac_table_params(q_bar=Q_BAR_FIXED)
    X, dx = make_grid(nominal_p, N)
    Hbar, Qbar, q_equilibrium = make_reference_profiles(
        X,
        nominal_p,
        reference=REFERENCE_MODE,
        q_bar_fixed=Q_BAR_FIXED,
    )

    p = make_cac_table_params(
        q_bar=q_equilibrium,
        h_bar_0=float(Hbar[0]),
        h_bar_L=float(Hbar[-1]),
    )
    return p, X, dx, Hbar, Qbar


# ============================================================
# 6. Chebyshev initial conditions
# ============================================================

def sample_truncated_chebyshev(
    xi: np.ndarray,
    rng,
    order: int,
    max_amplitude: float,
    coefficient_decay: float,
) -> np.ndarray:
    """Generate one smooth truncated-Chebyshev perturbation."""
    if order < 0:
        raise ValueError("CHEB_ORDER must be nonnegative.")
    if max_amplitude < 0.0:
        raise ValueError("Chebyshev amplitude must be nonnegative.")
    if coefficient_decay < 0.0:
        raise ValueError("CHEB_COEFF_DECAY must be nonnegative.")
    if max_amplitude == 0.0:
        return np.zeros_like(xi, dtype=float)

    mode_index = np.arange(order + 1, dtype=float)
    coefficient_scale = 1.0 / np.power(mode_index + 1.0, coefficient_decay)
    coefficients = rng.normal(loc=0.0, scale=coefficient_scale)

    perturbation = np.polynomial.chebyshev.chebval(xi, coefficients)
    max_abs = float(np.max(np.abs(perturbation)))
    if max_abs <= 1e-12:
        return np.zeros_like(xi, dtype=float)

    # Match the data collector/new Hybrid program: the realized maximum
    # amplitude is randomly selected from 35% to 100% of the configured bound.
    realized_amplitude = rng.uniform(0.35, 1.0) * max_amplitude
    return perturbation * (realized_amplitude / max_abs)


def random_initial_condition_func(X, Hbar, Qbar, rng):
    """Generate H0,Q0 with the same Chebyshev distribution as the new program."""
    X = np.asarray(X, dtype=float)
    Hbar = np.asarray(Hbar, dtype=float)
    Qbar = np.asarray(Qbar, dtype=float)
    xi = 2.0 * X / float(X[-1]) - 1.0

    # Reject the whole candidate instead of clipping individual grid points, so
    # the smooth Chebyshev shape is preserved.
    for _ in range(100):
        H_perturbation = sample_truncated_chebyshev(
            xi,
            rng,
            CHEB_ORDER,
            H_CHEB_AMPLITUDE,
            CHEB_COEFF_DECAY,
        )
        Q_perturbation = sample_truncated_chebyshev(
            xi,
            rng,
            CHEB_ORDER,
            Q_CHEB_AMPLITUDE,
            CHEB_COEFF_DECAY,
        )

        H0 = Hbar + H_perturbation
        Q0 = Qbar + Q_perturbation

        finite = np.all(np.isfinite(H0)) and np.all(np.isfinite(Q0))
        valid_depth = float(np.min(H0)) > 0.20
        valid_discharge = float(np.min(Q0)) > 0.0

        if finite and valid_depth and valid_discharge:
            return H0, Q0

    raise RuntimeError(
        "Failed to generate a valid Chebyshev initial condition after 100 "
        "attempts. Reduce H_CHEB_AMPLITUDE or Q_CHEB_AMPLITUDE."
    )


# ============================================================
# 7. Reward class
# ============================================================

class CACTrackingSmoothReward(BaseReward):
    """
    Discrete reward at the control instant t_k:

        r_k = -alpha * tracking_cost
              -beta  * state_change_cost
              -gamma * control_cost
              -delta * delta_control_cost.

    Definitions:
        tracking_cost
            = tracking_w_H * mean(((H_k-H_eq)/H_scale)^2)
            + tracking_w_Q * mean(((Q_k-Q_eq)/Q_scale)^2)

        state_change_cost
            = change_w_H * mean(((H_k-H_{k-1})/H_scale)^2)
            + change_w_Q * mean(((Q_k-Q_{k-1})/Q_scale)^2)

        control_cost
            = ((U_k-U_eq)/U_scale)^2

        delta_control_cost
            = (((U_k-U_eq)/U_scale)
               -((U_{k-1}-U_eq)/U_scale))^2

    The state-change term is an inter-sample increment penalty. It is not a
    time derivative because it is not divided by the control interval.

    Spatial means are used instead of unnormalized sums so that the reward scale
    does not grow with the number of grid points N.
    """

    def __init__(
        self,
        H_eq,
        Q_eq,
        delta_t=control_sample_rate,
        H_scale=0.30,
        Q_scale=3.0,
        U_scale=0.10,
        alpha=1.0,
        beta=0.10,
        gamma=0.05,
        delta=0.10,
        tracking_w_H=1.0,
        tracking_w_Q=1.0,
        change_w_H=1.0,
        change_w_Q=1.0,
        U_ref=U_EQ_VALUE,
        terminal_tracking_weight=0.0,
        truncate_penalty=-200.0,
    ):
        self.H_eq = np.asarray(H_eq, dtype=float)
        self.Q_eq = np.asarray(Q_eq, dtype=float)
        self.N = len(self.H_eq)

        # Retained for compatibility with the environment interface.
        self.delta_t = float(delta_t)

        self.H_scale = float(H_scale)
        self.Q_scale = float(Q_scale)
        self.U_scale = float(U_scale)

        self.alpha = float(alpha)
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.delta = float(delta)

        self.tracking_w_H = float(tracking_w_H)
        self.tracking_w_Q = float(tracking_w_Q)
        self.change_w_H = float(change_w_H)
        self.change_w_Q = float(change_w_Q)

        self.U_ref = float(U_ref)
        self.terminal_tracking_weight = float(terminal_tracking_weight)
        self.truncate_penalty = float(truncate_penalty)

        self.prev_H: Optional[np.ndarray] = None
        self.prev_Q: Optional[np.ndarray] = None
        self.prev_action: Optional[float] = None

        self.last_terms = {
            "tracking_cost": 0.0,
            "state_change_cost": 0.0,
            "control_cost": 0.0,
            "delta_control_cost": 0.0,
        }

    def _reset_memory(self):
        self.prev_H = None
        self.prev_Q = None
        self.prev_action = None

    def _split_state(self, state_vec):
        state_vec = np.asarray(state_vec, dtype=float).reshape(-1)
        if state_vec.size < 2 * self.N:
            raise ValueError(
                f"PDE state length is {state_vec.size}, expected at least {2 * self.N}."
            )
        H = state_vec[:self.N]
        Q = state_vec[self.N:2 * self.N]
        return H, Q

    def _action_value(self, action):
        # The PDE environment passes the physical gate U_L to the reward.
        if action is None:
            return self.U_ref
        value = float(np.asarray(action).reshape(-1)[0])
        return float(
            np.nan_to_num(
                value,
                nan=self.U_ref,
                posinf=self.U_ref,
                neginf=self.U_ref,
            )
        )

    def reward(
        self,
        uVec=None,
        time_index=None,
        terminate=None,
        truncate=None,
        action=None,
        control_sample_rate=None,
    ):
        if uVec is None or time_index is None:
            return 0.0

        if int(time_index) <= 1:
            self._reset_memory()

        if truncate:
            self._reset_memory()
            return float(self.truncate_penalty)

        Hk, Qk = self._split_state(uVec[time_index])

        if (
            not np.all(np.isfinite(Hk))
            or not np.all(np.isfinite(Qk))
            or np.any(Hk <= 0.0)
        ):
            self._reset_memory()
            return float(self.truncate_penalty)

        H_error_tilde = (Hk - self.H_eq) / self.H_scale
        Q_error_tilde = (Qk - self.Q_eq) / self.Q_scale

        tracking_cost = (
            self.tracking_w_H * np.mean(H_error_tilde ** 2)
            + self.tracking_w_Q * np.mean(Q_error_tilde ** 2)
        )

        if self.prev_H is None or self.prev_Q is None:
            state_change_cost = 0.0
        else:
            delta_H_tilde = (Hk - self.prev_H) / self.H_scale
            delta_Q_tilde = (Qk - self.prev_Q) / self.Q_scale

            state_change_cost = (
                self.change_w_H * np.mean(delta_H_tilde ** 2)
                + self.change_w_Q * np.mean(delta_Q_tilde ** 2)
            )

        Uk = self._action_value(action)
        U_tilde = (Uk - self.U_ref) / self.U_scale
        control_cost = U_tilde ** 2

        if self.prev_action is None:
            delta_control_cost = 0.0
        else:
            U_prev_tilde = (self.prev_action - self.U_ref) / self.U_scale
            delta_control_cost = (U_tilde - U_prev_tilde) ** 2

        reward_value = (
            -self.alpha * tracking_cost
            -self.beta * state_change_cost
            -self.gamma * control_cost
            -self.delta * delta_control_cost
        )

        if terminate:
            reward_value -= self.terminal_tracking_weight * tracking_cost

        self.prev_H = Hk.copy()
        self.prev_Q = Qk.copy()
        self.prev_action = Uk

        self.last_terms = {
            "tracking_cost": float(tracking_cost),
            "state_change_cost": float(state_change_cost),
            "control_cost": float(control_cost),
            "delta_control_cost": float(delta_control_cost),
        }

        return float(reward_value)


# ============================================================
# 8. Embedded Hadamard DeepONet feature extractor
# ============================================================

def preprocess_obs(obs: torch.Tensor) -> torch.Tensor:
    if not isinstance(obs, torch.Tensor):
        obs = torch.as_tensor(obs)
    return obs.float()


class BranchNet(nn.Module):
    def __init__(self, shape=N, output_dim=LATENT_DIM):
        super().__init__()
        self.shape = int(shape)
        self.output_dim = int(output_dim)

        self.conv1 = nn.Conv1d(
            in_channels=2,
            out_channels=16,
            kernel_size=5,
            stride=2,
        )
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(
            in_channels=16,
            out_channels=32,
            kernel_size=5,
            stride=2,
        )

        with torch.no_grad():
            tmp = torch.zeros(1, 2, self.shape)
            tmp = self.relu(self.conv1(tmp))
            tmp = self.relu(self.conv2(tmp))
            flat_dim = tmp.reshape(1, -1).shape[1]

        self.fc1 = nn.Linear(flat_dim, self.output_dim)

    def forward(self, x):
        x = preprocess_obs(x)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        if x.shape[1] != 2 * self.shape:
            raise ValueError(
                f"Branch input dimension is {x.shape[1]}, "
                f"expected {2 * self.shape}."
            )

        x = x.reshape(x.shape[0], 2, self.shape)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = self.relu(self.fc1(x))
        return x


class TrunkNet(nn.Module):
    def __init__(self, dim_x=1, width=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_x, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, width),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x.float())


class DeepONetModified(nn.Module):
    """
    Hadamard-fusion DeepONet:

        b(H,Q) in R^p,
        t(L)   in R^p,
        z      = b(H,Q) * t(L) in R^p.

    The physical-gate control head is retained only so that the complete
    pretrained checkpoint can be loaded strictly. It is not used to form the
    SAC action in this method.
    """

    def __init__(self, shape=N, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = int(latent_dim)

        self.branch = BranchNet(shape, output_dim=self.latent_dim)
        self.trunk = TrunkNet(dim_x=1, width=self.latent_dim)

        self.fc3 = nn.Linear(self.latent_dim, 32)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, 1)

    def extract_latent(self, obs, grid):
        b = self.branch(obs)
        t = self.trunk(grid)

        if t.shape[0] == 1:
            t = t.expand(b.shape[0], -1)
        elif t.shape[0] != b.shape[0]:
            raise ValueError(
                f"Branch batch {b.shape[0]} and trunk batch "
                f"{t.shape[0]} do not match."
            )

        return b * t

    def control_head(self, z):
        y = self.relu(self.fc3(z))
        y = self.relu(self.fc4(y))
        return self.fc5(y)

    def forward(self, obs, grid, return_latent=False):
        z = self.extract_latent(obs, grid)
        u_physical = self.control_head(z)
        if return_latent:
            return u_physical, z
        return u_physical


class DeepONetSACFeatureExtractor(BaseFeaturesExtractor):
    """
    Representation-only DeepONet embedding.

    Input:
        raw [H_1,...,H_N,Q_1,...,Q_N].

    Output to the standard SAC actor and critic:
        z = b(H,Q) * t(L) in R^102.

    The DeepONet physical control output is not blended with the SAC action.
    """

    def __init__(
        self,
        observation_space: spaces.Box,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        query_point: float = 1.0,
        freeze_deeponet: bool = False,
    ):
        if features_dim != SAC_FEATURE_DIM:
            raise ValueError(
                f"features_dim must be {SAC_FEATURE_DIM}, got {features_dim}."
            )

        super().__init__(observation_space, features_dim)

        self.query_point = float(query_point)
        self.net1 = DeepONetModified(N, latent_dim=LATENT_DIM)

        self.register_buffer(
            "grid",
            torch.tensor([[self.query_point]], dtype=torch.float32),
        )

        self._load_pretrained(pretrained_path)
        self._apply_freeze(freeze_all=freeze_deeponet)

    def _apply_freeze(self, freeze_all=False):
        if freeze_all:
            for param in self.net1.parameters():
                param.requires_grad = False
            print("All DeepONet parameters are frozen.")
            return

        # Same representation-transfer strategy as the latest program.
        for param in self.net1.branch.conv1.parameters():
            param.requires_grad = False
        for param in self.net1.branch.conv2.parameters():
            param.requires_grad = False

        for param in self.net1.branch.fc1.parameters():
            param.requires_grad = True

        for param in self.net1.trunk.parameters():
            param.requires_grad = False

        last_linear = None
        for module in reversed(list(self.net1.trunk.net)):
            if isinstance(module, nn.Linear):
                last_linear = module
                break

        if last_linear is not None:
            for param in last_linear.parameters():
                param.requires_grad = True

        # This method does not use the DeepONet control head.
        for layer in [self.net1.fc3, self.net1.fc4, self.net1.fc5]:
            for param in layer.parameters():
                param.requires_grad = False

        trainable = sum(
            p.numel() for p in self.net1.parameters() if p.requires_grad
        )
        total = sum(p.numel() for p in self.net1.parameters())

        print(
            f"DeepONet feature transfer: trainable={trainable}/{total} "
            f"({100.0 * trainable / max(total, 1):.2f}%)."
        )
        print("Frozen: Branch conv1/conv2, previous Trunk layers, control head.")
        print("Trainable: Branch fc1 and the last Trunk Linear layer.")

    @staticmethod
    def _clean_state_dict_keys(state_dict):
        cleaned = {}
        prefixes = ["module.", "model.", "net1.", "deeponet."]

        for key, value in state_dict.items():
            new_key = key
            changed = True
            while changed:
                changed = False
                for prefix in prefixes:
                    if new_key.startswith(prefix):
                        new_key = new_key[len(prefix):]
                        changed = True
            cleaned[new_key] = value

        return cleaned

    def _load_pretrained(self, pretrained_path: str):
        path = Path(pretrained_path)

        if not path.exists():
            raise FileNotFoundError(
                f"Pretrained Hadamard DeepONet checkpoint not found: {path}"
            )

        state_dict = torch.load(str(path), map_location="cpu")

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]

        if not isinstance(state_dict, dict):
            raise TypeError(
                f"Unsupported checkpoint type: {type(state_dict).__name__}"
            )

        cleaned = self._clean_state_dict_keys(state_dict)

        try:
            self.net1.load_state_dict(cleaned, strict=True)
        except RuntimeError as exc:
            raise RuntimeError(
                "The checkpoint is incompatible with the Hadamard-fusion "
                "DeepONet. The required architecture uses z=b*t and "
                "fc3=Linear(102,32).\n"
                f"Original error:\n{exc}"
            ) from exc

        print(f"Loaded pretrained Hadamard DeepONet: {path}")

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        obs = preprocess_obs(observations)
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)

        if obs.shape[1] != RAW_STATE_DIM:
            raise ValueError(
                f"Raw observation dimension is {obs.shape[1]}, "
                f"expected {RAW_STATE_DIM}."
            )

        z = self.net1.extract_latent(
            obs,
            self.grid.to(obs.device),
        )
        return z.view(z.size(0), -1)


# ============================================================
# 9. Separate learning rate for DeepONet representation
# ============================================================

def _make_param_groups_with_deeponet_lr(module: nn.Module):
    deeponet_params = []
    other_params = []

    for name, param in module.named_parameters():
        if not param.requires_grad:
            continue

        if "features_extractor.net1" in name or name.startswith("net1."):
            deeponet_params.append(param)
        else:
            other_params.append(param)

    groups = []

    if other_params:
        groups.append({
            "params": other_params,
            "lr": learning_rate,
            "is_deeponet_group": False,
        })

    if deeponet_params:
        groups.append({
            "params": deeponet_params,
            "lr": deeponet_learning_rate,
            "is_deeponet_group": True,
        })

    if not groups:
        raise RuntimeError("No trainable parameters were found for the optimizer.")

    return groups


def set_deeponet_small_lr_optimizers(model):
    actor_groups = _make_param_groups_with_deeponet_lr(model.actor)
    critic_groups = _make_param_groups_with_deeponet_lr(model.critic)

    model.actor.optimizer = torch.optim.Adam(actor_groups, eps=1e-5)
    model.critic.optimizer = torch.optim.Adam(critic_groups, eps=1e-5)

    n_actor_deep = sum(
        p.numel()
        for group in actor_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_actor_other = sum(
        p.numel()
        for group in actor_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_deep = sum(
        p.numel()
        for group in critic_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_other = sum(
        p.numel()
        for group in critic_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )

    print("Custom optimizers with separate DeepONet learning rate are set.")
    print(f"Actor params: DeepONet={n_actor_deep}, other={n_actor_other}")
    print(f"Critic params: DeepONet={n_critic_deep}, other={n_critic_other}")
    print(
        f"Base SAC learning_rate={learning_rate}, "
        f"DeepONet learning_rate={deeponet_learning_rate}"
    )


class DeepONetFeatureSAC(SAC):
    """Keep the DeepONet parameter groups at their smaller learning rate."""

    def _update_learning_rate(self, optimizers):
        super()._update_learning_rate(optimizers)

        if not isinstance(optimizers, (list, tuple)):
            optimizers = [optimizers]

        current_lr = self.lr_schedule(self._current_progress_remaining)

        for optimizer in optimizers:
            for group in optimizer.param_groups:
                if group.get("is_deeponet_group", False):
                    group["lr"] = deeponet_learning_rate
                elif "is_deeponet_group" in group:
                    group["lr"] = current_lr


# ============================================================
# 10. Environment creation
# ============================================================

def make_env():
    p, X, dx, H_eq, Q_eq = build_collection_matched_reference()

    # These constructor values are fallbacks only. Every reset uses the
    # Chebyshev initial-condition generator below.
    H_init = H_eq.copy()
    Q_init = Q_eq.copy()

    reward_class = CACTrackingSmoothReward(
        H_eq=H_eq,
        Q_eq=Q_eq,
        delta_t=control_sample_rate,
        H_scale=H_scale,
        Q_scale=Q_scale,
        U_scale=U_scale,
        alpha=reward_alpha,
        beta=reward_beta,
        gamma=reward_gamma,
        delta=reward_delta,
        tracking_w_H=tracking_w_H,
        tracking_w_Q=tracking_w_Q,
        change_w_H=change_w_H,
        change_w_Q=change_w_Q,
        U_ref=U_EQ_VALUE,
        terminal_tracking_weight=terminal_tracking_weight,
        truncate_penalty=truncate_penalty,
    )

    env = CACSaintVenantPDE1D(
        params=p,
        X_grid=X,
        Hbar=H_eq,
        Qbar=Q_eq,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=random_initial_condition_func,
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=reward_class,
        normalize=True,
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    # Expose the exact ODE reference for diagnostics and test-program reuse.
    env.reference_X = X
    env.reference_H = H_eq
    env.reference_Q = Q_eq
    env.reference_dx = dx

    print(
        "Collection-matched ODE reference: "
        f"Hbar(0)={H_eq[0]:.9f}, Hbar(L)={H_eq[-1]:.9f}, "
        f"Qbar={Q_eq[0]:.9f}, dx={dx:.6f}"
    )

    return env

# ============================================================
# 11. Training
# ============================================================

def train_with_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    env = make_env()

    reset_obs, _ = env.reset(seed=seed)
    reset_obs = np.asarray(reset_obs, dtype=np.float32).reshape(-1)
    print(f"Raw observation shape: {reset_obs.shape}")

    if reset_obs.shape != (RAW_STATE_DIM,):
        raise RuntimeError(
            f"Expected raw observation shape {(RAW_STATE_DIM,)}, "
            f"got {reset_obs.shape}."
        )

    policy_kwargs = dict(
        features_extractor_class=DeepONetSACFeatureExtractor,
        features_extractor_kwargs=dict(
            features_dim=SAC_FEATURE_DIM,
            pretrained_path=DEEPNET_PRETRAINED_PATH,
            query_point=1.0,
            freeze_deeponet=False,
        ),
        net_arch=dict(pi=[256, 256], qf=[256, 256]),
        optimizer_kwargs={"eps": 1e-6},
        share_features_extractor=False,
    )

    model = DeepONetFeatureSAC(
        policy="MlpPolicy",
        policy_kwargs=policy_kwargs,
        env=env,
        learning_rate=learning_rate,
        buffer_size=buffer_size,
        batch_size=batch_size,
        tau=tau,
        gamma=sac_gamma,
        # Keep these commented to match the latest comparison programs.
        # ent_coef=ent_coef,
        # learning_starts=learning_starts,
        verbose=1,
        tensorboard_log=f"./tbsactrainx/seed_{seed}/",
    )

    set_deeponet_small_lr_optimizers(model)

    checkpoint_callback = CheckpointCallback(
        save_freq=10000,
        save_path=f"./sactrainx/seed_{seed}/",
        name_prefix="sac_cac_deeponet_feature_raw_hq_ode_cheb",
        save_replay_buffer=True,
        save_vecnormalize=True,
    )

    start_time = time.perf_counter()

    model.learn(
        total_timesteps=total_timesteps,
        callback=checkpoint_callback,
    )

    training_time = time.perf_counter() - start_time
    hours = int(training_time // 3600)
    minutes = int((training_time % 3600) // 60)
    seconds = training_time % 60

    print("\n" + "=" * 70)
    print(f"Seed {seed} training completed")
    print(
        f"Training time for this seed: "
        f"{hours:02d} h {minutes:02d} min {seconds:06.2f} s"
    )
    print(f"Total training time: {training_time:.2f} seconds")
    print("=" * 70)

    save_path = (
        f"./sactrainx/seed_{seed}/"
        "sac_cac_deeponet_feature_raw_hq_ode_cheb_final"
    )
    model.save(save_path)
    env.close()

    return model, save_path


def main():
    print("DeepONet-feature SAC: raw H/Q + ODE reference + Chebyshev IC")
    print("=" * 90)
    print(f"N={N}, T={T}, dt={dt}, control_sample_rate={control_sample_rate}")
    print(f"Physical gate U_L in [{min_control_value}, {max_control_value}]")
    print("Method: DeepONet is used only as the SAC feature extractor")
    print("Final action: standard SAC actor output; no prior-action mixing")
    print("Observation: raw physical [H_1,...,H_N,Q_1,...,Q_N]")
    print(f"Observation dimension={RAW_STATE_DIM}")
    print(
        f"DeepONet feature: z=b(H,Q)*t(L), dimension={SAC_FEATURE_DIM}"
    )
    print(f"DeepONet checkpoint: {DEEPNET_PRETRAINED_PATH}")
    print(
        "Transfer strategy: freeze Branch conv1/conv2 and previous Trunk "
        "layers; train Branch fc1 and the last Trunk Linear layer"
    )
    print(
        f"Reference: Qbar={Q_BAR_FIXED} fixed; Hbar solved by steady ODE "
        f"(mode={REFERENCE_MODE})"
    )
    print(
        "Reward: -alpha*tracking - beta*state_change "
        "- gamma*control - delta*delta_control"
    )
    print(
        f"Reward weights: alpha={reward_alpha}, beta={reward_beta}, "
        f"gamma={reward_gamma}, delta={reward_delta}"
    )
    print(
        f"Tracking H/Q weights=({tracking_w_H}, {tracking_w_Q}); "
        f"state-change H/Q weights=({change_w_H}, {change_w_Q})"
    )
    print(f"Reward scales: H={H_scale}, Q={Q_scale}, U={U_scale}")
    print(
        "Chebyshev initial conditions: "
        f"order={CHEB_ORDER}, A_H={H_CHEB_AMPLITUDE}, "
        f"A_Q={Q_CHEB_AMPLITUDE}, decay={CHEB_COEFF_DECAY}"
    )
    print(
        f"Learning rates: SAC={learning_rate}, "
        f"DeepONet={deeponet_learning_rate}"
    )

    for index, seed in enumerate(seeds):
        print(f"\n===== Training {index + 1}/{len(seeds)}, seed={seed} =====")
        _, save_path = train_with_seed(seed)
        print(f"Saved model: {save_path}")


if __name__ == "__main__":
    main()


DeepONet-feature SAC: raw H/Q + ODE reference + Chebyshev IC
N=102, T=40000.0, dt=5.0, control_sample_rate=400.0
Physical gate U_L in [0.15, 0.4]
Method: DeepONet is used only as the SAC feature extractor
Final action: standard SAC actor output; no prior-action mixing
Observation: raw physical [H_1,...,H_N,Q_1,...,Q_N]
Observation dimension=204
DeepONet feature: z=b(H,Q)*t(L), dimension=102
DeepONet checkpoint: modedlNet10000QqQqQ14444_hadamard.pt
Transfer strategy: freeze Branch conv1/conv2 and previous Trunk layers; train Branch fc1 and the last Trunk Linear layer
Reference: Qbar=11.93 fixed; Hbar solved by steady ODE (mode=ode)
Reward: -alpha*tracking - beta*state_change - gamma*control - delta*delta_control
Reward weights: alpha=1.0, beta=0.1, gamma=0.05, delta=0.1
Tracking H/Q weights=(1.0, 1.0); state-change H/Q weights=(1.0, 1.0)
Reward scales: H=0.3, Q=3.0, U=0.1
Chebyshev initial conditions: order=6, A_H=0.3, A_Q=2.5, decay=2.0
Learning rates: SAC=0.0001, DeepONet=1e-06

=====

In [4]:
# -*- coding: utf-8 -*-
"""
Hybrid SAC using the NEW Hadamard-fusion DeepONet for the CAC Saint-Venant system.

All environment, equilibrium, initial-condition, state-space, reward, action
range, SAC hyperparameters, and random seeds are aligned with the latest
raw-HQ TDPSAC training program.

The method retained from the uploaded hybrid program is:

1. The new DeepONet supplies the Hadamard-fused feature
       z = Branch(H,Q) * Trunk(L).
2. The same new DeepONet supplies a physical downstream-gate prior U_prior,
   pretrained from y_ul.dat, which is mapped to the normalized action interval.
3. The applied normalized action remains
       a_hybrid = (1-alpha_hybrid)*a_prior + alpha_hybrid*a_SAC.
4. Unlike the latest partial-freeze program, all DeepONet parameters remain
   trainable with the smaller DeepONet learning rate 1e-6. This preserves the
   transfer/fine-tuning method of the uploaded program while using the new
   DeepONet architecture and checkpoint.

The observation is raw [H,Q] with no cumulative-error state.

Required external module:
    cac_saint_venant_env_no_bs.py
"""

from __future__ import annotations

import os
import random
import time
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import torch
from scipy.integrate import solve_ivp
from gymnasium import spaces
from torch import nn

from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import (
    BaseCallback,
    CallbackList,
    CheckpointCallback,
)
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.sac.policies import Actor, SACPolicy

from cac_saint_venant_env_no_bs import CACParams, CACSaintVenantPDE1D

try:
    from pde_control_gym.src.rewards.base_reward import BaseReward
except Exception:
    class BaseReward:
        pass


# ============================================================
# 1. Simulation parameters
# ============================================================

N = 102
T = 40000.0
dt = 5.0
control_sample_rate = 400.0

max_state_value = 1e6

# The PDE environment maps normalized SAC actions a in [-1,1] to the physical
# downstream gate U_L in this interval. These limits match the data collector.
min_control_value = 0.15
max_control_value = 0.40

# State/reward normalization scales. H and Q remain raw in the observation;
# these scales are used only inside the reward calculation.
H_scale = 0.30
Q_scale = 3.0
U_scale = 0.10

obs_clip = 50.0

cfl = 0.45
check_cfl = True
well_balanced = True


# ============================================================
# 2. Collection-matched equilibrium and initial-condition settings
# ============================================================

Q_BAR_FIXED = 11.93
REFERENCE_MODE = "ode"
U_EQ_VALUE = 0.30

CHEB_ORDER = 6
H_CHEB_AMPLITUDE = 0.30
Q_CHEB_AMPLITUDE = 2.50
CHEB_COEFF_DECAY = 2.0


# ============================================================
# 3. SAC parameters
# ============================================================

total_timesteps = int(1e5)
learning_rate = 1e-4

# Use a much smaller learning rate for trainable DeepONet parameters.
deeponet_learning_rate = 1e-6

buffer_size = 100000
batch_size = 256
tau = 0.005
sac_gamma = 0.995
ent_coef = 0.005
learning_starts = 5000

seeds = [42,43,44,45,46]

LATENT_DIM = 102
RAW_STATE_DIM = 2 * N
SAC_FEATURE_DIM = LATENT_DIM

# This checkpoint must come from the Hadamard-fusion DeepONet pretraining code.
DEEPNET_PRETRAINED_PATH = "modedlNet10000QqQqQ14444_hadamard.pt"


# ============================================================
# 4. Reward parameters
# ============================================================

# r_k = -alpha*tracking - beta*state_change - gamma*control - delta*delta_control
reward_alpha = 1.0
reward_beta = 0.10
reward_gamma = 0.05
reward_delta = 0.10

# Separate H/Q weights for tracking and inter-sample state-change terms.
tracking_w_H = 1.0
tracking_w_Q = 1.0
change_w_H = 1.0
change_w_Q = 1.0

terminal_tracking_weight = 0.0
truncate_penalty = -200.0


# ============================================================
# 5. CAC physical parameters, ODE reference, and Chebyshev initial conditions
# ============================================================

def make_cac_table_params(
    q_bar: float = Q_BAR_FIXED,
    h_bar_0: float = 3.80,
    h_bar_L: float = 4.70,
) -> CACParams:
    """Create the CAC parameter object used by collection and RL training."""
    return CACParams(
        L=11239.0,
        B=40.0,
        g=9.81,
        I=7.92e-5,
        K=33.0,
        mu0=0.4,
        muL=0.4,
        z_up=5.00,
        U0_bar=0.1548,
        U_L_bar=U_EQ_VALUE,
        h_s=4.094,
        Q_bar=float(q_bar),
        H_bar_0=float(h_bar_0),
        H_bar_L=float(h_bar_L),
        H_min=0.20,
        max_state_value=1e10,
    )


def make_grid(p: CACParams, n_grid: int) -> Tuple[np.ndarray, float]:
    X = np.linspace(0.0, p.L, n_grid)
    dx = p.L / (n_grid - 1)
    return X, dx


def downstream_equilibrium_depth_from_q(Q: float, p: CACParams) -> float:
    """Compute Hbar(L) from the downstream equilibrium gate relation."""
    Q = float(Q)
    if not np.isfinite(Q) or Q <= 0.0:
        raise ValueError("The equilibrium discharge must be positive and finite.")

    kinetic_head = (
        Q**2 / (2.0 * p.g * p.B**2 * p.muL**2)
    ) ** (1.0 / 3.0)
    return float(p.h_s + p.U_L_bar + kinetic_head)


def integrate_steady_fixed_q_ode(
    X: np.ndarray,
    p: CACParams,
    Q_candidate: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Integrate the steady Saint-Venant ODE backward from x=L to x=0."""
    X = np.asarray(X, dtype=float)
    if X.ndim != 1 or X.size < 2:
        raise ValueError("X must be a one-dimensional grid with at least two points.")
    if not np.all(np.diff(X) > 0.0):
        raise ValueError("X must be strictly increasing.")

    Q_candidate = float(Q_candidate)
    if not np.isfinite(Q_candidate) or Q_candidate <= 0.0:
        raise ValueError("Q_candidate must be positive and finite.")

    Hbar_L = downstream_equilibrium_depth_from_q(Q_candidate, p)

    def rhs(_x: float, state: np.ndarray) -> np.ndarray:
        H = float(state[0])
        Q = float(state[1])

        if not np.isfinite(H) or not np.isfinite(Q):
            raise FloatingPointError("Non-finite state in steady reference ODE.")
        if H <= float(p.H_min):
            raise FloatingPointError("Steady reference ODE violates minimum depth.")

        area = p.B * H
        hydraulic_radius = p.B * H / (p.B + 2.0 * H)
        friction_slope = Q * abs(Q) / (
            p.K**2 * area**2 * hydraulic_radius ** (4.0 / 3.0)
        )

        denominator = p.g * p.B * H - Q**2 / (p.B * H**2)
        if abs(denominator) <= 1e-10:
            raise FloatingPointError(
                "Steady ODE approaches the critical-flow singularity."
            )

        dHdx = p.g * p.B * H * (p.I - friction_slope) / denominator
        if not np.isfinite(dHdx):
            raise FloatingPointError("Non-finite dHbar/dx in steady ODE.")

        return np.asarray([dHdx, 0.0], dtype=float)

    sol = solve_ivp(
        fun=rhs,
        t_span=(float(X[-1]), float(X[0])),
        y0=np.asarray([Hbar_L, Q_candidate], dtype=float),
        t_eval=X[::-1],
        rtol=1e-9,
        atol=1e-11,
        max_step=p.L / 1000.0,
    )

    if not sol.success:
        raise RuntimeError(f"Failed to solve steady ODE: {sol.message}")
    if sol.y.shape[1] != X.size:
        raise RuntimeError("Steady ODE did not return all requested grid points.")

    Hbar = np.asarray(sol.y[0][::-1], dtype=float)
    Qbar = np.asarray(sol.y[1][::-1], dtype=float)

    if not np.all(np.isfinite(Hbar)) or not np.all(np.isfinite(Qbar)):
        raise FloatingPointError("Steady reference contains NaN or Inf.")
    if float(np.min(Hbar)) <= float(p.H_min):
        raise RuntimeError("Steady Hbar profile violates the minimum depth.")

    return Hbar, Qbar


def solve_steady_reference_ode(
    X: np.ndarray,
    p: CACParams,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Solve Hbar(x) for prescribed spatially constant Qbar=11.93."""
    q_bar_fixed = float(q_bar_fixed)
    Hbar, _ = integrate_steady_fixed_q_ode(X, p, q_bar_fixed)
    Qbar = np.full_like(X, q_bar_fixed, dtype=float)
    return Hbar, Qbar, q_bar_fixed


def make_reference_profiles(
    X: np.ndarray,
    p: CACParams,
    reference: str = REFERENCE_MODE,
    q_bar_fixed: float = Q_BAR_FIXED,
) -> Tuple[np.ndarray, np.ndarray, float]:
    if reference in {"ode", "strict"}:
        return solve_steady_reference_ode(X, p, q_bar_fixed=q_bar_fixed)
    if reference == "linear":
        Hbar = p.H_bar_0 + (p.H_bar_L - p.H_bar_0) * (X / p.L)
        Qbar = np.full_like(X, q_bar_fixed, dtype=float)
        return Hbar, Qbar, float(q_bar_fixed)
    raise ValueError("reference must be 'ode', 'strict', or 'linear'.")


def build_collection_matched_reference():
    """Build one internally consistent p, X, Hbar, Qbar tuple."""
    nominal_p = make_cac_table_params(q_bar=Q_BAR_FIXED)
    X, dx = make_grid(nominal_p, N)
    Hbar, Qbar, q_equilibrium = make_reference_profiles(
        X,
        nominal_p,
        reference=REFERENCE_MODE,
        q_bar_fixed=Q_BAR_FIXED,
    )

    p = make_cac_table_params(
        q_bar=q_equilibrium,
        h_bar_0=float(Hbar[0]),
        h_bar_L=float(Hbar[-1]),
    )
    return p, X, dx, Hbar, Qbar


def sample_truncated_chebyshev(
    xi: np.ndarray,
    rng,
    order: int,
    max_amplitude: float,
    coefficient_decay: float,
) -> np.ndarray:
    """Sample the same smooth truncated-Chebyshev perturbation as collection."""
    if order < 0:
        raise ValueError("CHEB_ORDER must be nonnegative.")
    if max_amplitude < 0.0:
        raise ValueError("Chebyshev amplitude must be nonnegative.")
    if coefficient_decay < 0.0:
        raise ValueError("CHEB_COEFF_DECAY must be nonnegative.")
    if max_amplitude == 0.0:
        return np.zeros_like(xi, dtype=float)

    mode_index = np.arange(order + 1, dtype=float)
    coefficient_scale = 1.0 / np.power(mode_index + 1.0, coefficient_decay)
    coefficients = rng.normal(loc=0.0, scale=coefficient_scale)

    perturbation = np.polynomial.chebyshev.chebval(xi, coefficients)
    max_abs = float(np.max(np.abs(perturbation)))
    if max_abs <= 1e-12:
        return np.zeros_like(xi, dtype=float)

    realized_amplitude = rng.uniform(0.35, 1.0) * max_amplitude
    return perturbation * (realized_amplitude / max_abs)


def random_initial_condition_func(X, Hbar, Qbar, rng):
    """Generate H0,Q0 with the same Chebyshev distribution as collection."""
    X = np.asarray(X, dtype=float)
    Hbar = np.asarray(Hbar, dtype=float)
    Qbar = np.asarray(Qbar, dtype=float)
    xi = 2.0 * X / float(X[-1]) - 1.0

    # Rejection sampling preserves the polynomial shape; no pointwise clipping.
    for _ in range(100):
        H_perturbation = sample_truncated_chebyshev(
            xi,
            rng,
            CHEB_ORDER,
            H_CHEB_AMPLITUDE,
            CHEB_COEFF_DECAY,
        )
        Q_perturbation = sample_truncated_chebyshev(
            xi,
            rng,
            CHEB_ORDER,
            Q_CHEB_AMPLITUDE,
            CHEB_COEFF_DECAY,
        )

        H0 = Hbar + H_perturbation
        Q0 = Qbar + Q_perturbation

        finite = np.all(np.isfinite(H0)) and np.all(np.isfinite(Q0))
        valid_depth = float(np.min(H0)) > 0.20
        valid_discharge = float(np.min(Q0)) > 0.0

        if finite and valid_depth and valid_discharge:
            return H0, Q0

    raise RuntimeError(
        "Failed to generate a valid Chebyshev initial condition after 100 "
        "attempts. Reduce H_CHEB_AMPLITUDE or Q_CHEB_AMPLITUDE."
    )


# ============================================================
# 6. Reward class
# ============================================================

class CACTrackingSmoothReward(BaseReward):
    """
    Discrete reward at the control instant t_k:

        r_k = -alpha * tracking_cost
              -beta  * state_change_cost
              -gamma * control_cost
              -delta * delta_control_cost.

    Definitions:
        tracking_cost
            = tracking_w_H * mean(((H_k-H_eq)/H_scale)^2)
            + tracking_w_Q * mean(((Q_k-Q_eq)/Q_scale)^2)

        state_change_cost
            = change_w_H * mean(((H_k-H_{k-1})/H_scale)^2)
            + change_w_Q * mean(((Q_k-Q_{k-1})/Q_scale)^2)

        control_cost
            = ((U_k-U_eq)/U_scale)^2

        delta_control_cost
            = (((U_k-U_eq)/U_scale)
               -((U_{k-1}-U_eq)/U_scale))^2

    The state-change term is an inter-sample increment penalty. It is not a
    time derivative because it is not divided by the control interval.

    Spatial means are used instead of unnormalized sums so that the reward scale
    does not grow with the number of grid points N.
    """

    def __init__(
        self,
        H_eq,
        Q_eq,
        delta_t=control_sample_rate,
        H_scale=0.30,
        Q_scale=3.0,
        U_scale=0.10,
        alpha=1.0,
        beta=0.10,
        gamma=0.05,
        delta=0.10,
        tracking_w_H=1.0,
        tracking_w_Q=1.0,
        change_w_H=1.0,
        change_w_Q=1.0,
        U_ref=U_EQ_VALUE,
        terminal_tracking_weight=0.0,
        truncate_penalty=-200.0,
    ):
        self.H_eq = np.asarray(H_eq, dtype=float)
        self.Q_eq = np.asarray(Q_eq, dtype=float)
        self.N = len(self.H_eq)

        # Kept only for compatibility with the environment interface.
        self.delta_t = float(delta_t)

        self.H_scale = float(H_scale)
        self.Q_scale = float(Q_scale)
        self.U_scale = float(U_scale)

        self.alpha = float(alpha)
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.delta = float(delta)

        self.tracking_w_H = float(tracking_w_H)
        self.tracking_w_Q = float(tracking_w_Q)
        self.change_w_H = float(change_w_H)
        self.change_w_Q = float(change_w_Q)

        self.U_ref = float(U_ref)
        self.terminal_tracking_weight = float(terminal_tracking_weight)
        self.truncate_penalty = float(truncate_penalty)

        self.prev_H: Optional[np.ndarray] = None
        self.prev_Q: Optional[np.ndarray] = None
        self.prev_action: Optional[float] = None

        self.last_terms = {
            "tracking_cost": 0.0,
            "state_change_cost": 0.0,
            "control_cost": 0.0,
            "delta_control_cost": 0.0,
        }

    def _reset_memory(self):
        self.prev_H = None
        self.prev_Q = None
        self.prev_action = None

    def _split_state(self, state_vec):
        state_vec = np.asarray(state_vec, dtype=float).reshape(-1)
        if state_vec.size < 2 * self.N:
            raise ValueError(
                f"PDE state length is {state_vec.size}, expected at least {2 * self.N}."
            )
        H = state_vec[:self.N]
        Q = state_vec[self.N:2 * self.N]
        return H, Q

    def _action_value(self, action):
        # The environment passes the physical downstream gate U_L to the reward.
        if action is None:
            return self.U_ref
        value = float(np.asarray(action).reshape(-1)[0])
        return float(
            np.nan_to_num(
                value,
                nan=self.U_ref,
                posinf=self.U_ref,
                neginf=self.U_ref,
            )
        )

    def reward(
        self,
        uVec=None,
        time_index=None,
        terminate=None,
        truncate=None,
        action=None,
        control_sample_rate=None,
    ):
        if uVec is None or time_index is None:
            return 0.0

        if int(time_index) <= 1:
            self._reset_memory()

        if truncate:
            self._reset_memory()
            return float(self.truncate_penalty)

        Hk, Qk = self._split_state(uVec[time_index])

        if (
            not np.all(np.isfinite(Hk))
            or not np.all(np.isfinite(Qk))
            or np.any(Hk <= 0.0)
        ):
            self._reset_memory()
            return float(self.truncate_penalty)

        # 1) H/Q tracking cost
        H_error_tilde = (Hk - self.H_eq) / self.H_scale
        Q_error_tilde = (Qk - self.Q_eq) / self.Q_scale

        tracking_cost = (
            self.tracking_w_H * np.mean(H_error_tilde ** 2)
            + self.tracking_w_Q * np.mean(Q_error_tilde ** 2)
        )

        # 2) Inter-sample H/Q state-change cost
        if self.prev_H is None or self.prev_Q is None:
            state_change_cost = 0.0
        else:
            delta_H_tilde = (Hk - self.prev_H) / self.H_scale
            delta_Q_tilde = (Qk - self.prev_Q) / self.Q_scale

            state_change_cost = (
                self.change_w_H * np.mean(delta_H_tilde ** 2)
                + self.change_w_Q * np.mean(delta_Q_tilde ** 2)
            )

        # 3) Control-magnitude cost
        Uk = self._action_value(action)
        U_tilde = (Uk - self.U_ref) / self.U_scale
        control_cost = U_tilde ** 2

        # 4) Control-increment cost
        if self.prev_action is None:
            delta_control_cost = 0.0
        else:
            U_prev_tilde = (self.prev_action - self.U_ref) / self.U_scale
            delta_control_cost = (U_tilde - U_prev_tilde) ** 2

        reward_value = (
            -self.alpha * tracking_cost
            -self.beta * state_change_cost
            -self.gamma * control_cost
            -self.delta * delta_control_cost
        )

        if terminate:
            reward_value -= self.terminal_tracking_weight * tracking_cost

        self.prev_H = Hk.copy()
        self.prev_Q = Qk.copy()
        self.prev_action = Uk

        self.last_terms = {
            "tracking_cost": float(tracking_cost),
            "state_change_cost": float(state_change_cost),
            "control_cost": float(control_cost),
            "delta_control_cost": float(delta_control_cost),
        }

        return float(reward_value)


# ============================================================
# 7. DeepONet controller and hybrid SAC modules
# ============================================================

def preprocess_obs(obs: torch.Tensor) -> torch.Tensor:
    if not isinstance(obs, torch.Tensor):
        obs = torch.as_tensor(obs)
    return obs.float()


class BranchNet(nn.Module):
    def __init__(self, shape=N, output_dim=LATENT_DIM):
        super().__init__()
        self.shape = int(shape)
        self.output_dim = int(output_dim)

        self.conv1 = nn.Conv1d(
            in_channels=2,
            out_channels=16,
            kernel_size=5,
            stride=2,
        )
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(
            in_channels=16,
            out_channels=32,
            kernel_size=5,
            stride=2,
        )

        with torch.no_grad():
            tmp = torch.zeros(1, 2, self.shape)
            tmp = self.relu(self.conv1(tmp))
            tmp = self.relu(self.conv2(tmp))
            flat_dim = tmp.reshape(1, -1).shape[1]

        self.fc1 = nn.Linear(flat_dim, self.output_dim)

    def forward(self, x):
        x = preprocess_obs(x)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        if x.shape[1] != 2 * self.shape:
            raise ValueError(
                f"Branch input dimension is {x.shape[1]}, expected {2 * self.shape}."
            )

        x = x.reshape(x.shape[0], 2, self.shape)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = self.relu(self.fc1(x))
        return x


class TrunkNet(nn.Module):
    def __init__(self, dim_x=1, width=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_x, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, width),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x.float())


class DeepONetModified(nn.Module):
    """
    Hadamard-fusion DeepONet controller.

        b(s) in R^p,
        t(x) in R^p,
        z(s,x) = b(s) * t(x) in R^p,
        U_prior = f_ctrl(z) in R.
    """

    def __init__(self, shape=N, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = int(latent_dim)

        self.branch = BranchNet(shape, output_dim=self.latent_dim)
        self.trunk = TrunkNet(dim_x=1, width=self.latent_dim)

        # Hadamard fusion preserves latent_dim features.
        self.fc3 = nn.Linear(self.latent_dim, 32)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, 1)

    def extract_latent(self, obs, grid):
        b = self.branch(obs)       # [batch, latent_dim]
        t = self.trunk(grid)       # [n_query, latent_dim]

        if t.shape[0] == 1:
            t = t.expand(b.shape[0], -1)
        elif t.shape[0] != b.shape[0]:
            raise ValueError(
                f"Branch batch {b.shape[0]} and trunk batch {t.shape[0]} do not match."
            )

        z = b * t                  # Hadamard product
        return z

    def control_head(self, z):
        y = self.relu(self.fc3(z))
        y = self.relu(self.fc4(y))
        y = self.fc5(y)
        return y

    def forward(self, obs, grid, return_latent=False):
        z = self.extract_latent(obs, grid)
        u_prior_physical = self.control_head(z)

        if return_latent:
            return u_prior_physical, z

        return u_prior_physical


class HybridCACDeepONetFeatureExtractor(BaseFeaturesExtractor):
    """
    Return the Hadamard-fused DeepONet latent feature.

    Input observation and DeepONet input:
        [H_1,...,H_N,Q_1,...,Q_N]

    SAC feature:
        [z_1,...,z_p]
    """

    def __init__(
        self,
        observation_space: spaces.Box,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        query_point: float = 1.0,
        freeze_deeponet: bool = False,
    ):
        if features_dim != SAC_FEATURE_DIM:
            raise ValueError(
                f"features_dim must be {SAC_FEATURE_DIM}, got {features_dim}."
            )

        super().__init__(observation_space, features_dim)

        self.query_point = float(query_point)
        self.net1 = DeepONetModified(N, latent_dim=LATENT_DIM)

        # Normalized action used by the hybrid SAC actor.
        self.u_deep_norm: Optional[torch.Tensor] = None

        # Physical prior gate retained for diagnostics.
        self.u_prior_physical: Optional[torch.Tensor] = None

        self.register_buffer(
            "grid",
            torch.tensor([[self.query_point]], dtype=torch.float32),
        )

        self._load_pretrained(pretrained_path)
        self._apply_freeze(freeze_all=freeze_deeponet)

    def _apply_freeze(self, freeze_all=False):
        """Preserve the uploaded method: either freeze all or fine-tune all."""
        if freeze_all:
            for param in self.net1.parameters():
                param.requires_grad = False
            print("All DeepONet parameters are frozen.")
            return

        # Uploaded-method setting: all parameters of the NEW DeepONet are
        # trainable, but optimizer parameter groups keep their learning rate at
        # deeponet_learning_rate rather than the base SAC learning rate.
        for param in self.net1.parameters():
            param.requires_grad = True

        trainable = sum(
            p.numel() for p in self.net1.parameters() if p.requires_grad
        )
        total = sum(p.numel() for p in self.net1.parameters())

        print(
            f"New Hadamard DeepONet fully trainable: {trainable}/{total} "
            f"parameters ({100.0 * trainable / max(total, 1):.2f}%)."
        )
        print(
            "Branch, Trunk, and control head are all fine-tuned with the "
            f"smaller learning rate {deeponet_learning_rate}."
        )

    @staticmethod
    def _clean_state_dict_keys(state_dict):
        cleaned = {}
        prefixes = ["module.", "model.", "net1.", "deeponet."]

        for key, value in state_dict.items():
            new_key = key
            changed = True
            while changed:
                changed = False
                for prefix in prefixes:
                    if new_key.startswith(prefix):
                        new_key = new_key[len(prefix):]
                        changed = True
            cleaned[new_key] = value

        return cleaned

    def _load_pretrained(self, pretrained_path: str):
        path = Path(pretrained_path)

        if not path.exists():
            raise FileNotFoundError(
                f"Pretrained Hadamard DeepONet checkpoint not found: {path}"
            )

        state_dict = torch.load(str(path), map_location="cpu")

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]

        if not isinstance(state_dict, dict):
            raise TypeError(
                f"Unsupported checkpoint type: {type(state_dict).__name__}"
            )

        cleaned = self._clean_state_dict_keys(state_dict)

        try:
            self.net1.load_state_dict(cleaned, strict=True)
        except RuntimeError as exc:
            raise RuntimeError(
                "The checkpoint is incompatible with the Hadamard-fusion "
                "DeepONet. Re-pretrain DeepONet with fc3=Linear(102,32) and "
                "z=b*t before running this RL script.\n"
                f"Original error:\n{exc}"
            ) from exc

        print(f"Loaded pretrained Hadamard DeepONet: {path}")

    @staticmethod
    def _physical_gate_to_normalized_action(u_physical):
        scale = max_control_value - min_control_value
        if scale <= 0.0:
            raise ValueError("max_control_value must be greater than min_control_value.")
        return 2.0 * (u_physical - min_control_value) / scale - 1.0

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        obs = preprocess_obs(observations)
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)

        if obs.shape[1] != RAW_STATE_DIM:
            raise ValueError(
                f"Raw observation dimension is {obs.shape[1]}, "
                f"expected {RAW_STATE_DIM}."
            )

        u_prior_physical, z = self.net1(
            obs,
            self.grid.to(obs.device),
            return_latent=True,
        )

        # Clip the physical prior to the admissible gate interval before mapping.
        u_prior_physical = torch.clamp(
            u_prior_physical,
            min_control_value,
            max_control_value,
        )
        self.u_prior_physical = u_prior_physical

        u_deep_norm = self._physical_gate_to_normalized_action(u_prior_physical)
        self.u_deep_norm = torch.clamp(u_deep_norm, -1.0, 1.0)

        # Representation-level transfer: use the fused DeepONet feature z only.
        return z.view(z.size(0), -1)


class HybridCACActor(Actor):
    """SAC actor whose normalized output is blended with DeepONet prior action."""

    def __init__(self, *args, alpha: float = 0.4, **kwargs):
        super().__init__(*args, **kwargs)
        init_alpha = float(np.clip(alpha, 0.0, 1.0))

        self.register_buffer(
            "alpha_tensor",
            torch.tensor(init_alpha, dtype=torch.float32),
        )

        self.last_deep_action = None
        self.last_rl_action = None
        self.last_hybrid_action = None

    @property
    def alpha(self) -> float:
        return float(self.alpha_tensor.detach().cpu().item())

    def set_alpha(self, alpha: float):
        alpha_value = float(np.clip(alpha, 0.0, 1.0))
        with torch.no_grad():
            self.alpha_tensor.fill_(alpha_value)

    def _deep_action(self, obs: torch.Tensor) -> torch.Tensor:
        u_deep = getattr(self.features_extractor, "u_deep_norm", None)

        if u_deep is None or u_deep.shape[0] != obs.shape[0]:
            _ = self.features_extractor(obs)
            u_deep = self.features_extractor.u_deep_norm

        return torch.clamp(u_deep, -1.0, 1.0)

    def _hybridize(self, rl_action: torch.Tensor, obs: torch.Tensor) -> torch.Tensor:
        u_deep = self._deep_action(obs)

        alpha = self.alpha_tensor.to(
            device=rl_action.device,
            dtype=rl_action.dtype,
        )

        hybrid = (1.0 - alpha) * u_deep + alpha * rl_action
        hybrid = torch.clamp(hybrid, -1.0, 1.0)

        self.last_deep_action = u_deep.detach()
        self.last_rl_action = rl_action.detach()
        self.last_hybrid_action = hybrid.detach()

        return hybrid

    def forward(self, obs: torch.Tensor, deterministic: bool = False):
        rl_action = super().forward(obs, deterministic=deterministic)
        return self._hybridize(rl_action, obs)

    def action_log_prob(self, obs: torch.Tensor):
        # The log probability is that of the SAC stochastic component.
        # The deterministic affine hybrid mapping is not included in log_prob.
        rl_action, log_prob = super().action_log_prob(obs)
        hybrid_action = self._hybridize(rl_action, obs)
        return hybrid_action, log_prob


class HybridCACCritic(nn.Module):
    """Twin-Q critic using the Hadamard-fused DeepONet feature."""

    def __init__(
        self,
        observation_space: spaces.Box,
        action_space: spaces.Box,
        hidden_dim: int = 256,
        features_dim: int = SAC_FEATURE_DIM,
        pretrained_path: str = DEEPNET_PRETRAINED_PATH,
    ):
        super().__init__()

        self.features_extractor = HybridCACDeepONetFeatureExtractor(
            observation_space,
            features_dim=features_dim,
            pretrained_path=pretrained_path,
            query_point=1.0,
            freeze_deeponet=False,
        )

        action_dim = int(np.prod(action_space.shape))
        input_dim = features_dim + action_dim

        self.q1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.q2 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def set_training_mode(self, mode: bool):
        self.train(mode)

    def forward(self, obs: torch.Tensor, actions: torch.Tensor):
        features = self.features_extractor(obs)
        q_input = torch.cat([features, actions], dim=1)
        return self.q1(q_input), self.q2(q_input)


class HybridCACSACPolicy(SACPolicy):
    """SAC policy with Hadamard DeepONet representation/control transfer."""

    def __init__(
        self,
        *args,
        hybrid_alpha_initial: float = 0.4,
        deeponet_pretrained_path: str = DEEPNET_PRETRAINED_PATH,
        **kwargs,
    ):
        self.hybrid_alpha_initial = float(hybrid_alpha_initial)
        self.deeponet_pretrained_path = deeponet_pretrained_path

        kwargs.setdefault(
            "features_extractor_class",
            HybridCACDeepONetFeatureExtractor,
        )
        kwargs.setdefault(
            "features_extractor_kwargs",
            dict(
                features_dim=SAC_FEATURE_DIM,
                pretrained_path=deeponet_pretrained_path,
                query_point=1.0,
                freeze_deeponet=False,
            ),
        )

        super().__init__(*args, **kwargs)

    def make_actor(self, features_extractor=None):
        kwargs = self._update_features_extractor(
            self.actor_kwargs,
            features_extractor,
        )
        return HybridCACActor(
            **kwargs,
            alpha=self.hybrid_alpha_initial,
        ).to(self.device)

    def make_critic(self, features_extractor=None):
        return HybridCACCritic(
            self.observation_space,
            self.action_space,
            hidden_dim=256,
            features_dim=SAC_FEATURE_DIM,
            pretrained_path=self.deeponet_pretrained_path,
        ).to(self.device)

    def set_alpha(self, alpha: float):
        self.actor.set_alpha(alpha)


# ============================================================
# 8. Optimizer helper: separate DeepONet learning rate
# ============================================================

def _make_param_groups_with_deeponet_lr(module: nn.Module):
    deeponet_params = []
    other_params = []

    for name, param in module.named_parameters():
        if not param.requires_grad:
            continue

        if "features_extractor.net1" in name or name.startswith("net1."):
            deeponet_params.append(param)
        else:
            other_params.append(param)

    groups = []

    if other_params:
        groups.append(
            {
                "params": other_params,
                "lr": learning_rate,
                "is_deeponet_group": False,
            }
        )

    if deeponet_params:
        groups.append(
            {
                "params": deeponet_params,
                "lr": deeponet_learning_rate,
                "is_deeponet_group": True,
            }
        )

    return groups


def set_deeponet_small_lr_optimizers(model):
    actor_groups = _make_param_groups_with_deeponet_lr(model.actor)
    critic_groups = _make_param_groups_with_deeponet_lr(model.critic)

    model.actor.optimizer = torch.optim.Adam(actor_groups, eps=1e-5)
    model.critic.optimizer = torch.optim.Adam(critic_groups, eps=1e-5)

    n_actor_deep = sum(
        p.numel()
        for group in actor_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_actor_other = sum(
        p.numel()
        for group in actor_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_deep = sum(
        p.numel()
        for group in critic_groups
        if group.get("is_deeponet_group", False)
        for p in group["params"]
    )
    n_critic_other = sum(
        p.numel()
        for group in critic_groups
        if not group.get("is_deeponet_group", False)
        for p in group["params"]
    )

    print("Custom optimizers with separate DeepONet learning rate are set.")
    print(f"Actor params: DeepONet={n_actor_deep}, other={n_actor_other}")
    print(f"Critic params: DeepONet={n_critic_deep}, other={n_critic_other}")
    print(
        f"Base SAC learning_rate={learning_rate}, "
        f"DeepONet learning_rate={deeponet_learning_rate}"
    )


class HybridSAC(SAC):
    """Preserve the smaller learning rate of DeepONet parameter groups."""

    def _update_learning_rate(self, optimizers):
        super()._update_learning_rate(optimizers)

        if not isinstance(optimizers, (list, tuple)):
            optimizers = [optimizers]

        current_lr = self.lr_schedule(self._current_progress_remaining)

        for optimizer in optimizers:
            for group in optimizer.param_groups:
                if group.get("is_deeponet_group", False):
                    group["lr"] = deeponet_learning_rate
                elif "is_deeponet_group" in group:
                    group["lr"] = current_lr


# ============================================================
# 9. Hybrid alpha scheduling
# ============================================================

def alpha_schedule(step: int) -> float:
    """
    alpha is the SAC weight:
        alpha small -> DeepONet prior dominates,
        alpha large -> SAC compensation dominates.
    """
    if step < 10000:
        return 0.1
    if step < 20000:
        return 0.5
    return 0.9


class AlphaCallback(BaseCallback):
    def _on_training_start(self):
        alpha = alpha_schedule(self.num_timesteps)
        self.model.policy.set_alpha(alpha)
        print(f"Initial hybrid alpha = {alpha}")

    def _on_step(self):
        alpha = alpha_schedule(self.num_timesteps)
        self.model.policy.set_alpha(alpha)
        return True


# ============================================================
# 10. Environment creation
# ============================================================

def make_env():
    p, X, dx, H_eq, Q_eq = build_collection_matched_reference()

    # Constructor fallback is the reference itself. Every reset then samples
    # the same Chebyshev initial-condition distribution as data collection.
    H_init = H_eq.copy()
    Q_init = Q_eq.copy()

    reward_class = CACTrackingSmoothReward(
        H_eq=H_eq,
        Q_eq=Q_eq,
        delta_t=control_sample_rate,
        H_scale=H_scale,
        Q_scale=Q_scale,
        U_scale=U_scale,
        alpha=reward_alpha,
        beta=reward_beta,
        gamma=reward_gamma,
        delta=reward_delta,
        tracking_w_H=tracking_w_H,
        tracking_w_Q=tracking_w_Q,
        change_w_H=change_w_H,
        change_w_Q=change_w_Q,
        U_ref=U_EQ_VALUE,
        terminal_tracking_weight=terminal_tracking_weight,
        truncate_penalty=truncate_penalty,
    )

    base_env = CACSaintVenantPDE1D(
        params=p,
        X_grid=X,
        Hbar=H_eq,
        Qbar=Q_eq,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda observation: observation,
        reset_init_condition_func=random_initial_condition_func,
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=max_state_value,
        min_control_value=min_control_value,
        max_control_value=max_control_value,
        control_sample_rate=control_sample_rate,
        reward_class=reward_class,
        # SAC emits normalized actions; the environment maps them to physical U_L.
        normalize=True,
        # The SAC observation is the raw physical state [H,Q].
        observation_mode="raw",
        H_scale=H_scale,
        Q_scale=Q_scale,
        obs_clip=obs_clip,
        T=T,
        dt=dt,
        cfl=cfl,
        check_cfl=check_cfl,
        well_balanced=well_balanced,
    )

    env = base_env

    # Expose the exact reference for diagnostics and test-program reuse.
    env.reference_X = X
    env.reference_H = H_eq
    env.reference_Q = Q_eq
    env.reference_dx = dx

    print(
        "Collection-matched ODE reference: "
        f"Hbar(0)={H_eq[0]:.9f}, Hbar(L)={H_eq[-1]:.9f}, "
        f"Qbar={Q_eq[0]:.9f}, dx={dx:.6f}"
    )

    return env


# ============================================================
# 11. Training
# ============================================================

def train_with_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    env = make_env()

    reset_obs, _ = env.reset(seed=seed)
    print(f"Raw observation shape: {reset_obs.shape}")

    if reset_obs.shape != (RAW_STATE_DIM,):
        raise RuntimeError(
            f"Expected raw observation shape {(RAW_STATE_DIM,)}, "
            f"got {reset_obs.shape}."
        )

    policy_kwargs = dict(
        deeponet_pretrained_path=DEEPNET_PRETRAINED_PATH,
        hybrid_alpha_initial=alpha_schedule(0),
        net_arch=dict(pi=[256, 256], qf=[256, 256]),
        optimizer_kwargs={"eps": 1e-6},
        share_features_extractor=False,
    )

    model = HybridSAC(
        policy=HybridCACSACPolicy,
        policy_kwargs=policy_kwargs,
        env=env,
        learning_rate=learning_rate,
        buffer_size=buffer_size,
        batch_size=batch_size,
        tau=tau,
        gamma=sac_gamma,
        #ent_coef=ent_coef,
        #learning_starts=learning_starts,
        verbose=1,
        tensorboard_log=f"./tbsaccontrolx/seed_{seed}/",
    )

    set_deeponet_small_lr_optimizers(model)

    checkpoint_callback = CheckpointCallback(
        save_freq=10000,
        save_path=f"./saccontrolx/seed_{seed}/",
        name_prefix="sac_cac_new_hadamard_hybrid_all_trainable_raw_HQ",
        save_replay_buffer=True,
        save_vecnormalize=True,
    )

    alpha_callback = AlphaCallback()
    callbacks = CallbackList([alpha_callback, checkpoint_callback])

    start_time = time.perf_counter()

    model.learn(
        total_timesteps=total_timesteps,
        callback=callbacks,
    )

    training_time = time.perf_counter() - start_time
    hours = int(training_time // 3600)
    minutes = int((training_time % 3600) // 60)
    seconds = training_time % 60

    print("\n" + "=" * 70)
    print(f"Seed {seed} training completed")
    print(
        f"Training time for this seed: "
        f"{hours:02d} h {minutes:02d} min {seconds:06.2f} s"
    )
    print(f"Total training time: {training_time:.2f} seconds")
    print("=" * 70)

    save_path = (
        f"./saccontrolx/seed_{seed}/"
        "sac_cac_new_hadamard_hybrid_all_trainable_raw_HQ_final"
    )
    model.save(save_path)
    env.close()

    return model, save_path


def main():
    print("Hybrid SAC with NEW Hadamard DeepONet: all-parameter fine-tuning")
    print("=" * 90)
    print(f"N={N}, T={T}, dt={dt}, control_sample_rate={control_sample_rate}")
    print(f"Physical gate U_L in [{min_control_value}, {max_control_value}]")
    print("Observation: raw physical [H_1,...,H_N,Q_1,...,Q_N]")
    print(
        f"Observation dimension={RAW_STATE_DIM}, "
        f"SAC feature dimension={SAC_FEATURE_DIM}"
    )
    print("DeepONet input: raw [H,Q]")
    print("DeepONet output: physical U_L pretrained from y_ul.dat")
    print("DeepONet fusion: z=b(s)*t(L), followed by 102->32->16->1")
    print("Hybrid action: a=(1-alpha_hybrid)*a_prior+alpha_hybrid*a_SAC")
    print(f"DeepONet checkpoint: {DEEPNET_PRETRAINED_PATH}")
    print(
        "Transfer strategy retained from uploaded method: all NEW DeepONet "
        "parameters trainable at deeponet_learning_rate"
    )
    print(
        f"Reference: Qbar={Q_BAR_FIXED} fixed; Hbar solved by steady ODE "
        f"(mode={REFERENCE_MODE})"
    )
    print(
        "Reward: -alpha*tracking - beta*state_change "
        "- gamma*control - delta*delta_control"
    )
    print(
        f"Reward weights: alpha={reward_alpha}, beta={reward_beta}, "
        f"gamma={reward_gamma}, delta={reward_delta}"
    )
    print(
        f"Tracking H/Q weights=({tracking_w_H}, {tracking_w_Q}); "
        f"state-change H/Q weights=({change_w_H}, {change_w_Q})"
    )
    print(f"Reward scales: H={H_scale}, Q={Q_scale}, U={U_scale}")
    print("Cumulative-error state: removed")
    print(
        "Chebyshev initial conditions: "
        f"order={CHEB_ORDER}, A_H={H_CHEB_AMPLITUDE}, "
        f"A_Q={Q_CHEB_AMPLITUDE}, decay={CHEB_COEFF_DECAY}"
    )
    print(
        f"Learning rates: SAC={learning_rate}, "
        f"DeepONet={deeponet_learning_rate}"
    )

    for index, seed in enumerate(seeds):
        print(f"\n===== Training {index + 1}/{len(seeds)}, seed={seed} =====")
        _, save_path = train_with_seed(seed)
        print(f"Saved model: {save_path}")


if __name__ == "__main__":
    main()


Hybrid SAC with NEW Hadamard DeepONet: all-parameter fine-tuning
N=102, T=40000.0, dt=5.0, control_sample_rate=400.0
Physical gate U_L in [0.15, 0.4]
Observation: raw physical [H_1,...,H_N,Q_1,...,Q_N]
Observation dimension=204, SAC feature dimension=102
DeepONet input: raw [H,Q]
DeepONet output: physical U_L pretrained from y_ul.dat
DeepONet fusion: z=b(s)*t(L), followed by 102->32->16->1
Hybrid action: a=(1-alpha_hybrid)*a_prior+alpha_hybrid*a_SAC
DeepONet checkpoint: modedlNet10000QqQqQ14444_hadamard.pt
Transfer strategy retained from uploaded method: all NEW DeepONet parameters trainable at deeponet_learning_rate
Reference: Qbar=11.93 fixed; Hbar solved by steady ODE (mode=ode)
Reward: -alpha*tracking - beta*state_change - gamma*control - delta*delta_control
Reward weights: alpha=1.0, beta=0.1, gamma=0.05, delta=0.1
Tracking H/Q weights=(1.0, 1.0); state-change H/Q weights=(1.0, 1.0)
Reward scales: H=0.3, Q=3.0, U=0.1
Cumulative-error state: removed
Chebyshev initial conditions: or